# Importing Libraries & Datasets

In [1]:
import pandas as pd
import numpy as np
import os

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

print("Environment initialized successfully.")
print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)

Environment initialized successfully.
Pandas version: 2.2.3
NumPy version: 2.4.6


In [2]:
ball_by_ball = pd.read_csv("ball_by_ball.csv")
playing_eleven = pd.read_csv("playing_eleven.csv")
matches = pd.read_csv("matches.csv")
competition_team_player_mapping = pd.read_csv(
    "competition_team_player_mapping.csv"
)
teams = pd.read_csv("teams.csv")
competitions = pd.read_csv("competitions.csv")

datasets = {
    "ball_by_ball": ball_by_ball,
    "playing_eleven": playing_eleven,
    "matches": matches,
    "competition_team_player_mapping": competition_team_player_mapping,
    "teams": teams,
    "competitions": competitions
}

print("All datasets loaded successfully.")

All datasets loaded successfully.


# Dataset Inventory

In [3]:
inventory = []

for name, df in datasets.items():
    inventory.append({
        "Dataset": name,
        "Rows": df.shape[0],
        "Columns": df.shape[1],
        "Memory (MB)": round(df.memory_usage(deep=True).sum() / (1024**2), 3),
        "Duplicate Rows": df.duplicated().sum()
    })

inventory_df = pd.DataFrame(inventory)

print("DATASET INVENTORY")
display(inventory_df)


DATASET INVENTORY


,Dataset,Rows,Columns,Memory (MB),Duplicate Rows
0,ball_by_ball,200999,33,165.277,0
1,playing_eleven,28381,9,3.209,0
2,matches,883,22,0.462,0
3,competition_team_player_mapping,6413,6,1.388,0
4,teams,353,3,0.026,0
5,competitions,141,4,0.027,0


# Schema and Data Types

In [4]:
for name, df in datasets.items():

    print(f"DATASET: {name}")

    schema = pd.DataFrame({
        "Column": df.columns,
        "Data Type": df.dtypes.astype(str).values,
        "Non-Null Count": df.notna().sum().values,
        "Null Count": df.isna().sum().values,
        "Unique Values": df.nunique(dropna=True).values
    })

    display(schema)

DATASET: ball_by_ball


,Column,Data Type,Non-Null Count,Null Count,Unique Values
0,match_id,int64,200999,0,867
1,innings,int64,200999,0,4
2,overid,int64,200999,0,20
3,over_ball_num,int64,200999,0,13
4,innings_ball_num,int64,200999,0,136
5,batting_team_id,int64,200999,0,38
6,bowling_team_id,int64,200999,0,38
7,striker_id,int64,200999,0,888
8,striker_name,object,200999,0,885
9,non_striker_id,int64,200999,0,867


DATASET: playing_eleven


,Column,Data Type,Non-Null Count,Null Count,Unique Values
0,match_id,int64,28381,0,880
1,team_id,int64,28381,0,38
2,player_id,int64,28381,0,1082
3,batting_order,float64,19703,8678,13
4,is_played,bool,28381,0,2
5,is_captain,object,19714,8667,2
6,is_wk,object,19714,8667,2
7,is_impact,int64,28381,0,2
8,is_impact_off,int64,28381,0,2


DATASET: matches


,Column,Data Type,Non-Null Count,Null Count,Unique Values
0,match_id,int64,883,0,883
1,comp_id,int64,883,0,138
2,match_number,int64,883,0,75
3,match_date,object,883,0,686
4,stage,object,883,0,7
5,venue_id,int64,883,0,98
6,venue_country,object,883,0,17
7,home_team_id,int64,883,0,36
8,home_team,object,883,0,36
9,away_team_id,int64,883,0,36


... (2 outputs left out)

DATASET: teams


,Column,Data Type,Non-Null Count,Null Count,Unique Values
0,comp_id,int64,353,0,141
1,team_id,int64,353,0,38
2,team_name,object,353,0,38


DATASET: competitions


,Column,Data Type,Non-Null Count,Null Count,Unique Values
0,comp_id,int64,141,0,141
1,comp_name,object,141,0,141
2,match_format,object,141,0,2
3,season,object,141,0,8


# Identifier Relationships and Dataset Coverage

### Match ID Coverage

In [5]:
match_sets = {
    "matches": set(matches["match_id"]),
    "ball_by_ball": set(ball_by_ball["match_id"]),
    "playing_eleven": set(playing_eleven["match_id"])
}

print("MATCH COVERAGE SUMMARY")
print("-" * 65)

for name, ids in match_sets.items():
    print(f"{name:20s}: {len(ids):5d} unique matches")

print("\nCROSS-DATASET MATCH COVERAGE")
print("-" * 65)

for source in ["ball_by_ball", "playing_eleven"]:
    source_ids = match_sets[source]
    reference_ids = match_sets["matches"]

    print(f"\n{source} vs matches")
    print("Matches present in both:",
          len(source_ids & reference_ids))
    print("Matches missing from matches.csv:",
          len(source_ids - reference_ids))
    print("Matches missing from", source + ":",
          len(reference_ids - source_ids))

    if reference_ids - source_ids:
        print("Sample missing match IDs:",
              sorted(reference_ids - source_ids)[:20])

MATCH COVERAGE SUMMARY
-----------------------------------------------------------------
matches             :   883 unique matches
ball_by_ball        :   867 unique matches
playing_eleven      :   880 unique matches

CROSS-DATASET MATCH COVERAGE
-----------------------------------------------------------------

ball_by_ball vs matches
Matches present in both: 867
Matches missing from matches.csv: 0
Matches missing from ball_by_ball: 16
Sample missing match IDs: [31281, 31284, 31288, 32610, 32612, 32622, 36210, 36564, 38797, 40761, 41502, 41503, 42030, 46008, 48513, 50189]

playing_eleven vs matches
Matches present in both: 880
Matches missing from matches.csv: 0
Matches missing from playing_eleven: 3
Sample missing match IDs: [36210, 41503, 48513]


### Player ID coverage

In [6]:
player_sets = {
    "ball_striker": set(ball_by_ball["striker_id"]),
    "ball_non_striker": set(ball_by_ball["non_striker_id"]),
    "ball_bowler": set(ball_by_ball["bowler_id"]),
    "playing_eleven": set(playing_eleven["player_id"]),
    "player_mapping": set(competition_team_player_mapping["player_id"])
}

print("PLAYER ID COVERAGE")
print("-" * 65)

for name, ids in player_sets.items():
    print(f"{name:20s}: {len(ids):5d} unique IDs")

print("\nPLAYER ID OVERLAP WITH MAPPING")
print("-" * 65)

mapping_ids = player_sets["player_mapping"]

for name, ids in player_sets.items():
    if name == "player_mapping":
        continue

    print(f"\n{name}")
    print("Present in mapping:", len(ids & mapping_ids))
    print("Missing from mapping:", len(ids - mapping_ids))

    if ids - mapping_ids:
        print("Sample unmatched IDs:",
              sorted(ids - mapping_ids)[:20])

PLAYER ID COVERAGE
-----------------------------------------------------------------
ball_striker        :   888 unique IDs
ball_non_striker    :   867 unique IDs
ball_bowler         :   647 unique IDs
playing_eleven      :  1082 unique IDs
player_mapping      :  1217 unique IDs

PLAYER ID OVERLAP WITH MAPPING
-----------------------------------------------------------------

ball_striker
Present in mapping: 888
Missing from mapping: 0

ball_non_striker
Present in mapping: 867
Missing from mapping: 0

ball_bowler
Present in mapping: 647
Missing from mapping: 0

playing_eleven
Present in mapping: 1082
Missing from mapping: 0


### Team and competition relationships

In [7]:
print("TEAM ID COVERAGE")
print("-" * 65)

reference_team_ids = set(teams["team_id"])

for column in ["home_team_id", "away_team_id"]:
    ids = set(matches[column])

    print(f"\n{column}")
    print("Unique IDs:", len(ids))
    print("Found in teams.csv:", len(ids & reference_team_ids))
    print("Not found in teams.csv:", len(ids - reference_team_ids))

    if ids - reference_team_ids:
        print("Unmatched IDs:", sorted(ids - reference_team_ids))


print("\nCOMPETITION ID COVERAGE")
print("-" * 65)

reference_comp_ids = set(competitions["comp_id"])

for name, df, column in [
    ("matches", matches, "comp_id"),
    ("mapping", competition_team_player_mapping, "comp_id"),
    ("teams", teams, "comp_id")
]:
    ids = set(df[column])

    print(f"\n{name}")
    print("Unique competition IDs:", len(ids))
    print("Found in competitions.csv:", len(ids & reference_comp_ids))
    print("Not found:", len(ids - reference_comp_ids))

    if ids - reference_comp_ids:
        print("Unmatched IDs:", sorted(ids - reference_comp_ids))

TEAM ID COVERAGE
-----------------------------------------------------------------

home_team_id
Unique IDs: 36
Found in teams.csv: 36
Not found in teams.csv: 0

away_team_id
Unique IDs: 36
Found in teams.csv: 36
Not found in teams.csv: 0

COMPETITION ID COVERAGE
-----------------------------------------------------------------

matches
Unique competition IDs: 138
Found in competitions.csv: 138
Not found: 0

mapping
Unique competition IDs: 140
Found in competitions.csv: 140
Not found: 0

teams
Unique competition IDs: 141
Found in competitions.csv: 141
Not found: 0


### Investigate the 16 Missing Ball-by-Ball Matches

In [8]:
ball_match_ids = set(ball_by_ball["match_id"])
xi_match_ids = set(playing_eleven["match_id"])
all_match_ids = set(matches["match_id"])

missing_ball_ids = all_match_ids - ball_match_ids
missing_xi_ids = all_match_ids - xi_match_ids

# Create match-level coverage flags
coverage = matches.copy()

coverage["has_ball_by_ball"] = coverage["match_id"].isin(ball_match_ids)
coverage["has_playing_xi"] = coverage["match_id"].isin(xi_match_ids)

# Display missing ball-by-ball matches
missing_matches = coverage[
    ~coverage["has_ball_by_ball"]
].copy()

columns_to_show = [
    "match_id",
    "comp_id",
    "match_number",
    "match_date",
    "stage",
    "venue_country",
    "home_team",
    "away_team",
    "result",
    "win_team_id",
    "has_playing_xi"
]

print("MATCHES WITHOUT BALL-BY-BALL DATA")
display(
    missing_matches[columns_to_show]
    .sort_values(["comp_id", "match_date"])
)

print("\nCOVERAGE OVERLAP")
print("Missing from both datasets:",
      len(missing_ball_ids & missing_xi_ids))

print("Missing only from ball-by-ball:",
      len(missing_ball_ids - missing_xi_ids))

print("Missing only from playing XI:",
      len(missing_xi_ids - missing_ball_ids))

MATCHES WITHOUT BALL-BY-BALL DATA


,match_id,comp_id,match_number,match_date,stage,venue_country,home_team,away_team,result,win_team_id,has_playing_xi
396,31281,2193,63,2024-05-13,League,India,Gujarat Titans,Kolkata Knight Riders,Match Abandoned,NaN,True
399,31284,2193,66,2024-05-16,League,India,Sunrisers Hyderabad,Gujarat Titans,Match Abandoned,NaN,True
403,31288,2193,70,2024-05-19,League,India,Rajasthan Royals,Kolkata Knight Riders,Match Abandoned,NaN,True
425,32610,2354,1,2024-05-22,Others,United Kingdom,England,Pakistan,Match Abandoned,NaN,True
427,32612,2354,3,2024-05-28,Others,United Kingdom,England,Pakistan,Match Abandoned,NaN,True
447,32622,2398,3,2024-05-20,Others,Netherlands,Ireland,Scotland,Match Abandoned,NaN,True
453,38797,2442,3,2024-12-14,League,South Africa,South Africa,Pakistan,Match Abandoned,NaN,True
481,36564,2548,3,2024-09-15,League,United Kingdom,England,Australia,Match Abandoned,NaN,True
498,36210,2628,3,2025-09-14,League,United Kingdom,England,South Africa,Match Abandoned,NaN,False
593,40761,2725,59,2025-05-17,League,India,Royal Challengers Bengaluru,Kolkata Knight Riders,Match Abandoned,NaN,True



COVERAGE OVERLAP
Missing from both datasets: 3
Missing only from ball-by-ball: 13
Missing only from playing XI: 0


### Validate Team-Competition Relationships

In [9]:
valid_combinations = set(
    zip(teams["comp_id"], teams["team_id"])
)

# Validate home and away teams
home_combinations = set(
    zip(matches["comp_id"], matches["home_team_id"])
)

away_combinations = set(
    zip(matches["comp_id"], matches["away_team_id"])
)

invalid_home = home_combinations - valid_combinations
invalid_away = away_combinations - valid_combinations

print("TEAM-COMPETITION VALIDATION")
print("-" * 60)

print("Invalid home team combinations:", len(invalid_home))
print("Invalid away team combinations:", len(invalid_away))

print("\nInvalid home combinations:")
print(sorted(invalid_home)[:20])

print("\nInvalid away combinations:")
print(sorted(invalid_away)[:20])


TEAM-COMPETITION VALIDATION
------------------------------------------------------------
Invalid home team combinations: 0
Invalid away team combinations: 0

Invalid home combinations:
[]

Invalid away combinations:
[]


# Player Identity Integrity

### Build a unified player identity table

In [10]:
identity_sources = []

identity_sources.append(
    ball_by_ball[["striker_id", "striker_name"]]
    .rename(columns={
        "striker_id": "player_id",
        "striker_name": "player_name"
    })
    .assign(source="ball_striker")
)

identity_sources.append(
    ball_by_ball[["non_striker_id", "non_striker_name"]]
    .rename(columns={
        "non_striker_id": "player_id",
        "non_striker_name": "player_name"
    })
    .assign(source="ball_non_striker")
)

identity_sources.append(
    ball_by_ball[["bowler_id", "bowler_name"]]
    .rename(columns={
        "bowler_id": "player_id",
        "bowler_name": "player_name"
    })
    .assign(source="ball_bowler")
)

identity_sources.append(
    competition_team_player_mapping[["player_id", "player_name"]]
    .assign(source="player_mapping")
)

player_identity = pd.concat(
    identity_sources,
    ignore_index=True
)

print("Total identity observations:", len(player_identity))
print("Unique player IDs:", player_identity["player_id"].nunique())
print("Unique player names:", player_identity["player_name"].nunique())

display(player_identity.head(10))

Total identity observations: 609410
Unique player IDs: 1217
Unique player names: 1210


,player_id,player_name,source
0,1187,Jason Roy,ball_striker
1,1187,Jason Roy,ball_striker
2,1187,Jason Roy,ball_striker
3,1187,Jason Roy,ball_striker
4,1187,Jason Roy,ball_striker
5,1187,Jason Roy,ball_striker
6,6273,Tom Banton,ball_striker
7,6273,Tom Banton,ball_striker
8,6273,Tom Banton,ball_striker
9,6273,Tom Banton,ball_striker


### Check whether one player ID has multiple names

In [11]:
id_name_counts = (
    player_identity
    .groupby("player_id")
    .agg(
        unique_names=("player_name", "nunique"),
        names=("player_name", lambda x: sorted(x.dropna().unique())),
        sources=("source", lambda x: sorted(x.unique()))
    )
    .reset_index()
)

id_conflicts = id_name_counts[
    id_name_counts["unique_names"] > 1
].copy()

print("PLAYER IDs WITH MULTIPLE NAMES:",
      len(id_conflicts))

display(id_conflicts.sort_values(
    "unique_names", ascending=False
))

PLAYER IDs WITH MULTIPLE NAMES: 0


,player_id,unique_names,names,sources


### Check whether one name has multiple player IDs

In [12]:
name_id_counts = (
    player_identity
    .dropna(subset=["player_name"])
    .groupby("player_name")
    .agg(
        unique_ids=("player_id", "nunique"),
        player_ids=("player_id", lambda x: sorted(x.unique())),
        sources=("source", lambda x: sorted(x.unique()))
    )
    .reset_index()
)

name_conflicts = name_id_counts[
    name_id_counts["unique_ids"] > 1
].copy()

print("PLAYER NAMES ASSOCIATED WITH MULTIPLE IDs:",
      len(name_conflicts))

display(name_conflicts.sort_values(
    "unique_ids", ascending=False
))

PLAYER NAMES ASSOCIATED WITH MULTIPLE IDs: 7


,player_name,unique_ids,player_ids,sources
20,Abdul Samad,2,"[6922, 7446]","[ball_bowler, ball_non_striker, ball_striker, ..."
175,Bashir Ahmad,2,"[29833, 39479]","[ball_bowler, ball_non_striker, ball_striker, ..."
176,Basil Hameed,2,"[8638, 39444]","[ball_bowler, ball_non_striker, ball_striker, ..."
350,Haider Ali,2,"[7338, 28756]","[ball_bowler, ball_non_striker, ball_striker, ..."
652,Mohammad Ali,2,"[3083, 20229]","[ball_bowler, ball_non_striker, ball_striker, ..."
694,Muhammad Farooq,2,"[27373, 28978]","[ball_bowler, ball_non_striker, ball_striker, ..."
854,Rashid Khan,2,"[4722, 20172]","[ball_bowler, ball_non_striker, ball_striker, ..."


### Inspect cross-source name consistency

In [13]:
source_identity = (
    player_identity
    .groupby(["player_id", "source"])["player_name"]
    .agg(lambda x: sorted(x.dropna().unique()))
    .unstack(fill_value=None)
    .reset_index()
)

print("PLAYER IDENTITY ACROSS SOURCES")
display(source_identity.head(20))


PLAYER IDENTITY ACROSS SOURCES


source,player_id,ball_bowler,ball_non_striker,ball_striker,player_mapping
0,8,NaN,[Mahendra Singh Dhoni],[Mahendra Singh Dhoni],[Mahendra Singh Dhoni]
1,23,[Piyush Chawla],[Piyush Chawla],[Piyush Chawla],[Piyush Chawla]
2,28,[Ishant Sharma],[Ishant Sharma],[Ishant Sharma],[Ishant Sharma]
3,35,NaN,NaN,NaN,[Nathan Bracken]
4,47,[Virat Kohli],[Virat Kohli],[Virat Kohli],[Virat Kohli]
5,88,NaN,[Faf du Plessis],[Faf du Plessis],[Faf du Plessis]
6,109,NaN,[Wriddhiman Saha],[Wriddhiman Saha],[Wriddhiman Saha]
7,126,NaN,[Rohit Sharma],[Rohit Sharma],[Rohit Sharma]
8,142,NaN,[Shikhar Dhawan],[Shikhar Dhawan],[Shikhar Dhawan]
9,143,[Dinesh Karthik],[Dinesh Karthik],[Dinesh Karthik],[Dinesh Karthik]


### Investigate the Seven Identity Conflicts

In [14]:
conflict_names = [
    "Abdul Samad",
    "Bashir Ahmad",
    "Basil Hameed",
    "Haider Ali",
    "Mohammad Ali",
    "Muhammad Farooq",
    "Rashid Khan"
]

conflict_ids = (
    name_conflicts[
        name_conflicts["player_name"].isin(conflict_names)
    ]["player_ids"]
    .explode()
    .astype(int)
    .unique()
)

# 1. Competition and team associations
print("COMPETITION AND TEAM ASSOCIATIONS")

mapping_conflicts = competition_team_player_mapping[
    competition_team_player_mapping["player_id"].isin(conflict_ids)
].sort_values(["player_name", "player_id", "comp_id"])

display(mapping_conflicts)

# 2. Playing XI appearances
print("\nPLAYING XI APPEARANCES")

xi_conflicts = playing_eleven[
    playing_eleven["player_id"].isin(conflict_ids)
].merge(
    matches[["match_id", "comp_id", "match_date"]],
    on="match_id",
    how="left"
)

display(
    xi_conflicts[
        ["player_id", "match_id", "comp_id",
         "match_date", "team_id", "batting_order",
         "is_played", "is_impact", "is_impact_off"]
    ].sort_values(["player_id", "match_date"])
)

# 3. Ball-by-ball participation
print("\nBALL-BY-BALL PARTICIPATION")

role_frames = []

for role, column in [
    ("striker", "striker_id"),
    ("non_striker", "non_striker_id"),
    ("bowler", "bowler_id")
]:
    temp = ball_by_ball[
        ball_by_ball[column].isin(conflict_ids)
    ][["match_id", column]].copy()

    temp = temp.rename(columns={column: "player_id"})
    temp["role"] = role
    role_frames.append(temp)

conflict_ball = pd.concat(role_frames, ignore_index=True)

ball_summary = (
    conflict_ball
    .groupby(["player_id", "role"])
    .agg(
        observations=("match_id", "size"),
        matches=("match_id", "nunique")
    )
    .reset_index()
)

display(ball_summary.sort_values(["player_id", "role"]))

# 4. Overall summary
print("\nIDENTITY CONFLICT SUMMARY")

display(
    mapping_conflicts.groupby(
        ["player_name", "player_id"]
    ).agg(
        competitions=("comp_id", "nunique"),
        teams=("team_id", "nunique")
    ).reset_index()
)


COMPETITION AND TEAM ASSOCIATIONS


,comp_id,comp_name,team_id,team_name,player_id,player_name
3651,2670,Mens T20 Emerging Teams Asia Cup 2024,341,Pakistan A,6922,Abdul Samad
4184,2793,Pakistan in New Zealand T20I Series 2025,25,Pakistan,6922,Abdul Samad
5076,3223,South Africa in Pakistan T20I Series 2025,25,Pakistan,6922,Abdul Samad
5280,3288,Pakistan T20I Tri-Series 2025,25,Pakistan,6922,Abdul Samad
5775,3317,Pakistan in Sri Lanka T20I Series 2026,25,Pakistan,6922,Abdul Samad
6354,3765,Sri Lanka in Pakistan T20I Series 2026,25,Pakistan,6922,Abdul Samad
307,1358,IPL 2022,242,Sunrisers Hyderabad,7446,Abdul Samad
1661,1715,IPL 2023,242,Sunrisers Hyderabad,7446,Abdul Samad
2625,2193,IPL 2024,242,Sunrisers Hyderabad,7446,Abdul Samad
4084,2725,IPL 2025,3509,Lucknow Super Giants,7446,Abdul Samad



PLAYING XI APPEARANCES


,player_id,match_id,comp_id,match_date,team_id,batting_order,is_played,is_impact,is_impact_off
0,4722,14097,1358,2022-03-28,3508,9.000,True,0,0
2,4722,14103,1358,2022-04-02,3508,8.000,True,0,0
4,4722,14109,1358,2022-04-08,3508,8.000,True,0,0
5,4722,14114,1358,2022-04-11,3508,8.000,True,0,0
6,4722,14117,1358,2022-04-14,3508,8.000,True,0,0
...,...,...,...,...,...,...,...,...,...
209,29833,46754,3228,2025-10-03,78,NaN,False,0,0
212,29833,46755,3228,2025-10-05,78,11.000,True,0,0
213,29833,47260,3254,2025-10-29,78,NaN,False,0,0
215,29833,47261,3254,2025-10-31,78,NaN,False,0,0



BALL-BY-BALL PARTICIPATION


,player_id,role,observations,matches
0,4722,bowler,2780,122
1,4722,non_striker,414,59
2,4722,striker,540,71
3,6922,non_striker,97,8
4,6922,striker,105,9
5,7338,non_striker,136,11
6,7338,striker,126,12
7,7446,bowler,9,3
8,7446,non_striker,440,38
9,7446,striker,441,41



IDENTITY CONFLICT SUMMARY


,player_name,player_id,competitions,teams
0,Abdul Samad,6922,6,2
1,Abdul Samad,7446,5,2
2,Bashir Ahmad,29833,4,1
3,Bashir Ahmad,39479,1,1
4,Basil Hameed,8638,5,1
5,Basil Hameed,39444,1,1
6,Haider Ali,7338,4,2
7,Haider Ali,28756,5,1
8,Mohammad Ali,3083,1,1
9,Mohammad Ali,20229,2,1


### Test Whether Conflicting IDs Coexist

In [15]:
from itertools import combinations

results = []

for name in conflict_names:

    ids = sorted(
        name_conflicts.loc[
            name_conflicts["player_name"] == name,
            "player_ids"
        ].iloc[0]
    )

    for id1, id2 in combinations(ids, 2):

        m1 = competition_team_player_mapping[
            competition_team_player_mapping["player_id"] == id1
        ]

        m2 = competition_team_player_mapping[
            competition_team_player_mapping["player_id"] == id2
        ]

        common_comp = set(m1["comp_id"]) & set(m2["comp_id"])

        common_team_comp = set(
            zip(m1["comp_id"], m1["team_id"])
        ) & set(
            zip(m2["comp_id"], m2["team_id"])
        )

        xi1 = playing_eleven[
            playing_eleven["player_id"] == id1
        ]

        xi2 = playing_eleven[
            playing_eleven["player_id"] == id2
        ]

        common_matches = set(xi1["match_id"]) & set(xi2["match_id"])

        same_match_same_team = set(
            zip(xi1["match_id"], xi1["team_id"])
        ) & set(
            zip(xi2["match_id"], xi2["team_id"])
        )

        results.append({
            "Player Name": name,
            "ID 1": id1,
            "ID 2": id2,
            "Common Competitions": sorted(common_comp),
            "Common Competition Count": len(common_comp),
            "Common Team-Competition Pairs": sorted(common_team_comp),
            "Common Team-Competition Count": len(common_team_comp),
            "Common XI Matches": sorted(common_matches),
            "Common XI Match Count": len(common_matches),
            "Same Match Same Team Count": len(same_match_same_team)
        })

coexistence_df = pd.DataFrame(results)

display(coexistence_df)

,Player Name,ID 1,ID 2,Common Competitions,Common Competition Count,Common Team-Competition Pairs,Common Team-Competition Count,Common XI Matches,Common XI Match Count,Same Match Same Team Count
0,Abdul Samad,6922,7446,[],0,[],0,[],0,0
1,Bashir Ahmad,29833,39479,[],0,[],0,[],0,0
2,Basil Hameed,8638,39444,[2670],1,"[(2670, 272)]",1,[],0,0
3,Haider Ali,7338,28756,[],0,[],0,[],0,0
4,Mohammad Ali,3083,20229,[],0,[],0,[],0,0
5,Muhammad Farooq,27373,28978,[2670],1,"[(2670, 272)]",1,[],0,0
6,Rashid Khan,4722,20172,[],0,[],0,[],0,0


# Team-Player-Competition Consistency

In [16]:
xi_combinations = playing_eleven[
    ["player_id", "team_id", "match_id"]
].merge(
    matches[["match_id", "comp_id"]],
    on="match_id",
    how="left"
)[["comp_id", "team_id", "player_id"]].drop_duplicates()

# Extract mapping combinations
mapping_combinations = competition_team_player_mapping[
    ["comp_id", "team_id", "player_id"]
].drop_duplicates()

# Compare combinations
xi_set = set(map(tuple, xi_combinations.values))
mapping_set = set(map(tuple, mapping_combinations.values))

unmatched_xi = xi_set - mapping_set

print("PLAYING XI MAPPING VALIDATION")
print("-" * 60)
print("Unique XI combinations:", len(xi_set))
print("Valid mapping combinations:", len(xi_set & mapping_set))
print("Unmatched XI combinations:", len(unmatched_xi))

if unmatched_xi:
    unmatched_df = pd.DataFrame(
        list(unmatched_xi),
        columns=["comp_id", "team_id", "player_id"]
    )

    display(
        unmatched_df.merge(
            competitions[["comp_id", "comp_name"]],
            on="comp_id",
            how="left"
        ).merge(
            teams[["comp_id", "team_id", "team_name"]],
            on=["comp_id", "team_id"],
            how="left"
        )
    )

PLAYING XI MAPPING VALIDATION
------------------------------------------------------------
Unique XI combinations: 5646
Valid mapping combinations: 5645
Unmatched XI combinations: 1


,comp_id,team_id,player_id,comp_name,team_name
0,2670,4444,29833,Mens T20 Emerging Teams Asia Cup 2024,Afghanistan A


#### Investigate the Bashir Ahmad Mismatch

In [17]:
target_comp = 2670
target_ids = [29833, 39479]

# Rebuild playing XI with competition information
xi_with_comp = playing_eleven.merge(
    matches[["match_id", "comp_id", "match_date"]],
    on="match_id",
    how="left"
)

# 1. Mapping records
print("PLAYER MAPPING RECORDS")

display(
    competition_team_player_mapping[
        (competition_team_player_mapping["comp_id"] == target_comp) &
        (competition_team_player_mapping["player_id"].isin(target_ids))
    ].sort_values(["player_id", "team_id"])
)

# 2. Playing XI records
print("\nPLAYING XI RECORDS")

bashir_xi = xi_with_comp[
    (xi_with_comp["comp_id"] == target_comp) &
    (xi_with_comp["player_id"].isin(target_ids))
].sort_values(["player_id", "match_id"])

display(bashir_xi)

# 3. Ball-by-ball participation
print("\nBALL-BY-BALL PARTICIPATION")

bashir_matches = set(bashir_xi["match_id"])

bashir_ball = ball_by_ball[
    (ball_by_ball["match_id"].isin(bashir_matches)) &
    (
        ball_by_ball["striker_id"].isin(target_ids) |
        ball_by_ball["non_striker_id"].isin(target_ids) |
        ball_by_ball["bowler_id"].isin(target_ids)
    )
]

display(
    bashir_ball[
        ["match_id", "batting_team_id", "bowling_team_id",
         "striker_id", "striker_name",
         "bowler_id", "bowler_name"]
    ].drop_duplicates()
)

print("\nSUMMARY")
print("Playing XI records:", len(bashir_xi))
print("Distinct XI matches:", bashir_xi["match_id"].nunique())
print("Ball-by-ball observations:", len(bashir_ball))

PLAYER MAPPING RECORDS


,comp_id,comp_name,team_id,team_name,player_id,player_name
3713,2670,Mens T20 Emerging Teams Asia Cup 2024,4444,Afghanistan A,39479,Bashir Ahmad



PLAYING XI RECORDS


,match_id,team_id,player_id,batting_order,is_played,is_captain,is_wk,is_impact,is_impact_off,comp_id,match_date
13137,37086,4444,29833,NaN,False,NaN,NaN,0,0,2670,2024-10-18
13271,37090,4444,29833,NaN,False,NaN,NaN,0,0,2670,2024-10-20
13370,37093,4444,29833,NaN,False,NaN,NaN,0,0,2670,2024-10-22
13539,37098,4444,29833,NaN,False,NaN,NaN,0,0,2670,2024-10-25
13568,37099,4444,29833,NaN,False,NaN,NaN,0,0,2670,2024-10-27



BALL-BY-BALL PARTICIPATION


,match_id,batting_team_id,bowling_team_id,striker_id,striker_name,bowler_id,bowler_name



SUMMARY
Playing XI records: 5
Distinct XI matches: 5
Ball-by-ball observations: 0


### Ball-by-Ball Player-Team-Competition Validation

In [18]:
ball_comp = ball_by_ball.merge(
    matches[["match_id", "comp_id"]],
    on="match_id",
    how="left"
)

# Build player-team-role combinations
striker_records = ball_comp[
    ["comp_id", "batting_team_id", "striker_id"]
].rename(columns={
    "batting_team_id": "team_id",
    "striker_id": "player_id"
})
striker_records["role"] = "striker"

non_striker_records = ball_comp[
    ["comp_id", "batting_team_id", "non_striker_id"]
].rename(columns={
    "batting_team_id": "team_id",
    "non_striker_id": "player_id"
})
non_striker_records["role"] = "non_striker"

bowler_records = ball_comp[
    ["comp_id", "bowling_team_id", "bowler_id"]
].rename(columns={
    "bowling_team_id": "team_id",
    "bowler_id": "player_id"
})
bowler_records["role"] = "bowler"

# Combine records
role_records = pd.concat(
    [striker_records, non_striker_records, bowler_records],
    ignore_index=True
)

# Unique combinations for each role
role_combinations = role_records[
    ["comp_id", "team_id", "player_id", "role"]
].drop_duplicates()

# Reference mapping
mapping_set = set(
    map(
        tuple,
        competition_team_player_mapping[
            ["comp_id", "team_id", "player_id"]
        ].drop_duplicates().values
    )
)

print("BALL-BY-BALL MAPPING VALIDATION")
print("-" * 65)

for role in ["striker", "non_striker", "bowler"]:

    subset = role_combinations[
        role_combinations["role"] == role
    ]

    role_set = set(
        map(
            tuple,
            subset[["comp_id", "team_id", "player_id"]].values
        )
    )

    unmatched = role_set - mapping_set

    print(f"\nROLE: {role}")
    print("Unique combinations:", len(role_set))
    print("Valid combinations:", len(role_set & mapping_set))
    print("Unmatched combinations:", len(unmatched))

    if unmatched:
        unmatched_df = pd.DataFrame(
            list(unmatched),
            columns=["comp_id", "team_id", "player_id"]
        )

        display(
            unmatched_df.merge(
                competitions[["comp_id", "comp_name"]],
                on="comp_id",
                how="left"
            ).merge(
                teams[["comp_id", "team_id", "team_name"]],
                on=["comp_id", "team_id"],
                how="left"
            )
        )

BALL-BY-BALL MAPPING VALIDATION
-----------------------------------------------------------------

ROLE: striker
Unique combinations: 4327
Valid combinations: 4327
Unmatched combinations: 0

ROLE: non_striker
Unique combinations: 4165
Valid combinations: 4165
Unmatched combinations: 0

ROLE: bowler
Unique combinations: 3101
Valid combinations: 3101
Unmatched combinations: 0


# Delivery-Level Cricket Data Quality Audit

### Runs, Extras and Boundary Consistency

In [19]:
print("1. RUNS DISTRIBUTION")
display(
    ball_by_ball["runs"]
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("runs")
    .reset_index(name="deliveries")
)

print("\n2. EXTRAS1 RUNS DISTRIBUTION")
display(
    ball_by_ball["extras1_runs"]
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("extras1_runs")
    .reset_index(name="deliveries")
)

print("\n3. EXTRAS_2 RUNS DISTRIBUTION")
display(
    ball_by_ball["extras_2_runs"]
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("extras_2_runs")
    .reset_index(name="deliveries")
)

print("\n4. EXTRAS TYPE DISTRIBUTION")
display(
    ball_by_ball["extras_type"]
    .value_counts(dropna=False)
    .rename_axis("extras_type")
    .reset_index(name="deliveries")
)

print("\n5. BOUNDARY AND WICKET COUNTS")
print("Four indicators:", ball_by_ball["is_four"].sum())
print("Six indicators:", ball_by_ball["is_six"].sum())
print("Wicket indicators:", ball_by_ball["is_wicket"].sum())

print("\n6. BOUNDARY FLAG CONSISTENCY")

print("Four flag but runs != 4:",
      ((ball_by_ball["is_four"]) &
       (ball_by_ball["runs"] != 4)).sum())

print("Six flag but runs != 6:",
      ((ball_by_ball["is_six"]) &
       (ball_by_ball["runs"] != 6)).sum())

print("Both four and six flags:",
      (ball_by_ball["is_four"] & ball_by_ball["is_six"]).sum())

print("\n7. COMBINED RUNS/EXTRAS DISTRIBUTION")

display(
    ball_by_ball.groupby(
        ["runs", "extras1_runs", "extras_2_runs"],
        dropna=False
    ).size()
    .reset_index(name="deliveries")
    .sort_values("deliveries", ascending=False)
    .head(30)
)

1. RUNS DISTRIBUTION


,runs,deliveries
0,0,79387
1,1,72267
2,2,12547
3,3,656
4,4,23303
5,5,25
6,6,12814



2. EXTRAS1 RUNS DISTRIBUTION


,extras1_runs,deliveries
0,1,576
1,2,7714
2,3,2748
3,4,529
4,5,264
5,6,189167
6,7,1



3. EXTRAS_2 RUNS DISTRIBUTION


,extras_2_runs,deliveries
0,0,200963
1,1,20
2,4,16



4. EXTRAS TYPE DISTRIBUTION


,extras_type,deliveries
0,NaN,189167
1,wide,7714
2,legbyes,2748
3,Overstep,576
4,byes,529
5,other,264
6,Penalty,1



5. BOUNDARY AND WICKET COUNTS
Four indicators: 23283
Six indicators: 12811
Wicket indicators: 10881

6. BOUNDARY FLAG CONSISTENCY
Four flag but runs != 4: 0
Six flag but runs != 6: 0
Both four and six flags: 0

7. COMBINED RUNS/EXTRAS DISTRIBUTION


,runs,extras1_runs,extras_2_runs,deliveries
15,1,6,0,72043
9,0,6,0,68028
24,4,6,0,23180
28,6,6,0,12741
18,2,6,0,12496
3,0,2,0,7714
4,0,3,0,2747
21,3,6,0,653
5,0,4,0,529
0,0,1,0,218


### Decode the Extras Fields

In [20]:
# 1. Cross-tabulation of extras type and extras1_runs
print("EXTRAS TYPE vs EXTRAS1_RUNS")

display(pd.crosstab(
    ball_by_ball["extras_type"].fillna("NO_EXTRAS"),
    ball_by_ball["extras1_runs"],
    margins=True
))

# 2. Cross-tabulation of extras type and extras_2_runs
print("\nEXTRAS TYPE vs EXTRAS_2_RUNS")

display(pd.crosstab(
    ball_by_ball["extras_type"].fillna("NO_EXTRAS"),
    ball_by_ball["extras_2_runs"],
    margins=True
))

# 3. Inspect actual deliveries without extras
print("\nSAMPLE DELIVERIES WITHOUT EXTRAS")

display(
    ball_by_ball[
        ball_by_ball["extras_type"].isna()
    ][[
        "match_id", "innings", "overid", "over_ball_num",
        "striker_name", "bowler_name", "runs",
        "extras_type", "extras1_runs", "extras_2_runs"
    ]].head(15)
)

# 4. Inspect actual wide deliveries
print("\nSAMPLE WIDE DELIVERIES")

display(
    ball_by_ball[
        ball_by_ball["extras_type"] == "wide"
    ][[
        "match_id", "innings", "overid", "over_ball_num",
        "striker_name", "bowler_name", "runs",
        "extras_type", "extras1_runs", "extras_2_runs"
    ]].head(15)
)

# 5. Inspect deliveries where runs = 4 but is_four is False
print("\nFOURS NOT FLAGGED AS BOUNDARIES")

display(
    ball_by_ball[
        (ball_by_ball["runs"] == 4) &
        (~ball_by_ball["is_four"])
    ][[
        "match_id", "innings", "overid", "over_ball_num",
        "striker_name", "bowler_name", "runs",
        "extras_type", "extras1_runs", "extras_2_runs",
        "is_four", "is_six"
    ]].head(30)
)

# 6. Inspect deliveries where runs = 6 but is_six is False
print("\nSIXES NOT FLAGGED AS BOUNDARIES")

display(
    ball_by_ball[
        (ball_by_ball["runs"] == 6) &
        (~ball_by_ball["is_six"])
    ][[
        "match_id", "innings", "overid", "over_ball_num",
        "striker_name", "bowler_name", "runs",
        "extras_type", "extras1_runs", "extras_2_runs",
        "is_four", "is_six"
    ]].head(15)
)

EXTRAS TYPE vs EXTRAS1_RUNS


extras1_runs,1,2,3,4,5,6,7,All
extras_type,,,,,,,,
NO_EXTRAS,0,0,0,0,0,189167,0,189167
Overstep,576,0,0,0,0,0,0,576
Penalty,0,0,0,0,0,0,1,1
byes,0,0,0,529,0,0,0,529
legbyes,0,0,2748,0,0,0,0,2748
other,0,0,0,0,264,0,0,264
wide,0,7714,0,0,0,0,0,7714
All,576,7714,2748,529,264,189167,1,200999



EXTRAS TYPE vs EXTRAS_2_RUNS


extras_2_runs,0,1,4,All
extras_type,,,,
NO_EXTRAS,189166,1,0,189167
Overstep,555,12,9,576
Penalty,1,0,0,1
byes,529,0,0,529
legbyes,2748,0,0,2748
other,250,7,7,264
wide,7714,0,0,7714
All,200963,20,16,200999



SAMPLE DELIVERIES WITHOUT EXTRAS


,match_id,innings,overid,over_ball_num,striker_name,bowler_name,runs,extras_type,extras1_runs,extras_2_runs
0,13511,1,1,1,Jason Roy,Sheldon Cottrell,0,NaN,6,0
1,13511,1,1,2,Jason Roy,Sheldon Cottrell,0,NaN,6,0
2,13511,1,1,3,Jason Roy,Sheldon Cottrell,0,NaN,6,0
3,13511,1,1,4,Jason Roy,Sheldon Cottrell,0,NaN,6,0
4,13511,1,1,5,Jason Roy,Sheldon Cottrell,6,NaN,6,0
5,13511,1,1,6,Jason Roy,Sheldon Cottrell,0,NaN,6,0
6,13511,1,2,1,Tom Banton,Jason Holder,0,NaN,6,0
7,13511,1,2,2,Tom Banton,Jason Holder,0,NaN,6,0
8,13511,1,2,3,Tom Banton,Jason Holder,0,NaN,6,0
9,13511,1,2,4,Tom Banton,Jason Holder,4,NaN,6,0


... (2 outputs left out)


FOURS NOT FLAGGED AS BOUNDARIES


,match_id,innings,overid,over_ball_num,striker_name,bowler_name,runs,extras_type,extras1_runs,extras_2_runs,is_four,is_six
7790,14113,1,1,3,Devdutt Padikkal,Dushmantha Chameera,4,NaN,6,0,False,False
10277,14123,1,3,6,Jos Buttler,Umesh Yadav,4,NaN,6,0,False,False
16372,14149,1,12,2,Devon Conway,Mitchell Marsh,4,NaN,6,0,False,False
23576,15609,2,17,8,Najibullah Zadran,Blessing Muzarabani,4,NaN,6,0,False,False
28308,16425,2,13,1,Afif Hossain,Wesley Madhevere,4,NaN,6,0,False,False
34269,16754,1,6,1,Najibullah Zadran,Curtis Campher,4,NaN,6,0,False,False
59426,20236,2,2,7,Yashasvi Jaiswal,Harshit Rana,4,NaN,6,0,False,False
67581,24350,2,6,6,Namboori Tilak Varma,Alzarri Joseph,4,NaN,6,0,False,False
75711,28958,1,3,2,Sadeera Samarawickrama,Richard Ngarava,4,NaN,6,0,False,False
75971,28959,1,5,1,Sean Williams,Dushmantha Chameera,4,NaN,6,0,False,False



SIXES NOT FLAGGED AS BOUNDARIES


,match_id,innings,overid,over_ball_num,striker_name,bowler_name,runs,extras_type,extras1_runs,extras_2_runs,is_four,is_six
36207,17396,1,19,3,Tim David,Jasprit Bumrah,6,NaN,6,0,False,False
95661,31277,1,7,5,Sai Sudharsan,Ravindra Jadeja,6,NaN,6,0,False,False
140761,40766,1,16,6,Hardik Pandya,Vyshak Kumar,6,NaN,6,0,False,False


### Investigate the ambiguous fields

In [21]:
# 1. Inspect every delivery where extras_2_runs is non-zero
print("ALL DELIVERIES WITH NON-ZERO EXTRAS_2_RUNS")

display(
    ball_by_ball[
        ball_by_ball["extras_2_runs"] != 0
    ][[
        "match_id", "innings", "overid", "over_ball_num",
        "striker_name", "bowler_name", "runs",
        "extras_type", "extras1_runs", "extras_2_runs",
        "is_four", "is_six", "is_wicket"
    ]].sort_values(
        ["extras_2_runs", "extras_type", "runs"]
    )
)

# 2. Runs distribution by extras type
print("\nRUNS DISTRIBUTION BY EXTRAS TYPE")

display(
    pd.crosstab(
        ball_by_ball["extras_type"].fillna("NO_EXTRAS"),
        ball_by_ball["runs"],
        margins=True
    )
)

# 3. Boundary flags against runs
print("\nRUNS vs IS_FOUR")

display(
    pd.crosstab(
        ball_by_ball["runs"],
        ball_by_ball["is_four"],
        margins=True
    )
)

print("\nRUNS vs IS_SIX")

display(
    pd.crosstab(
        ball_by_ball["runs"],
        ball_by_ball["is_six"],
        margins=True
    )
)

# 4. Inspect all boundary-flag mismatches with additional context
print("\nALL FOUR-RUN DELIVERIES NOT FLAGGED AS FOUR")

display(
    ball_by_ball[
        (ball_by_ball["runs"] == 4) &
        (~ball_by_ball["is_four"])
    ][[
        "match_id", "innings", "overid", "over_ball_num",
        "striker_name", "bowler_name", "runs",
        "stroke", "delivery_style", "extras_type",
        "extras1_runs", "extras_2_runs",
        "is_four", "is_six"
    ]].to_string(index=False)
)

print("\nALL SIX-RUN DELIVERIES NOT FLAGGED AS SIX")

display(
    ball_by_ball[
        (ball_by_ball["runs"] == 6) &
        (~ball_by_ball["is_six"])
    ][[
        "match_id", "innings", "overid", "over_ball_num",
        "striker_name", "bowler_name", "runs",
        "stroke", "delivery_style", "extras_type",
        "extras1_runs", "extras_2_runs",
        "is_four", "is_six"
    ]].to_string(index=False)
)

ALL DELIVERIES WITH NON-ZERO EXTRAS_2_RUNS


,match_id,innings,overid,over_ball_num,striker_name,bowler_name,runs,extras_type,extras1_runs,extras_2_runs,is_four,is_six,is_wicket
24453,15796,2,14,4,Lorcan Tucker,Umran Malik,0,Overstep,1,1,False,False,False
47440,20186,2,20,2,Krishnappa Gowtham,Tushar Deshpande,0,Overstep,1,1,False,False,False
50553,20199,2,10,4,Nitish Rana,Marco Jansen,0,Overstep,1,1,False,False,False
55141,20218,1,16,5,Marcus Stoinis,Kagiso Rabada,0,Overstep,1,1,False,False,False
83106,30068,1,6,2,David Warner,Mustafizur Rahman,0,Overstep,1,1,False,False,False
85169,30076,2,5,5,Sai Sudharsan,Manimaran Siddharth,0,Overstep,1,1,False,False,False
92934,31265,2,1,3,Sunil Narine,Lizaad Williams,0,Overstep,1,1,False,False,False
121459,38844,1,19,1,Jaker Ali Anik,Romario Shepherd,0,Overstep,1,1,False,False,False
122187,39811,1,6,1,Finn Allen,Abbas Afridi,0,Overstep,1,1,False,False,False
138578,40756,1,4,5,Ravichandran Ashwin,Yudhvir Singh,0,Overstep,1,1,False,False,False



RUNS DISTRIBUTION BY EXTRAS TYPE


runs,0,1,2,3,4,5,6,All
extras_type,,,,,,,,
NO_EXTRAS,68029,72043,12496,653,23180,25,12741,189167
Overstep,239,174,33,2,81,0,47,576
Penalty,1,0,0,0,0,0,0,1
byes,529,0,0,0,0,0,0,529
legbyes,2747,1,0,0,0,0,0,2748
other,128,49,18,1,42,0,26,264
wide,7714,0,0,0,0,0,0,7714
All,79387,72267,12547,656,23303,25,12814,200999



RUNS vs IS_FOUR


is_four,False,True,All
runs,,,
0,79387,0,79387
1,72267,0,72267
2,12547,0,12547
3,656,0,656
4,20,23283,23303
5,25,0,25
6,12814,0,12814
All,177716,23283,200999


... (2 outputs left out)


ALL FOUR-RUN DELIVERIES NOT FLAGGED AS FOUR


' match_id  innings  overid  over_ball_num           striker_name         bowler_name  runs              stroke delivery_style extras_type  extras1_runs  extras_2_runs  is_four  is_six\n    14113        1       1              3       Devdutt Padikkal Dushmantha Chameera     4         Cover Drive      Out Swing         NaN             6              0    False   False\n    14123        1       3              6            Jos Buttler         Umesh Yadav     4        Square drive         Yorker         NaN             6              0    False   False\n    14149        1      12              2           Devon Conway      Mitchell Marsh     4           Pull shot     Seamed Out         NaN             6              0    False   False\n    15609        2      17              8      Najibullah Zadran Blessing Muzarabani     4           Pull shot     Off Cutter         NaN             6              0    False   False\n    16425        2      13              1           Afif Hossain    Wesley


ALL SIX-RUN DELIVERIES NOT FLAGGED AS SIX


' match_id  innings  overid  over_ball_num  striker_name     bowler_name  runs      stroke delivery_style extras_type  extras1_runs  extras_2_runs  is_four  is_six\n    17396        1      19              3     Tim David  Jasprit Bumrah     6 Cover Drive      Full Toss         NaN             6              0    False   False\n    31277        1       7              5 Sai Sudharsan Ravindra Jadeja     6      Worked       Arm ball         NaN             6              0    False   False\n    40766        1      16              6 Hardik Pandya    Vyshak Kumar     6      Worked      Seamed In         NaN             6              0    False   False'

### Inspect unresolved records

In [22]:
# 1. The unexpected extras_2_runs record with no extras_type
print("NO-EXTRAS RECORD WITH NON-ZERO EXTRAS_2_RUNS")

display(
    ball_by_ball[
        ball_by_ball["extras_type"].isna() &
        (ball_by_ball["extras_2_runs"] != 0)
    ].T
)

# 2. All 36 records with non-zero extras_2_runs:
# inspect additional fields to identify the scoring convention
print("\nNON-ZERO EXTRAS_2_RUNS: EXTENDED CONTEXT")

display(
    ball_by_ball[
        ball_by_ball["extras_2_runs"] != 0
    ][[
        "match_id", "innings", "overid", "over_ball_num",
        "innings_ball_num", "delivery_ist_time",
        "striker_name", "bowler_name", "runs",
        "extras_type", "extras1_runs", "extras_2_runs",
        "dismissal_type", "dismissed_player_id",
        "stroke", "delivery_style"
    ]].to_string(index=False)
)

# 3. Inspect all 23 boundary-flag mismatches with full context
mismatch_mask = (
    ((ball_by_ball["runs"] == 4) & (~ball_by_ball["is_four"])) |
    ((ball_by_ball["runs"] == 6) & (~ball_by_ball["is_six"]))
)

print("\nBOUNDARY-FLAG MISMATCHES: FULL CONTEXT")

display(
    ball_by_ball.loc[mismatch_mask].T
)

# 4. Summarize mismatch counts by match
print("\nMISMATCH COUNTS BY MATCH")

display(
    ball_by_ball.loc[mismatch_mask]
    .groupby("match_id")
    .agg(
        mismatch_deliveries=("runs", "size"),
        runs_values=("runs", lambda x: list(x))
    )
    .sort_values("mismatch_deliveries", ascending=False)
)

NO-EXTRAS RECORD WITH NON-ZERO EXTRAS_2_RUNS


,36854
match_id,17555
innings,1
overid,5
over_ball_num,4
innings_ball_num,29
batting_team_id,22
bowling_team_id,25
striker_id,728
striker_name,Dawid Malan
non_striker_id,6272



NON-ZERO EXTRAS_2_RUNS: EXTENDED CONTEXT


' match_id  innings  overid  over_ball_num  innings_ball_num       delivery_ist_time           striker_name         bowler_name  runs extras_type  extras1_runs  extras_2_runs dismissal_type  dismissed_player_id               stroke delivery_style\n    13523        1       1              4                 4 2022-02-18 19:02:19 UTC           Ishan Kishan        Akeal Hosein     0    Overstep             1              4            NaN                  NaN           Leg Glance       Orthodox\n    13614        1      18              5               111 2022-02-15 15:01:07 UTC          Dasun Shanaka      Mitchell Starc     0       other             5              4            NaN                  NaN                  NaN         Beamer\n    14112        1      11              5                66 2022-04-10 16:25:26 UTC           David Warner  Varun Chakravarthy     0    Overstep             1              4            NaN                  NaN               Switch         Carrom\n    14127  


BOUNDARY-FLAG MISMATCHES: FULL CONTEXT


,7790,10277,16372,23576,28308,34269,36207,59426,67581,75711,75971,78514,78545,81653,95661,105337,106985,130944,132806,134818,140761,181056,197429
match_id,14113,14123,14149,15609,16425,16754,17396,20236,24350,28958,28959,29816,29817,30062,31277,34427,35429,40722,40730,40739,40766,50342,51384
innings,1,1,1,2,2,1,1,2,2,1,1,2,1,1,1,2,2,1,2,1,1,2,2
overid,1,3,12,17,13,6,19,2,6,3,5,20,6,9,7,19,5,20,3,19,16,13,13
over_ball_num,3,6,2,8,1,1,3,7,6,2,1,6,1,2,5,4,5,6,6,1,6,5,3
innings_ball_num,3,19,72,106,74,33,115,13,38,14,25,127,31,53,42,114,29,124,18,112,102,84,76
batting_team_id,3,3,1,78,21,78,20,3,18,27,72,28,28,1,3508,27,315,2,5,7,2,3,72
bowling_team_id,3509,8,7,72,72,23,18,8,28,72,27,20,20,3508,1,18,24,7,4,5,4,242,21
striker_id,6314,813,1941,2533,5882,2533,6396,6819,7897,3669,1541,2633,2687,167,23450,8661,30927,37052,6314,46066,3330,8182,2463
striker_name,Devdutt Padikkal,Jos Buttler,Devon Conway,Najibullah Zadran,Afif Hossain,Najibullah Zadran,Tim David,Yashasvi Jaiswal,Namboori Tilak Varma,Sadeera Samarawickrama,Sean Williams,Jason Holder,Roston Chase,Ajinkya Rahane,Sai Sudharsan,Matheesha Pathirana,Saiteja Mukkamalla,Naman Dhir,Devdutt Padikkal,Vipraj Nigam,Hardik Pandya,Donovan Ferreira,Ryan Burl
non_striker_id,813,6314,4755,6433,3280,3544,6518,2258,1699,3672,7046,3275,4986,6059,6288,19316,5877,6272,47,20702,37052,152,21790



MISMATCH COUNTS BY MATCH


,mismatch_deliveries,runs_values
match_id,,
14113,1,[4]
14123,1,[4]
14149,1,[4]
15609,1,[4]
16425,1,[4]
16754,1,[4]
17396,1,[6]
20236,1,[4]
24350,1,[4]


### Delivery sequence and record integrity

In [23]:
# 1. Check duplicate delivery identifiers
print("DUPLICATE MATCH-INNINGS-BALL IDENTIFIERS")

print(
    "Duplicate (match_id, innings, innings_ball_num):",
    ball_by_ball.duplicated(
        ["match_id", "innings", "innings_ball_num"]
    ).sum()
)

print(
    "Duplicate (match_id, innings, overid, over_ball_num):",
    ball_by_ball.duplicated(
        ["match_id", "innings", "overid", "over_ball_num"]
    ).sum()
)

# 2. Check innings and over number ranges
print("\nINNINGS VALUE COUNTS")
display(ball_by_ball["innings"].value_counts().sort_index())

print("\nOVER ID VALUE COUNTS")
display(ball_by_ball["overid"].value_counts().sort_index())

print("\nOVER BALL NUMBER VALUE COUNTS")
display(ball_by_ball["over_ball_num"].value_counts().sort_index())

# 3. Check innings ball number ranges
print("\nINNINGS BALL NUMBER SUMMARY")

display(
    ball_by_ball.groupby(["match_id", "innings"])["innings_ball_num"]
    .agg(["count", "min", "max", "nunique"])
    .describe()
)

# 4. Check whether innings_ball_num is sequential within each innings
sequence_check = (
    ball_by_ball
    .sort_values(["match_id", "innings", "innings_ball_num"])
    .groupby(["match_id", "innings"])["innings_ball_num"]
    .agg(
        first_ball="min",
        last_ball="max",
        unique_balls="nunique",
        delivery_rows="size"
    )
)

sequence_check["expected_unique_if_sequential"] = (
    sequence_check["last_ball"] - sequence_check["first_ball"] + 1
)

sequence_check["has_gaps_or_duplicates"] = (
    sequence_check["unique_balls"] !=
    sequence_check["expected_unique_if_sequential"]
)

print("\nINNINGS WITH NON-SEQUENTIAL BALL NUMBERS")
display(
    sequence_check[
        sequence_check["has_gaps_or_duplicates"]
    ].head(30)
)

print(
    "\nTotal innings with gaps or duplicates:",
    sequence_check["has_gaps_or_duplicates"].sum()
)

# 5. Check over-ball numbering within each over
over_sequence = (
    ball_by_ball
    .groupby(["match_id", "innings", "overid"])["over_ball_num"]
    .agg(
        delivery_rows="size",
        min_ball="min",
        max_ball="max",
        unique_ball_numbers="nunique"
    )
)

print("\nOVERS WITH MORE THAN SIX RECORDED DELIVERIES")
display(
    over_sequence[
        over_sequence["delivery_rows"] > 6
    ].head(30)
)

print(
    "\nTotal overs with more than six recorded deliveries:",
    (over_sequence["delivery_rows"] > 6).sum()
)

print("\nOVERS WITH DUPLICATE OVER-BALL NUMBERS")
display(
    over_sequence[
        over_sequence["unique_ball_numbers"] <
        over_sequence["delivery_rows"]
    ].head(30)
)

print(
    "\nTotal overs with duplicate over-ball numbers:",
    (
        over_sequence["unique_ball_numbers"] <
        over_sequence["delivery_rows"]
    ).sum()
)

DUPLICATE MATCH-INNINGS-BALL IDENTIFIERS
Duplicate (match_id, innings, innings_ball_num): 0
Duplicate (match_id, innings, overid, over_ball_num): 0

INNINGS VALUE COUNTS


,count
innings,
1,105727
2,95234
3,25
4,13



OVER ID VALUE COUNTS


,count
overid,
1,10818
2,10788
3,10645
4,10638
5,10615
6,10542
7,10441
8,10476
9,10415



OVER BALL NUMBER VALUE COUNTS


,count
over_ball_num,
1,32350
2,32257
3,32169
4,32066
5,31937
6,31791
7,6710
8,1364
9,279


... (5 outputs left out)

delivery_rows  min_ball  max_ball  \
match_id innings overid                                      
13511    1       7                   7         1         7   
                 8                   8         1         8   
13512    1       10                  8         1         8   
                 12                  7         1         7   
                 14                  7         1         7   
                 16                  7         1         7   
                 17                  7         1         7   
                 18                  7         1         7   
                 19                  7         1         7   
         2       1                   7         1         7   
                 18                  7         1         7   
                 20                  8         1         8   
13513    1       4                   7         1         7   
                 5                   7         1         7   
                 10                  7         1         7   
                 14                  7         1         7   
                 19                  7         1         7   
         2       2                   7         1         7   
                 3                   7         1         7   
                 9                   7         1         7   
                 11                  7         1         7   
                 12                  8         1         8   
                 20                  7         1         7   
13514    1       11                  8         1         8   
                 13                  7         1         7   
                 14                  7         1         7   
                 17                  7         1         7   
                 19                  9         1         9   
         2       10                  7         1         7   
                 14                  7         1         7   

                         unique_ball_numbers  
match_id innings overid                       
13511    1       7                         7  
                 8                         8  
13512    1       10                        8  
                 12                        7  
                 14                        7  
                 16                        7  
                 17                        7  
                 18                        7  
                 19                        7  
         2       1                         7  
                 18                        7  
                 20                        8  
13513    1       4                         7  
                 5                         7  
                 10                        7  
                 14                        7  
                 19                        7  
         2       2                         7  
                 3                         7  
                 9                         7  
                 11                        7  
                 12                        8  
                 20                        7  
13514    1       11                        8  
                 13                        7  
                 14                        7  
                 17                        7  
                 19                        9  
         2       10                        7  
                 14                        7


Total overs with more than six recorded deliveries: 6710

OVERS WITH DUPLICATE OVER-BALL NUMBERS


,,,delivery_rows,min_ball,max_ball,unique_ball_numbers
match_id,innings,overid,,,,



Total overs with duplicate over-ball numbers: 0


### Investigate special innings and extended overs

In [24]:
# 1. Inspect innings 3 and 4
special_innings = ball_by_ball[
    ball_by_ball["innings"].isin([3, 4])
]

special_match_ids = special_innings["match_id"].unique()

print("SPECIAL INNINGS: MATCH DETAILS")

display(
    matches[
        matches["match_id"].isin(special_match_ids)
    ][[
        "match_id", "comp_id", "match_number", "match_date",
        "stage", "home_team", "away_team", "result"
    ]].sort_values(["match_date", "match_id"])
)

print("\nSPECIAL INNINGS: DELIVERY COUNTS")

display(
    special_innings.groupby(
        ["match_id", "innings"]
    ).agg(
        delivery_count=("innings_ball_num", "size"),
        first_ball=("innings_ball_num", "min"),
        last_ball=("innings_ball_num", "max"),
        batting_teams=("batting_team_id", "nunique"),
        bowling_teams=("bowling_team_id", "nunique")
    ).reset_index()
)

# 2. Summarize overs with more than six recorded deliveries
extended_overs = (
    over_sequence[
        over_sequence["delivery_rows"] > 6
    ]
    .reset_index()
    .sort_values(
        ["delivery_rows", "max_ball"],
        ascending=False
    )
)

print("\nEXTENDED OVER SUMMARY")
print("Number of overs with more than six records:", len(extended_overs))

display(
    extended_overs["delivery_rows"].value_counts().sort_index()
)

print("\nEXAMPLES OF EXTENDED OVERS")

extended_keys = extended_overs.head(15)[
    ["match_id", "innings", "overid"]
]

extended_examples = ball_by_ball.merge(
    extended_keys,
    on=["match_id", "innings", "overid"],
    how="inner"
)

display(
    extended_examples[[
        "match_id", "innings", "overid", "over_ball_num",
        "runs", "extras_type", "extras1_runs", "extras_2_runs"
    ]].sort_values(
        ["match_id", "innings", "overid", "over_ball_num"]
    ).to_string(index=False)
)

# 3. Check extras-type distribution in extended overs
print("\nEXTRAS TYPE DISTRIBUTION IN EXTENDED OVERS")

display(
    pd.crosstab(
        extended_examples["extras_type"].fillna("NO_EXTRAS"),
        extended_examples["over_ball_num"],
        margins=True
    )
)

# 4. Check whether over_ball_num > 6 occurs only in extended overs
print("\nDELIVERIES WITH OVER_BALL_NUM GREATER THAN SIX")

late_ball_numbers = ball_by_ball[
    ball_by_ball["over_ball_num"] > 6
]

display(
    late_ball_numbers["extras_type"]
    .fillna("NO_EXTRAS")
    .value_counts()
)

print(
    "\nNumber of deliveries with over_ball_num > 6:",
    len(late_ball_numbers)
)

SPECIAL INNINGS: MATCH DETAILS


,match_id,comp_id,match_number,match_date,stage,home_team,away_team,result
478,34429,2542,3,2024-07-30,Others,Sri Lanka,India,India Tied with Srilanka (India Won the Super ...
557,40725,2725,32,2025-04-16,League,Delhi Capitals,Rajasthan Royals,Delhi Capitals Tied with Rajasthan Royals (DC ...
679,45512,3133,18,2025-09-26,Super 4,India,Sri Lanka,India won the Super Over(Match Tied)
745,48027,3289,13,2025-11-21,Semi Final,Bangladesh A,India A,Bangladesh A Won The Super Over
747,48029,3289,15,2025-11-23,Final,Bangladesh A,Pakistan A,Pakistan A Won the super over
785,50359,3298,38,2026-04-26,League,Lucknow Super Giants,Kolkata Knight Riders,Kolkata Knight Riders Won by Super over (Match...



SPECIAL INNINGS: DELIVERY COUNTS


,match_id,innings,delivery_count,first_ball,last_ball,batting_teams,bowling_teams
0,34429,3,4,1,4,1,1
1,34429,4,1,1,1,1,1
2,40725,3,6,1,6,1,1
3,40725,4,4,1,4,1,1
4,45512,3,6,1,6,1,1
5,45512,4,1,1,1,1,1
6,48027,3,2,1,2,1,1
7,48027,4,2,1,2,1,1
8,48029,3,4,1,4,1,1
9,48029,4,4,1,4,1,1



EXTENDED OVER SUMMARY
Number of overs with more than six records: 6710


,count
delivery_rows,
7,5346
8,1085
9,222
10,42
11,13
13,2


... (3 outputs left out)

over_ball_num,1,2,3,4,5,6,7,8,9,10,11,12,13,All
extras_type,,,,,,,,,,,,,,
NO_EXTRAS,9,8,5,5,2,4,7,11,9,12,15,0,2,89
Overstep,1,1,3,2,1,3,1,0,0,0,0,0,0,12
legbyes,0,0,1,0,0,0,0,0,0,0,0,0,0,1
other,0,0,0,0,1,0,0,0,0,0,0,0,0,1
wide,5,6,6,8,11,8,7,4,6,3,0,2,0,66
All,15,15,15,15,15,15,15,15,15,15,15,2,2,169



DELIVERIES WITH OVER_BALL_NUM GREATER THAN SIX


,count
extras_type,
NO_EXTRAS,7735
wide,473
legbyes,142
byes,37
Overstep,22
other,20



Number of deliveries with over_ball_num > 6: 8429


### Verify delivery numbering against illegal deliveries

In [25]:
# Create a temporary diagnostic table; original data remains unchanged
over_audit = (
    ball_by_ball
    .groupby(["match_id", "innings", "overid"])
    .agg(
        delivery_rows=("over_ball_num", "size"),
        max_sequence=("over_ball_num", "max"),
        wide_count=("extras_type", lambda x: (x == "wide").sum()),
        overstep_count=("extras_type", lambda x: (x == "Overstep").sum()),
        other_extras_count=("extras_type", lambda x: x.isin(["other", "legbyes", "byes"]).sum()),
        first_sequence=("over_ball_num", "min"),
        unique_sequences=("over_ball_num", "nunique")
    )
    .reset_index()
)

over_audit["known_illegal_deliveries"] = (
    over_audit["wide_count"] + over_audit["overstep_count"]
)

over_audit["deliveries_after_known_illegal"] = (
    over_audit["delivery_rows"] - over_audit["known_illegal_deliveries"]
)

# 1. Overall summary
print("OVER SEQUENCING SUMMARY")

display(
    over_audit[[
        "delivery_rows", "wide_count", "overstep_count",
        "other_extras_count", "deliveries_after_known_illegal"
    ]].describe()
)

# 2. Compare overs with more than six deliveries
print("\nOVERS WITH MORE THAN SIX DELIVERIES: SUMMARY")

display(
    over_audit[
        over_audit["delivery_rows"] > 6
    ][[
        "delivery_rows", "wide_count", "overstep_count",
        "deliveries_after_known_illegal"
    ]].describe()
)

# 3. Check whether over sequence numbers are continuous from 1
print("\nOVERS WITH NON-CONTINUOUS OVER-BALL SEQUENCES")

display(
    over_audit[
        (over_audit["first_sequence"] != 1) |
        (over_audit["unique_sequences"] != over_audit["delivery_rows"]) |
        (over_audit["max_sequence"] != over_audit["delivery_rows"])
    ].head(30)
)

# 4. Check the relationship between sequence length and known illegal deliveries
print("\nDELIVERY COUNT vs KNOWN ILLEGAL DELIVERIES")

display(
    pd.crosstab(
        over_audit["delivery_rows"],
        over_audit["known_illegal_deliveries"],
        margins=True
    )
)

# 5. Check whether any over has more than six deliveries
# without any wide or overstep
print("\nOVERS WITH MORE THAN SIX DELIVERIES BUT NO WIDES/OVERSTEPS")

display(
    over_audit[
        (over_audit["delivery_rows"] > 6) &
        (over_audit["known_illegal_deliveries"] == 0)
    ].head(30)
)

print(
    "Count:",
    (
        (over_audit["delivery_rows"] > 6) &
        (over_audit["known_illegal_deliveries"] == 0)
    ).sum()
)

OVER SEQUENCING SUMMARY


,delivery_rows,wide_count,overstep_count,other_extras_count,deliveries_after_known_illegal
count,"32,350.000","32,350.000","32,350.000","32,350.000","32,350.000"
mean,6.213,0.238,0.018,0.109,5.957
std,0.717,0.539,0.139,0.336,0.431
min,1.000,0.000,0.000,0.000,0.000
25%,6.000,0.000,0.000,0.000,6.000
50%,6.000,0.000,0.000,0.000,6.000
75%,6.000,0.000,0.000,0.000,6.000
max,13.000,7.000,3.000,4.000,9.000



OVERS WITH MORE THAN SIX DELIVERIES: SUMMARY


,delivery_rows,wide_count,overstep_count,deliveries_after_known_illegal
count,"6,710.000","6,710.000","6,710.000","6,710.000"
mean,7.256,1.135,0.084,6.037
std,0.571,0.609,0.293,0.200
min,7.000,0.000,0.000,5.000
25%,7.000,1.000,0.000,6.000
50%,7.000,1.000,0.000,6.000
75%,7.000,1.000,0.000,6.000
max,13.000,7.000,3.000,9.000



OVERS WITH NON-CONTINUOUS OVER-BALL SEQUENCES


,match_id,innings,overid,delivery_rows,max_sequence,wide_count,overstep_count,other_extras_count,first_sequence,unique_sequences,known_illegal_deliveries,deliveries_after_known_illegal



DELIVERY COUNT vs KNOWN ILLEGAL DELIVERIES


known_illegal_deliveries,0,1,2,3,4,5,7,All
delivery_rows,,,,,,,,
1,92,1,0,0,0,0,0,93
2,82,6,0,0,0,0,0,88
3,84,18,1,0,0,0,0,103
4,108,19,2,0,0,0,0,129
5,118,23,5,0,0,0,0,146
6,25056,22,2,1,0,0,0,25081
7,145,5199,2,0,0,0,0,5346
8,5,63,1016,1,0,0,0,1085
9,1,2,21,198,0,0,0,222



OVERS WITH MORE THAN SIX DELIVERIES BUT NO WIDES/OVERSTEPS


,match_id,innings,overid,delivery_rows,max_sequence,wide_count,overstep_count,other_extras_count,first_sequence,unique_sequences,known_illegal_deliveries,deliveries_after_known_illegal
87,13513,1,10,7,7,0,0,1,1,7,0,7
196,13515,2,19,7,7,0,0,2,1,7,0,7
413,13614,1,18,7,7,0,0,1,1,7,0,7
417,13614,2,2,7,7,0,0,3,1,7,0,7
708,14098,2,20,7,7,0,0,1,1,7,0,7
804,14101,1,17,7,7,0,0,1,1,7,0,7
1055,14107,2,16,7,7,0,0,1,1,7,0,7
1089,14108,2,14,7,7,0,0,1,1,7,0,7
1091,14108,2,16,7,7,0,0,1,1,7,0,7
1172,14110,2,17,7,7,0,0,1,1,7,0,7


Count: 151


### Investigate the 151 exceptional overs

In [26]:
# Identify overs with >6 deliveries and no wide/overstep
exceptional_keys = over_audit[
    (over_audit["delivery_rows"] > 6) &
    (over_audit["known_illegal_deliveries"] == 0)
][["match_id", "innings", "overid"]]

exceptional_deliveries = ball_by_ball.merge(
    exceptional_keys,
    on=["match_id", "innings", "overid"],
    how="inner"
)

# 1. How many deliveries and extras categories?
print("EXCEPTIONAL OVERS: EXTRAS TYPE DISTRIBUTION")

display(
    exceptional_deliveries["extras_type"]
    .fillna("NO_EXTRAS")
    .value_counts()
)

print("\nEXCEPTIONAL OVERS: EXTRAS TYPE BY OVER")

display(
    pd.crosstab(
        exceptional_deliveries["extras_type"].fillna("NO_EXTRAS"),
        exceptional_deliveries["over_ball_num"],
        margins=True
    )
)

# 2. Inspect all exceptional overs, grouped by over
print("\nEXCEPTIONAL OVER DETAILS")

display(
    exceptional_deliveries[[
        "match_id", "innings", "overid", "over_ball_num",
        "striker_name", "bowler_name", "runs",
        "extras_type", "extras1_runs", "extras_2_runs",
        "is_wicket", "delivery_ist_time"
    ]].sort_values(
        ["match_id", "innings", "overid", "over_ball_num"]
    ).to_string(index=False)
)

# 3. Link exceptional overs to competition and season
exceptional_match_summary = (
    exceptional_deliveries[["match_id"]]
    .drop_duplicates()
    .merge(
        matches[["match_id", "comp_id", "match_date"]],
        on="match_id",
        how="left"
    )
    .merge(
        competitions[["comp_id", "comp_name", "season", "match_format"]],
        on="comp_id",
        how="left"
    )
)

print("\nEXCEPTIONAL OVERS BY COMPETITION AND SEASON")

display(
    exceptional_match_summary.groupby(
        ["comp_name", "season", "match_format"],
        dropna=False
    )
    .agg(
        affected_matches=("match_id", "nunique")
    )
    .sort_values("affected_matches", ascending=False)
    .head(30)
)

# 4. Summarize how many exceptional overs occur in each match
print("\nMATCHES WITH THE MOST EXCEPTIONAL OVERS")

display(
    exceptional_keys.groupby("match_id")
    .size()
    .sort_values(ascending=False)
    .head(20)
)

EXCEPTIONAL OVERS: EXTRAS TYPE DISTRIBUTION


,count
extras_type,
NO_EXTRAS,885
other,157
legbyes,14
byes,8



EXCEPTIONAL OVERS: EXTRAS TYPE BY OVER


over_ball_num,1,2,3,4,5,6,7,8,9,All
extras_type,,,,,,,,,,
NO_EXTRAS,119,125,128,117,122,121,147,5,1,885
byes,2,2,0,2,0,2,0,0,0,8
legbyes,1,3,3,1,2,1,3,0,0,14
other,29,21,20,31,27,27,1,1,0,157
All,151,151,151,151,151,151,151,6,1,1064



EXCEPTIONAL OVER DETAILS


' match_id  innings  overid  over_ball_num         striker_name           bowler_name  runs extras_type  extras1_runs  extras_2_runs  is_wicket       delivery_ist_time\n    13513        1      10              1        Rovman Powell           Tymal Mills     0       other             5              0      False 2022-01-27 02:47:10 UTC\n    13513        1      10              2        Rovman Powell           Tymal Mills     6         NaN             6              0      False 2022-01-27 02:48:39 UTC\n    13513        1      10              3        Rovman Powell           Tymal Mills     1         NaN             6              0      False 2022-01-27 03:02:56 UTC\n    13513        1      10              4      Nicholas Pooran           Tymal Mills     2         NaN             6              0      False 2022-01-27 03:03:23 UTC\n    13513        1      10              5      Nicholas Pooran           Tymal Mills     1         NaN             6              0      False 2022-01-27 03:03


EXCEPTIONAL OVERS BY COMPETITION AND SEASON


,,,affected_matches
comp_name,season,match_format,
IPL 2023,2023,T20,18
IPL 2022,2022,T20,14
IPL 2026,2026,T20,11
IPL 2024,2024,T20,11
IPL 2025,2025,T20,10
Asia Cup 2025,2025,T20I,4
South Africa in Australia T20I Series 2025,2025,T20I,3
Zimbabwe in Bangladesh T20I Series 2024,2024,T20I,3
England in West Indies T20I Series 2022,2022,T20I,2



MATCHES WITH THE MOST EXCEPTIONAL OVERS


,0
match_id,
36096,3
14108,2
13614,2
50390,2
50348,2
20207,2
15796,2
30063,2
20249,2


In [53]:
# Rebuild over-level audit independently

over_audit = (
    ball_by_ball.groupby(["match_id", "innings", "overid"])
    .agg(
        delivery_rows=("over_ball_num", "size"),
        max_sequence=("over_ball_num", "max"),
        wide_count=("extras_type", lambda x: (x == "wide").sum()),
        overstep_count=("extras_type", lambda x: (x == "Overstep").sum()),
        other_extras_count=(
            "extras_type",
            lambda x: x.isin(["other", "legbyes", "byes"]).sum()
        )
    )
    .reset_index()
)

# More than six deliveries, but no recorded wides or oversteps
exceptional_overs = over_audit[
    (over_audit["delivery_rows"] > 6) &
    (over_audit["wide_count"] == 0) &
    (over_audit["overstep_count"] == 0)
].copy()

print("Exceptional overs:", len(exceptional_overs))
print("Total deliveries in exceptional overs:",
      exceptional_overs["delivery_rows"].sum())

print("\nDISTRIBUTION BY DELIVERY COUNT")
display(
    exceptional_overs["delivery_rows"]
    .value_counts()
    .sort_index()
    .to_frame("overs")
)

print("\nEXTRAS CLASSIFICATION WITHIN THESE OVERS")

exceptional_keys = exceptional_overs[
    ["match_id", "innings", "overid"]
]

exceptional_deliveries = ball_by_ball.merge(
    exceptional_keys,
    on=["match_id", "innings", "overid"],
    how="inner"
)

display(
    exceptional_deliveries["extras_type"]
    .fillna("NO_EXTRAS")
    .value_counts()
    .to_frame("delivery_count")
)

Exceptional overs: 151
Total deliveries in exceptional overs: 1064

DISTRIBUTION BY DELIVERY COUNT


,overs
delivery_rows,
7,145
8,5
9,1



EXTRAS CLASSIFICATION WITHIN THESE OVERS


,delivery_count
extras_type,
NO_EXTRAS,885
other,157
legbyes,14
byes,8


In [54]:
# 1. Inspect all overs with 8 or 9 delivery records
print("OVERS WITH 8 OR 9 DELIVERIES")

long_overs = exceptional_overs[
    exceptional_overs["delivery_rows"] >= 8
].sort_values(["match_id", "innings", "overid"])

display(long_overs.to_string(index=False))

# 2. Inspect every delivery in those overs
long_keys = long_overs[["match_id", "innings", "overid"]]

long_deliveries = ball_by_ball.merge(
    long_keys,
    on=["match_id", "innings", "overid"],
    how="inner"
)

print("\nDELIVERY DETAILS FOR 8- AND 9-DELIVERY OVERS")

display(
    long_deliveries[[
        "match_id", "innings", "overid", "over_ball_num",
        "striker_name", "bowler_name", "runs",
        "extras_type", "extras1_runs", "extras_2_runs",
        "is_wicket"
    ]].sort_values(
        ["match_id", "innings", "overid", "over_ball_num"]
    ).to_string(index=False)
)

# 3. Inspect 'other' extras in all 151 exceptional overs
print("\nOTHER EXTRAS IN EXCEPTIONAL OVERS")

display(
    exceptional_deliveries[
        exceptional_deliveries["extras_type"] == "other"
    ][[
        "match_id", "innings", "overid", "over_ball_num",
        "striker_name", "bowler_name", "runs",
        "extras1_runs", "extras_2_runs", "is_wicket"
    ]].to_string(index=False)
)

OVERS WITH 8 OR 9 DELIVERIES


' match_id  innings  overid  delivery_rows  max_sequence  wide_count  overstep_count  other_extras_count\n    25158        2      13              8             8           0               0                   2\n    32618        2       2              8             8           0               0                   2\n    36096        1      19              9             9           0               0                   3\n    36560        1       5              8             8           0               0                   2\n    45391        2      17              8             8           0               0                   2\n    50359        2      20              8             8           0               0                   4'


DELIVERY DETAILS FOR 8- AND 9-DELIVERY OVERS


' match_id  innings  overid  over_ball_num     striker_name      bowler_name  runs extras_type  extras1_runs  extras_2_runs  is_wicket\n    25158        2      13              1      Dawid Malan  Lockie Ferguson     0         NaN             6              0       True\n    25158        2      13              2 Liam Livingstone  Lockie Ferguson     2         NaN             6              0      False\n    25158        2      13              3 Liam Livingstone  Lockie Ferguson     1       other             5              0      False\n    25158        2      13              4      Harry Brook  Lockie Ferguson     2       other             5              0      False\n    25158        2      13              5      Harry Brook  Lockie Ferguson     1         NaN             6              0      False\n    25158        2      13              6 Liam Livingstone  Lockie Ferguson     1         NaN             6              0      False\n    25158        2      13              7      Harry B


OTHER EXTRAS IN EXCEPTIONAL OVERS


' match_id  innings  overid  over_ball_num         striker_name           bowler_name  runs  extras1_runs  extras_2_runs  is_wicket\n    13513        1      10              1        Rovman Powell           Tymal Mills     0             5              0      False\n    13515        2      19              5         Sam Billings      Sheldon Cottrell     2             5              0      False\n    13614        1      18              5        Dasun Shanaka        Mitchell Starc     0             5              4      False\n    13614        2       2              1          Ashton Agar        Nuwan Thushara     0             5              0      False\n    14098        2      20              1    Bhuvneshwar Kumar   Nathan Coulter Nile     0             5              0      False\n    14101        1      17              4        Kagiso Rabada           Tim Southee     0             5              0      False\n    14107        2      16              5      Patrick Cummins           Da

In [55]:
# 1. Distribution of runs and additional runs for "other"
other_audit = ball_by_ball[
    ball_by_ball["extras_type"] == "other"
].copy()

print("OTHER EXTRAS — RUNS DISTRIBUTION")
display(
    other_audit.groupby(
        ["runs", "extras_2_runs"], dropna=False
    ).size().reset_index(name="delivery_count")
    .sort_values("delivery_count", ascending=False)
    .to_string(index=False)
)

# 2. Delivery sequence positions of "other" extras
print("\nOTHER EXTRAS BY OVER SEQUENCE NUMBER")
display(
    other_audit["over_ball_num"]
    .value_counts()
    .sort_index()
    .to_frame("delivery_count")
)

# 3. Other extras associated with wickets or boundaries
print("\nOTHER EXTRAS — WICKET AND BOUNDARY FLAGS")
display(
    other_audit.groupby(
        ["is_wicket", "is_four", "is_six"]
    ).size().reset_index(name="delivery_count")
    .sort_values("delivery_count", ascending=False)
    .to_string(index=False)
)

# 4. Competition and season distribution
other_with_match = other_audit.merge(
    matches[["match_id", "comp_id", "match_date"]],
    on="match_id",
    how="left"
).merge(
    competitions[["comp_id", "comp_name", "season"]],
    on="comp_id",
    how="left"
)

print("\nOTHER EXTRAS BY COMPETITION AND SEASON")
display(
    other_with_match.groupby(
        ["comp_name", "season"]
    ).size().reset_index(name="delivery_count")
    .sort_values("delivery_count", ascending=False)
    .head(25)
    .to_string(index=False)
)

OTHER EXTRAS — RUNS DISTRIBUTION


' runs  extras_2_runs  delivery_count\n    0              0             114\n    1              0              49\n    4              0              42\n    6              0              26\n    2              0              18\n    0              1               7\n    0              4               7\n    3              0               1'


OTHER EXTRAS BY OVER SEQUENCE NUMBER


,delivery_count
over_ball_num,
1,44
2,35
3,33
4,44
5,47
6,41
7,14
8,5
9,1



OTHER EXTRAS — WICKET AND BOUNDARY FLAGS


' is_wicket  is_four  is_six  delivery_count\n     False    False   False             193\n     False     True   False              42\n     False    False    True              26\n      True    False   False               3'


OTHER EXTRAS BY COMPETITION AND SEASON


'                                              comp_name  season  delivery_count\n                                               IPL 2023    2023              36\n                                               IPL 2022    2022              25\n                                               IPL 2026    2026              24\n                                               IPL 2024    2024              19\n                                               IPL 2025    2025              19\n                            Mens Twenty20 Asia Cup 2022    2022               5\n           South Africa in West Indies T20I Series 2024    2024               5\n    Bangladesh in United Arab Emirates T20I Series 2025    2025               5\n                      India in England T20I Series 2026    2026               4\n                Zimbabwe in Bangladesh T20I Series 2024    2024               4\n              South Africa in India T20I Series 2022/23 2022/23               4\n                           

In [56]:
# Identify deliveries after the sixth sequence position
exceptional_deliveries["after_sixth"] = (
    exceptional_deliveries["over_ball_num"] > 6
)

# Classify potentially relevant extras
exceptional_deliveries["other_extra_category"] = (
    exceptional_deliveries["extras_type"].isin(
        ["other", "legbyes", "byes"]
    )
)

# Summarise each exceptional over
exceptional_over_check = (
    exceptional_deliveries.groupby(
        ["match_id", "innings", "overid"]
    )
    .apply(
        lambda g: pd.Series({
            "delivery_rows": len(g),
            "deliveries_after_six": g["after_sixth"].sum(),
            "other_extras_total": g["other_extra_category"].sum(),
            "other_extras_after_six": (
                g["after_sixth"] & g["other_extra_category"]
            ).sum(),
            "other_extras_within_first_six": (
                (~g["after_sixth"]) & g["other_extra_category"]
            ).sum()
        }),
        include_groups=False
    )
    .reset_index()
)

exceptional_over_check["unexplained_after_six"] = (
    exceptional_over_check["deliveries_after_six"] -
    exceptional_over_check["other_extras_after_six"]
)

print("SUMMARY OF ADDITIONAL DELIVERIES")
display(
    exceptional_over_check[
        ["deliveries_after_six", "other_extras_after_six",
         "unexplained_after_six"]
    ].sum().to_frame("total")
)

print("\nDISTRIBUTION OF EXPLANATION STATUS")
exceptional_over_check["explanation_status"] = np.select(
    [
        exceptional_over_check["unexplained_after_six"] == 0,
        exceptional_over_check["unexplained_after_six"] > 0
    ],
    [
        "All post-sixth deliveries have other/byes/legbyes classification",
        "One or more post-sixth deliveries lack that classification"
    ],
    default="Unexpected negative difference"
)

display(
    exceptional_over_check["explanation_status"]
    .value_counts()
    .to_frame("overs")
)

print("\nOVERS WITH UNEXPLAINED POST-SIXTH DELIVERIES")
display(
    exceptional_over_check[
        exceptional_over_check["unexplained_after_six"] > 0
    ].sort_values("unexplained_after_six", ascending=False)
    .to_string(index=False)
)

SUMMARY OF ADDITIONAL DELIVERIES


,total
deliveries_after_six,158
other_extras_after_six,5
unexplained_after_six,153



DISTRIBUTION OF EXPLANATION STATUS


,overs
explanation_status,
One or more post-sixth deliveries lack that classification,148
All post-sixth deliveries have other/byes/legbyes classification,3



OVERS WITH UNEXPLAINED POST-SIXTH DELIVERIES


' match_id  innings  overid  delivery_rows  deliveries_after_six  other_extras_total  other_extras_after_six  other_extras_within_first_six  unexplained_after_six                                         explanation_status\n    36560        1       5              8                     2                   2                       0                              2                      2 One or more post-sixth deliveries lack that classification\n    45391        2      17              8                     2                   2                       0                              2                      2 One or more post-sixth deliveries lack that classification\n    25158        2      13              8                     2                   2                       0                              2                      2 One or more post-sixth deliveries lack that classification\n    32618        2       2              8                     2                   2                       0    

In [57]:
# All deliveries after the sixth sequence position
post_sixth = exceptional_deliveries[
    exceptional_deliveries["over_ball_num"] > 6
].copy()

print("TOTAL POST-SIXTH DELIVERIES:", len(post_sixth))

# Add match and competition context
post_sixth = post_sixth.merge(
    matches[["match_id", "match_date", "home_team", "away_team", "result", "comp_id"]],
    on="match_id",
    how="left"
).merge(
    competitions[["comp_id", "comp_name", "season"]],
    on="comp_id",
    how="left"
)

print("\nALL POST-SIXTH DELIVERY RECORDS")

display(
    post_sixth[[
        "match_id", "match_date", "comp_name", "season",
        "innings", "overid", "over_ball_num",
        "striker_name", "bowler_name",
        "runs", "extras_type", "extras1_runs",
        "extras_2_runs", "is_wicket", "result"
    ]].sort_values(
        ["match_id", "innings", "overid", "over_ball_num"]
    ).to_string(index=False)
)

print("\nPOST-SIXTH EXTRAS DISTRIBUTION")

display(
    post_sixth["extras_type"]
    .fillna("NO_EXTRAS")
    .value_counts()
    .to_frame("delivery_count")
)

TOTAL POST-SIXTH DELIVERIES: 158

ALL POST-SIXTH DELIVERY RECORDS


' match_id match_date                                               comp_name  season  innings  overid  over_ball_num         striker_name           bowler_name  runs extras_type  extras1_runs  extras_2_runs  is_wicket                                               result\n    13513 2022-01-26                 England in West Indies T20I Series 2022    2022        1      10              7        Rovman Powell           Tymal Mills     1         NaN             6              0      False                           West Indies Won by 20 runs\n    13515 2022-01-30                 England in West Indies T20I Series 2022    2022        2      19              7         Sam Billings      Sheldon Cottrell     1         NaN             6              0      False                           West Indies Won by 17 runs\n    13614 2022-02-15                 Sri Lanka in Australia T20I Series 2022    2022        1      18              7  Chamika Karunaratne        Mitchell Starc     1         NaN      


POST-SIXTH EXTRAS DISTRIBUTION


,delivery_count
extras_type,
NO_EXTRAS,153
legbyes,3
other,2


In [58]:
# Representative exceptional overs: 7, 8 and 9 deliveries
sample_match_ids = [13513, 25158, 36096, 50359]

sample_keys = exceptional_overs[
    exceptional_overs["match_id"].isin(sample_match_ids)
][["match_id", "innings", "overid"]]

sample_rows = ball_by_ball.merge(
    sample_keys,
    on=["match_id", "innings", "overid"],
    how="inner"
)

# Include the first two deliveries of the following over
next_over_keys = sample_keys.copy()
next_over_keys["overid"] += 1

next_over_rows = ball_by_ball.merge(
    next_over_keys,
    on=["match_id", "innings", "overid"],
    how="inner"
)

sequence_context = pd.concat(
    [sample_rows, next_over_rows],
    ignore_index=True
).sort_values(["match_id", "innings", "overid", "over_ball_num"])

print("DELIVERY SEQUENCE AROUND SELECTED EXCEPTIONAL OVERS")

display(
    sequence_context[[
        "match_id", "innings", "overid", "over_ball_num",
        "innings_ball_num", "striker_name", "bowler_name",
        "runs", "extras_type", "extras_2_runs",
        "is_wicket", "delivery_ist_time"
    ]].to_string(index=False)
)

DELIVERY SEQUENCE AROUND SELECTED EXCEPTIONAL OVERS


' match_id  innings  overid  over_ball_num  innings_ball_num     striker_name      bowler_name  runs extras_type  extras_2_runs  is_wicket       delivery_ist_time\n    13513        1      10              1                57    Rovman Powell      Tymal Mills     0       other              0      False 2022-01-27 02:47:10 UTC\n    13513        1      10              2                58    Rovman Powell      Tymal Mills     6         NaN              0      False 2022-01-27 02:48:39 UTC\n    13513        1      10              3                59    Rovman Powell      Tymal Mills     1         NaN              0      False 2022-01-27 03:02:56 UTC\n    13513        1      10              4                60  Nicholas Pooran      Tymal Mills     2         NaN              0      False 2022-01-27 03:03:23 UTC\n    13513        1      10              5                61  Nicholas Pooran      Tymal Mills     1         NaN              0      False 2022-01-27 03:03:46 UTC\n    13513        1   

In [59]:
# Focus on the transition around the over boundary
boundary_context = sequence_context[
    (
        (sequence_context["over_ball_num"] >= 6) &
        (sequence_context["overid"].isin(sample_keys["overid"].unique()))
    ) |
    (sequence_context["over_ball_num"] <= 2)
].copy()

display(
    boundary_context[[
        "match_id", "innings", "overid", "over_ball_num",
        "innings_ball_num", "striker_name", "bowler_name",
        "runs", "extras_type", "extras_2_runs",
        "is_wicket", "delivery_ist_time"
    ]].sort_values(
        ["match_id", "innings", "overid", "over_ball_num"]
    ).to_string(index=False)
)

' match_id  innings  overid  over_ball_num  innings_ball_num     striker_name      bowler_name  runs extras_type  extras_2_runs  is_wicket       delivery_ist_time\n    13513        1      10              1                57    Rovman Powell      Tymal Mills     0       other              0      False 2022-01-27 02:47:10 UTC\n    13513        1      10              2                58    Rovman Powell      Tymal Mills     6         NaN              0      False 2022-01-27 02:48:39 UTC\n    13513        1      10              6                62    Rovman Powell      Tymal Mills     0         NaN              0      False 2022-01-27 03:04:05 UTC\n    13513        1      10              7                63    Rovman Powell      Tymal Mills     1         NaN              0      False 2022-01-27 03:04:23 UTC\n    13513        1      11              1                64    Rovman Powell      Adil Rashid     3         NaN              0      False 2022-01-27 03:05:05 UTC\n    13513        1   

In [60]:
# Check whether the bowler changes immediately after exceptional overs

exceptional_keys = exceptional_overs[
    ["match_id", "innings", "overid"]
].copy()

exceptional_keys["next_overid"] = exceptional_keys["overid"] + 1

current_bowlers = (
    ball_by_ball.merge(
        exceptional_keys,
        on=["match_id", "innings", "overid"],
        how="inner"
    )
    .groupby(["match_id", "innings", "overid"])
    .agg(
        last_bowler=("bowler_id", "last"),
        last_bowler_name=("bowler_name", "last"),
        last_sequence=("over_ball_num", "max")
    )
    .reset_index()
)

next_bowlers = (
    ball_by_ball.groupby(["match_id", "innings", "overid"])
    .agg(
        next_bowler=("bowler_id", "first"),
        next_bowler_name=("bowler_name", "first")
    )
    .reset_index()
)

bowler_transition = current_bowlers.merge(
    exceptional_keys,
    on=["match_id", "innings", "overid"],
    how="left"
).merge(
    next_bowlers,
    left_on=["match_id", "innings", "next_overid"],
    right_on=["match_id", "innings", "overid"],
    how="left",
    suffixes=("", "_next")
)

bowler_transition["bowler_changed"] = (
    bowler_transition["last_bowler"] !=
    bowler_transition["next_bowler"]
)

print("BOWLER TRANSITION SUMMARY")
display(
    bowler_transition["bowler_changed"]
    .value_counts(dropna=False)
    .to_frame("overs")
)

print("\nEXCEPTIONS WHERE BOWLER DID NOT CHANGE")
display(
    bowler_transition[
        bowler_transition["bowler_changed"] == False
    ].to_string(index=False)
)

BOWLER TRANSITION SUMMARY


,overs
bowler_changed,
True,151



EXCEPTIONS WHERE BOWLER DID NOT CHANGE


'Empty DataFrame\nColumns: [match_id, innings, overid, last_bowler, last_bowler_name, last_sequence, next_overid, overid_next, next_bowler, next_bowler_name, bowler_changed]\nIndex: []'

### Wicket and dismissal consistency

In [27]:
# 1. Compare wicket flag with dismissal fields
print("IS_WICKET vs DISMISSAL TYPE PRESENCE")

display(
    pd.crosstab(
        ball_by_ball["is_wicket"],
        ball_by_ball["dismissal_type"].notna(),
        margins=True
    )
)

print("\nIS_WICKET vs DISMISSED PLAYER PRESENCE")

display(
    pd.crosstab(
        ball_by_ball["is_wicket"],
        ball_by_ball["dismissed_player_id"].notna(),
        margins=True
    )
)

# 2. Dismissal type distribution
print("\nDISMISSAL TYPE DISTRIBUTION")

display(
    ball_by_ball["dismissal_type"]
    .value_counts(dropna=False)
)

# 3. Check wicket records with missing dismissal information
print("\nWICKET RECORDS WITH MISSING DISMISSAL TYPE")

display(
    ball_by_ball[
        ball_by_ball["is_wicket"] &
        ball_by_ball["dismissal_type"].isna()
    ][[
        "match_id", "innings", "overid", "over_ball_num",
        "striker_name", "non_striker_name", "bowler_name",
        "is_wicket", "dismissal_type_id",
        "dismissal_type", "dismissed_player_id"
    ]].head(30)
)

# 4. Check non-wicket records with dismissal information
print("\nNON-WICKET RECORDS WITH DISMISSAL TYPE")

display(
    ball_by_ball[
        (~ball_by_ball["is_wicket"]) &
        ball_by_ball["dismissal_type"].notna()
    ][[
        "match_id", "innings", "overid", "over_ball_num",
        "striker_name", "bowler_name",
        "is_wicket", "dismissal_type_id",
        "dismissal_type", "dismissed_player_id"
    ]].head(30)
)

# 5. Check dismissal type ID and name consistency
print("\nDISMISSAL TYPE ID vs DISMISSAL TYPE")

display(
    pd.crosstab(
        ball_by_ball["dismissal_type"].fillna("MISSING"),
        ball_by_ball["dismissal_type_id"].fillna(-1),
        margins=True
    )
)

# 6. Compare dismissed player with striker and non-striker
wicket_rows = ball_by_ball[
    ball_by_ball["is_wicket"]
].copy()

print("\nDISMISSED PLAYER MATCH WITH STRIKER / NON-STRIKER")

display(
    pd.DataFrame({
        "comparison": [
            "Dismissed player equals striker",
            "Dismissed player equals non-striker",
            "Dismissed player equals neither",
            "Dismissed player ID missing"
        ],
        "count": [
            (wicket_rows["dismissed_player_id"] == wicket_rows["striker_id"]).sum(),
            (wicket_rows["dismissed_player_id"] == wicket_rows["non_striker_id"]).sum(),
            (
                wicket_rows["dismissed_player_id"].notna() &
                (wicket_rows["dismissed_player_id"] != wicket_rows["striker_id"]) &
                (wicket_rows["dismissed_player_id"] != wicket_rows["non_striker_id"])
            ).sum(),
            wicket_rows["dismissed_player_id"].isna().sum()
        ]
    })
)

IS_WICKET vs DISMISSAL TYPE PRESENCE


dismissal_type,False,True,All
is_wicket,,,
False,190118,0,190118
True,0,10881,10881
All,190118,10881,200999



IS_WICKET vs DISMISSED PLAYER PRESENCE


dismissed_player_id,False,True,All
is_wicket,,,
False,190106,12,190118
True,1,10880,10881
All,190107,10892,200999



DISMISSAL TYPE DISTRIBUTION


,count
dismissal_type,
NaN,190118
Caught,7557
Bowled,1775
LBW,709
RunOut,600
Stump,214
Hit Wicket,17
Retire Out,6
Obstracting the field,2


... (4 outputs left out)


DISMISSAL TYPE ID vs DISMISSAL TYPE


dismissal_type_id,-1.000,1.000,2.000,3.000,4.000,5.000,6.000,9.000,10.000,All
dismissal_type,,,,,,,,,,
Bowled,0,1775,0,0,0,0,0,0,0,1775
Caught,0,0,7557,0,0,0,0,0,0,7557
Hit Wicket,0,0,0,0,0,0,17,0,0,17
LBW,0,0,0,709,0,0,0,0,0,709
MISSING,190118,0,0,0,0,0,0,0,0,190118
Obstracting the field,0,0,0,0,0,0,0,0,2,2
Obstructing the field,0,0,0,0,0,0,0,0,1,1
Retire Out,0,0,0,0,0,0,0,6,0,6
RunOut,0,0,0,0,600,0,0,0,0,600



DISMISSED PLAYER MATCH WITH STRIKER / NON-STRIKER


,comparison,count
0,Dismissed player equals striker,10606
1,Dismissed player equals non-striker,274
2,Dismissed player equals neither,0
3,Dismissed player ID missing,1


### Inspect the remaining wicket anomalies

In [28]:
# 1. The single wicket record with missing dismissed_player_id
print("WICKET WITH MISSING DISMISSED PLAYER ID")

display(
    ball_by_ball[
        ball_by_ball["is_wicket"] &
        ball_by_ball["dismissed_player_id"].isna()
    ].T
)

# 2. Inspect both spellings of dismissal type ID 10
print("\nDISMISSAL TYPE ID 10 RECORDS")

display(
    ball_by_ball[
        ball_by_ball["dismissal_type_id"] == 10
    ][[
        "match_id", "innings", "overid", "over_ball_num",
        "striker_id", "striker_name",
        "non_striker_id", "non_striker_name",
        "dismissed_player_id",
        "dismissal_type_id", "dismissal_type"
    ]].to_string(index=False)
)

# 3. Dismissal type counts by innings
print("\nDISMISSAL TYPE BY INNINGS")

display(
    pd.crosstab(
        ball_by_ball["dismissal_type"].fillna("NO_WICKET"),
        ball_by_ball["innings"],
        margins=True
    )
)

# 4. Check whether dismissed player IDs exist in player mapping
dismissed_ids = set(
    ball_by_ball["dismissed_player_id"].dropna().astype(int).unique()
)

mapped_ids = set(
    competition_team_player_mapping["player_id"].unique()
)

print("\nDISMISSED PLAYER MAPPING COVERAGE")
print("Unique dismissed player IDs:", len(dismissed_ids))
print("IDs present in mapping:", len(dismissed_ids & mapped_ids))
print("IDs missing from mapping:", len(dismissed_ids - mapped_ids))

if dismissed_ids - mapped_ids:
    print("Unmapped IDs:", sorted(dismissed_ids - mapped_ids))

WICKET WITH MISSING DISMISSED PLAYER ID


,127899
match_id,40709
innings,2
overid,19
over_ball_num,5
innings_ball_num,117
batting_team_id,2
bowling_team_id,3509
striker_id,3330
striker_name,Hardik Pandya
non_striker_id,7897



DISMISSAL TYPE ID 10 RECORDS


' match_id  innings  overid  over_ball_num  striker_id         striker_name  non_striker_id non_striker_name  dismissed_player_id  dismissal_type_id        dismissal_type\n    31279        2      16              5         152      Ravindra Jadeja            6059  Ruturaj Gaikwad              152.000             10.000 Obstracting the field\n    37097        2       3              3       19309         Lahiru Udara            2001   Yashodha Lanka            2,001.000             10.000 Obstracting the field\n    50359        1       5              7       26906 Angkrish Raghuvanshi            5939    Cameron Green           26,906.000             10.000 Obstructing the field'


DISMISSAL TYPE BY INNINGS


innings,1,2,3,4,All
dismissal_type,,,,,
Bowled,883,889,3,0,1775
Caught,3922,3627,7,1,7557
Hit Wicket,9,8,0,0,17
LBW,351,358,0,0,709
NO_WICKET,100097,89996,13,12,190118
Obstracting the field,0,2,0,0,2
Obstructing the field,1,0,0,0,1
Retire Out,2,4,0,0,6
RunOut,360,238,2,0,600



DISMISSED PLAYER MAPPING COVERAGE
Unique dismissed player IDs: 814
IDs present in mapping: 814
IDs missing from mapping: 0


### Delivery player and team participation

In [30]:
# Build playing XI lookup sets
xi_all = set(
    zip(
        playing_eleven["match_id"],
        playing_eleven["team_id"],
        playing_eleven["player_id"]
    )
)

xi_played = set(
    zip(
        playing_eleven.loc[
            playing_eleven["is_played"] == True, "match_id"
        ],
        playing_eleven.loc[
            playing_eleven["is_played"] == True, "team_id"
        ],
        playing_eleven.loc[
            playing_eleven["is_played"] == True, "player_id"
        ]
    )
)

# Validate delivery participants
role_checks = []

for role, player_col, team_col in [
    ("striker", "striker_id", "batting_team_id"),
    ("non_striker", "non_striker_id", "batting_team_id"),
    ("bowler", "bowler_id", "bowling_team_id")
]:

    temp = ball_by_ball[
        ["match_id", team_col, player_col]
    ].drop_duplicates().copy()

    temp["role"] = role

    temp["in_playing_eleven"] = temp.apply(
        lambda r: (
            False if pd.isna(r[player_col])
            else (
                r["match_id"],
                r[team_col],
                r[player_col]
            ) in xi_all
        ),
        axis=1
    )

    temp["marked_as_played"] = temp.apply(
        lambda r: (
            False if pd.isna(r[player_col])
            else (
                r["match_id"],
                r[team_col],
                r[player_col]
            ) in xi_played
        ),
        axis=1
    )

    role_checks.append(temp)

role_validation = pd.concat(
    role_checks,
    ignore_index=True
)

# Summary
print("PLAYER ROLE VALIDATION SUMMARY")

display(
    role_validation.groupby("role").agg(
        unique_match_team_player_combinations=(
            "in_playing_eleven", "size"
        ),
        missing_from_playing_eleven=(
            "in_playing_eleven",
            lambda x: (~x).sum()
        ),
        not_marked_as_played=(
            "marked_as_played",
            lambda x: (~x).sum()
        )
    )
)

# Exceptions
print("\nPLAYER COMBINATIONS MISSING FROM PLAYING XI")

display(
    role_validation[
        ~role_validation["in_playing_eleven"]
    ].to_string(index=False)
)

print("\nPLAYER COMBINATIONS IN XI BUT NOT MARKED AS PLAYED")

display(
    role_validation[
        role_validation["in_playing_eleven"] &
        ~role_validation["marked_as_played"]
    ].to_string(index=False)
)

PLAYER ROLE VALIDATION SUMMARY


,unique_match_team_player_combinations,missing_from_playing_eleven,not_marked_as_played
role,,,
bowler,10094,3,3
non_striker,12596,5,5
striker,13729,6,6



PLAYER COMBINATIONS MISSING FROM PLAYING XI


' match_id  batting_team_id  striker_id        role  in_playing_eleven  marked_as_played  non_striker_id  bowling_team_id  bowler_id\n    37087          272.000  36,153.000     striker              False             False             NaN              NaN        NaN\n    37087          272.000  39,444.000     striker              False             False             NaN              NaN        NaN\n    37092          272.000  36,153.000     striker              False             False             NaN              NaN        NaN\n    37092          272.000  39,444.000     striker              False             False             NaN              NaN        NaN\n    37095          272.000  36,153.000     striker              False             False             NaN              NaN        NaN\n    37095          272.000  39,444.000     striker              False             False             NaN              NaN        NaN\n    37087          272.000         NaN non_striker              Fals


PLAYER COMBINATIONS IN XI BUT NOT MARKED AS PLAYED


'Empty DataFrame\nColumns: [match_id, batting_team_id, striker_id, role, in_playing_eleven, marked_as_played, non_striker_id, bowling_team_id, bowler_id]\nIndex: []'

#### INVESTIGATE PLAYER-XI EXCEPTIONS

In [31]:
# Build a clean exception table from delivery data
exception_rows = []

for role, player_col, name_col, team_col in [
    ("striker", "striker_id", "striker_name", "batting_team_id"),
    ("non_striker", "non_striker_id", "non_striker_name", "batting_team_id"),
    ("bowler", "bowler_id", "bowler_name", "bowling_team_id")
]:
    temp = ball_by_ball[
        ["match_id", team_col, player_col, name_col]
    ].drop_duplicates().copy()

    temp = temp.rename(columns={
        team_col: "team_id",
        player_col: "player_id",
        name_col: "player_name"
    })

    temp["role"] = role

    temp["in_playing_eleven"] = temp.apply(
        lambda r: (
            False if pd.isna(r["player_id"])
            else (
                r["match_id"], r["team_id"], r["player_id"]
            ) in xi_all
        ),
        axis=1
    )

    exception_rows.append(
        temp[~temp["in_playing_eleven"]]
    )

exception_details = pd.concat(
    exception_rows,
    ignore_index=True
)

# Add competition and match context
exception_details = exception_details.merge(
    matches[
        ["match_id", "comp_id", "match_date",
         "home_team", "away_team"]
    ],
    on="match_id",
    how="left"
)

print("ALL DELIVERY PARTICIPATION EXCEPTIONS")
display(
    exception_details.sort_values(
        ["match_id", "role", "player_id"],
        na_position="last"
    ).to_string(index=False)
)

# Check the XI records for every affected match and team
print("\nPLAYING XI RECORDS FOR AFFECTED MATCHES")

affected_keys = exception_details[
    ["match_id", "team_id"]
].drop_duplicates()

for _, key in affected_keys.iterrows():
    print(
        f"\nMatch {int(key['match_id'])}, "
        f"Team {int(key['team_id'])}"
    )

    display(
        playing_eleven[
            (playing_eleven["match_id"] == key["match_id"]) &
            (playing_eleven["team_id"] == key["team_id"])
        ].merge(
            competition_team_player_mapping[
                ["comp_id", "team_id", "player_id", "player_name"]
            ].drop_duplicates(),
            on=["team_id", "player_id"],
            how="left"
        ).to_string(index=False)
    )

# Check mapping records for actual exception player IDs
print("\nCOMPETITION MAPPING RECORDS FOR EXCEPTION PLAYERS")

exception_ids = (
    exception_details["player_id"]
    .dropna()
    .astype(int)
    .unique()
)

display(
    competition_team_player_mapping[
        competition_team_player_mapping["player_id"].isin(exception_ids)
    ].sort_values(
        ["player_id", "comp_id"]
    ).to_string(index=False)
)

ALL DELIVERY PARTICIPATION EXCEPTIONS


' match_id  team_id  player_id  player_name        role  in_playing_eleven  comp_id match_date            home_team            away_team\n    37087      272      39444 Basil Hameed      bowler              False     2670 2024-10-19 United Arab Emirates                 Oman\n    37087      272      36153  Syed Haider non_striker              False     2670 2024-10-19 United Arab Emirates                 Oman\n    37087      272      39444 Basil Hameed non_striker              False     2670 2024-10-19 United Arab Emirates                 Oman\n    37087      272      36153  Syed Haider     striker              False     2670 2024-10-19 United Arab Emirates                 Oman\n    37087      272      39444 Basil Hameed     striker              False     2670 2024-10-19 United Arab Emirates                 Oman\n    37092      272      39444 Basil Hameed      bowler              False     2670 2024-10-21              India A United Arab Emirates\n    37092      272      39444 Basil Hame


PLAYING XI RECORDS FOR AFFECTED MATCHES

Match 37087, Team 272


' match_id  team_id  player_id  batting_order  is_played is_captain is_wk  is_impact  is_impact_off  comp_id         player_name\n    37087      272      19754            NaN      False        NaN   NaN          0              0     1551           Raja Khan\n    37087      272      19754            NaN      False        NaN   NaN          0              0     1601           Raja Khan\n    37087      272      19754            NaN      False        NaN   NaN          0              0     2670           Raja Khan\n    37087      272      19754            NaN      False        NaN   NaN          0              0     2975           Raja Khan\n    37087      272       8217            NaN      False        NaN   NaN          0              0     1997         Ansh Tandon\n    37087      272       8217            NaN      False        NaN   NaN          0              0     2670         Ansh Tandon\n    37087      272      27466            NaN      False        NaN   NaN          0             


Match 37092, Team 272


' match_id  team_id  player_id  batting_order  is_played is_captain is_wk  is_impact  is_impact_off  comp_id         player_name\n    37092      272       8217            NaN      False        NaN   NaN          0              0     1997         Ansh Tandon\n    37092      272       8217            NaN      False        NaN   NaN          0              0     2670         Ansh Tandon\n    37092      272      19754            NaN      False        NaN   NaN          0              0     1551           Raja Khan\n    37092      272      19754            NaN      False        NaN   NaN          0              0     1601           Raja Khan\n    37092      272      19754            NaN      False        NaN   NaN          0              0     2670           Raja Khan\n    37092      272      19754            NaN      False        NaN   NaN          0              0     2975           Raja Khan\n    37092      272      27466            NaN      False        NaN   NaN          0             


Match 37095, Team 272


' match_id  team_id  player_id  batting_order  is_played is_captain is_wk  is_impact  is_impact_off  comp_id         player_name\n    37095      272      27465            NaN      False        NaN   NaN          0              0     2670     Nilansh Keswani\n    37095      272      27465            NaN      False        NaN   NaN          0              0     3473     Nilansh Keswani\n    37095      272      42548            NaN      False        NaN   NaN          0              0     1997 Muhammad Jawadullah\n    37095      272      42548            NaN      False        NaN   NaN          0              0     2670 Muhammad Jawadullah\n    37095      272      42548            NaN      False        NaN   NaN          0              0     2975 Muhammad Jawadullah\n    37095      272      42548            NaN      False        NaN   NaN          0              0     3133 Muhammad Jawadullah\n    37095      272      42548            NaN      False        NaN   NaN          0             


COMPETITION MAPPING RECORDS FOR EXCEPTION PLAYERS


' comp_id                             comp_name  team_id            team_name  player_id  player_name\n    2670 Mens T20 Emerging Teams Asia Cup 2024      272 United Arab Emirates      36153  Syed Haider\n    2670 Mens T20 Emerging Teams Asia Cup 2024      272 United Arab Emirates      39444 Basil Hameed'

#### UAE PLAYER IDENTITY CONFLICT

In [32]:
target_ids = [36153, 39444, 8638]

# A. Delivery participation by player and match
print("A. DELIVERY PARTICIPATION")

for pid in target_ids:
    print(f"\nPlayer ID: {pid}")

    player_deliveries = ball_by_ball[
        (ball_by_ball["striker_id"] == pid) |
        (ball_by_ball["non_striker_id"] == pid) |
        (ball_by_ball["bowler_id"] == pid)
    ]

    display(
        player_deliveries.groupby(
            ["match_id", "striker_name", "bowler_name"]
        ).agg(
            delivery_rows=("match_id", "size"),
            striker_records=("striker_id", lambda x: (x == pid).sum()),
            non_striker_records=(
                "non_striker_id", lambda x: (x == pid).sum()
            ),
            bowler_records=("bowler_id", lambda x: (x == pid).sum())
        ).reset_index().to_string(index=False)
    )


# B. Exact playing XI records for affected matches
print("\nB. PLAYING XI RECORDS")

affected_matches = sorted(
    exception_details["match_id"].dropna().unique()
)

display(
    playing_eleven[
        playing_eleven["match_id"].isin(affected_matches) &
        playing_eleven["team_id"].eq(272)
    ].sort_values(["match_id", "player_id"]).to_string(index=False)
)


# C. Mapping records for the three target IDs
print("\nC. COMPETITION MAPPING")

display(
    competition_team_player_mapping[
        competition_team_player_mapping["player_id"].isin(target_ids)
    ].sort_values(["player_id", "comp_id"]).to_string(index=False)
)


# D. Check whether IDs 36153 and 39444 ever coexist in any XI
print("\nD. COEXISTENCE IN PLAYING XI")

xi_target = playing_eleven[
    playing_eleven["player_id"].isin(target_ids)
]

display(
    xi_target.sort_values(["match_id", "player_id"]).to_string(index=False)
)

A. DELIVERY PARTICIPATION

Player ID: 36153


' match_id     striker_name       bowler_name  delivery_rows  striker_records  non_striker_records  bowler_records\n    37087     Basil Hameed      Aamir Kaleem              9                0                    9               0\n    37087     Basil Hameed       Mehran Khan              1                0                    1               0\n    37087     Basil Hameed    Sufyan Mehmood              6                0                    6               0\n    37087  Nilansh Keswani       Mehran Khan              9                0                    9               0\n    37087  Nilansh Keswani   Mohammad Nadeem              3                0                    3               0\n    37087  Nilansh Keswani Samay Shrivastava              5                0                    5               0\n    37087   Sanchit Sharma       Mehran Khan              1                0                    1               0\n    37087      Syed Haider      Aamir Kaleem              8                8   


Player ID: 39444


' match_id    striker_name          bowler_name  delivery_rows  striker_records  non_striker_records  bowler_records\n    37087    Aamir Kaleem         Basil Hameed              6                0                    0               6\n    37087    Basil Hameed         Aamir Kaleem              9                9                    0               0\n    37087    Basil Hameed          Mehran Khan              1                1                    0               0\n    37087    Basil Hameed       Sufyan Mehmood              6                6                    0               0\n    37087  Jatinder Singh         Basil Hameed              6                0                    0               6\n    37087  Karan Sonavale         Basil Hameed              3                0                    0               3\n    37087     Syed Haider         Aamir Kaleem              8                0                    8               0\n    37087     Syed Haider          Mehran Khan              1  


Player ID: 8638


' match_id            striker_name         bowler_name  delivery_rows  striker_records  non_striker_records  bowler_records\n    16863            Adnan Idrees        Basil Hameed              1                0                    0               1\n    16863            Basil Hameed     Mohamed Shafeeq              5                5                    0               0\n    16863            Basil Hameed         Sayed Monib              2                2                    0               0\n    16863             Bilal Tahir        Basil Hameed              2                0                    0               2\n    16863             Edson Silva        Basil Hameed              1                0                    0               1\n    16863            Meet Bhavsar        Basil Hameed              6                0                    0               6\n    16863          Mohammed Aslam        Basil Hameed              3                0                    0               3\n    168

... (2 outputs left out)


C. COMPETITION MAPPING


' comp_id                                            comp_name  team_id            team_name  player_id  player_name\n    1551                         Asia Cup T20 Qualifiers 2022      272 United Arab Emirates       8638 Basil Hameed\n    1601  Bangladesh in United Arab Emirates T20I Series 2022      272 United Arab Emirates       8638 Basil Hameed\n    1997 New Zealand in United Arab Emirates T20I Series 2023      272 United Arab Emirates       8638 Basil Hameed\n    2670                Mens T20 Emerging Teams Asia Cup 2024      272 United Arab Emirates       8638 Basil Hameed\n    3313     Ireland in United Arab Emirates T20I Series 2026      272 United Arab Emirates       8638 Basil Hameed\n    2670                Mens T20 Emerging Teams Asia Cup 2024      272 United Arab Emirates      36153  Syed Haider\n    2670                Mens T20 Emerging Teams Asia Cup 2024      272 United Arab Emirates      39444 Basil Hameed'


D. COEXISTENCE IN PLAYING XI


' match_id  team_id  player_id  batting_order  is_played is_captain is_wk  is_impact  is_impact_off\n    16863      272       8638          5.000       True      False False          0              0\n    16864      272       8638          6.000       True      False False          0              0\n    16866      272       8638          6.000       True      False False          0              0\n    17631      272       8638          6.000       True      False False          0              0\n    17632      272       8638          6.000       True      False False          0              0\n    24792      272       8638          6.000       True      False False          0              0\n    24793      272       8638          5.000       True      False False          0              0\n    24794      272       8638          6.000       True      False False          0              0\n    37087      272       8638          6.000       True       True False          0              0\

#### PLAYER IDENTITY CROSS-CHECK

In [33]:
target_ids = [36153, 39444, 8638]

# 1. Build delivery participation summary
participation = []

for pid in target_ids:
    rows = ball_by_ball[
        (ball_by_ball["striker_id"] == pid) |
        (ball_by_ball["non_striker_id"] == pid) |
        (ball_by_ball["bowler_id"] == pid)
    ]

    for match_id, group in rows.groupby("match_id"):
        participation.append({
            "player_id": pid,
            "player_name": (
                group["striker_name"].eq(
                    group["striker_name"].iloc[0]
                ).any()
            ),
            "match_id": match_id,
            "delivery_rows": len(group),
            "striker_records": (group["striker_id"] == pid).sum(),
            "non_striker_records": (
                group["non_striker_id"] == pid
            ).sum(),
            "bowler_records": (group["bowler_id"] == pid).sum()
        })

participation_df = pd.DataFrame(participation)

# 2. Add match and competition context
participation_df = participation_df.merge(
    matches[
        ["match_id", "comp_id", "match_date",
         "home_team", "away_team"]
    ],
    on="match_id",
    how="left"
)

print("A. DELIVERY MATCH SUMMARY")

display(
    participation_df.sort_values(
        ["player_id", "match_id"]
    ).to_string(index=False)
)


# 3. Compare Basil Hameed IDs in the same matches
print("\nB. BASIL HAMEED ID COMPARISON")

basil_matches = sorted(
    participation_df.loc[
        participation_df["player_id"] == 39444,
        "match_id"
    ].unique()
)

basil_comparison = []

for mid in basil_matches:
    xi_match = playing_eleven[
        (playing_eleven["match_id"] == mid) &
        (playing_eleven["team_id"] == 272)
    ]

    basil_comparison.append({
        "match_id": mid,
        "id_39444_in_delivery": True,
        "id_39444_in_XI": (
            (xi_match["player_id"] == 39444).any()
        ),
        "id_8638_in_XI": (
            (xi_match["player_id"] == 8638).any()
        ),
        "id_8638_marked_played": (
            xi_match.loc[
                xi_match["player_id"] == 8638,
                "is_played"
            ].eq(True).any()
        )
    })

display(pd.DataFrame(basil_comparison).to_string(index=False))


# 4. Check Syed Haider XI presence in all available records
print("\nC. SYED HAIDER ID 36153")

syed_xi = playing_eleven[
    playing_eleven["player_id"] == 36153
]

print("Playing XI records:", len(syed_xi))
print("Matches in delivery data:",
      participation_df.loc[
          participation_df["player_id"] == 36153,
          "match_id"
      ].nunique())

display(syed_xi.to_string(index=False))


# 5. Compact totals
print("\nD. OVERALL TOTALS")

display(
    participation_df.groupby("player_id").agg(
        matches=("match_id", "nunique"),
        delivery_rows=("delivery_rows", "sum"),
        striker_records=("striker_records", "sum"),
        non_striker_records=("non_striker_records", "sum"),
        bowler_records=("bowler_records", "sum")
    )
)

A. DELIVERY MATCH SUMMARY


' player_id  player_name  match_id  delivery_rows  striker_records  non_striker_records  bowler_records  comp_id match_date            home_team            away_team\n      8638         True     16863             33                7                    2              24     1551 2022-08-21 United Arab Emirates               Kuwait\n      8638         True     16864             70               28                   24              18     1551 2022-08-22 United Arab Emirates            Singapore\n      8638         True     16866             55               12                   19              24     1551 2022-08-24            Hong Kong United Arab Emirates\n      8638         True     17631             23                4                    6              13     1601 2022-09-25 United Arab Emirates           Bangladesh\n      8638         True     17632             75               42                   33               0     1601 2022-09-27 United Arab Emirates           Bangladesh\n   


B. BASIL HAMEED ID COMPARISON


' match_id  id_39444_in_delivery  id_39444_in_XI  id_8638_in_XI  id_8638_marked_played\n    37087                  True           False           True                   True\n    37092                  True           False           True                   True\n    37095                  True           False           True                   True'


C. SYED HAIDER ID 36153
Playing XI records: 0
Matches in delivery data: 3


'Empty DataFrame\nColumns: [match_id, team_id, player_id, batting_order, is_played, is_captain, is_wk, is_impact, is_impact_off]\nIndex: []'


D. OVERALL TOTALS


,matches,delivery_rows,striker_records,non_striker_records,bowler_records
player_id,,,,,
8638,9,444,146,158,140
36153,3,105,49,56,0
39444,3,106,34,42,30


### Delivery timestamps and bowling-speed audit

In [35]:
# 1. MATCH DATE ALIGNMENT
print("1. MATCH DATE ALIGNMENT")

date_check = ball_by_ball[
    ["match_id", "innings", "innings_ball_num",
     "delivery_ist_time"]
].copy()

date_check["delivery_time"] = pd.to_datetime(
    date_check["delivery_ist_time"],
    errors="coerce",
    utc=True
)

match_dates = matches[["match_id", "match_date"]].copy()

match_dates["parsed_match_date"] = pd.to_datetime(
    match_dates["match_date"],
    errors="coerce"
)

date_check = date_check.merge(
    match_dates[["match_id", "parsed_match_date"]],
    on="match_id",
    how="left"
)

# Compare calendar dates only; do not assume timezone conversion
date_check["delivery_date_utc"] = (
    date_check["delivery_time"].dt.date
)

date_check["match_date_only"] = (
    date_check["parsed_match_date"].dt.date
)

date_check["date_difference_days"] = date_check.apply(
    lambda r: (
        (r["delivery_date_utc"] - r["match_date_only"]).days
        if pd.notna(r["delivery_date_utc"]) and
           pd.notna(r["match_date_only"])
        else np.nan
    ),
    axis=1
)

print("Unparseable delivery timestamps:",
      date_check["delivery_time"].isna().sum())

print("Unparseable match dates:",
      date_check["parsed_match_date"].isna().sum())

print("\nDate difference distribution:")
display(
    date_check["date_difference_days"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\nExamples with date difference beyond one day:")
display(
    date_check[
        date_check["date_difference_days"].abs() > 1
    ].head(30).to_string(index=False)
)


# 2. BOWLING SPEED QUALITY
print("\n2. BOWLING SPEED DISTRIBUTION")

print(ball_by_ball["bowl_speed"].describe())

print("\nMissing speeds:",
      ball_by_ball["bowl_speed"].isna().sum())

print("Zero speeds:",
      (ball_by_ball["bowl_speed"] == 0).sum())

print("Negative speeds:",
      (ball_by_ball["bowl_speed"] < 0).sum())

print("\nLowest 20 distinct speeds:")
display(
    ball_by_ball["bowl_speed"]
    .dropna()
    .sort_values()
    .drop_duplicates()
    .head(20)
)

print("\nHighest 20 distinct speeds:")
display(
    ball_by_ball["bowl_speed"]
    .dropna()
    .sort_values(ascending=False)
    .drop_duplicates()
    .head(20)
)

print("\nSpeed distribution by bowling type:")
display(
    ball_by_ball.groupby("bowling_type")["bowl_speed"]
    .agg(["count", "min", "median", "mean", "max"])
    .sort_values("count", ascending=False)
)


# 3. DUPLICATE TIMESTAMP INVESTIGATION
print("\n3. DUPLICATE TIMESTAMP INVESTIGATION")

duplicate_times = ball_by_ball[
    ball_by_ball["delivery_ist_time"].duplicated(keep=False)
].copy()

print("Rows involved in repeated timestamp values:",
      len(duplicate_times))

print("Distinct repeated timestamp values:",
      duplicate_times["delivery_ist_time"].nunique())

print("\nRepeated timestamps within the same match:")
same_match_duplicates = (
    duplicate_times.groupby(
        ["match_id", "delivery_ist_time"]
    ).size().reset_index(name="records")
)

display(
    same_match_duplicates[
        same_match_duplicates["records"] > 1
    ].head(30).to_string(index=False)
)

1. MATCH DATE ALIGNMENT
Unparseable delivery timestamps: 0
Unparseable match dates: 0

Date difference distribution:


,count
date_difference_days,
-1,1681
0,190569
1,8009
2,740



Examples with date difference beyond one day:


' match_id  innings  innings_ball_num       delivery_ist_time             delivery_time parsed_match_date delivery_date_utc match_date_only  date_difference_days\n    38626        1                 1 2024-12-11 11:35:44 UTC 2024-12-11 11:35:44+00:00        2024-12-09        2024-12-11      2024-12-09                     2\n    38626        1                 2 2024-12-11 11:36:11 UTC 2024-12-11 11:36:11+00:00        2024-12-09        2024-12-11      2024-12-09                     2\n    38626        1                 3 2024-12-11 11:36:39 UTC 2024-12-11 11:36:39+00:00        2024-12-09        2024-12-11      2024-12-09                     2\n    38626        1                 4 2024-12-11 11:38:18 UTC 2024-12-11 11:38:18+00:00        2024-12-09        2024-12-11      2024-12-09                     2\n    38626        1                 5 2024-12-11 11:38:43 UTC 2024-12-11 11:38:43+00:00        2024-12-09        2024-12-11      2024-12-09                     2\n    38626        1         


2. BOWLING SPEED DISTRIBUTION
count   200,999.000
mean         71.414
std          60.012
min          -1.000
25%           0.000
50%          91.700
75%         130.400
max       1,456.100
Name: bowl_speed, dtype: float64

Missing speeds: 0
Zero speeds: 77861
Negative speeds: 1

Lowest 20 distinct speeds:


,bowl_speed
6756,-1.000
111724,0.000
27721,1.320
2507,6.310
54100,6.500
40514,8.500
100169,9.360
50906,9.900
684,10.000
7483,11.700


... (2 outputs left out)


Speed distribution by bowling type:


,count,min,median,mean,max
bowling_type,,,,,
Right-arm Leg-Break,28027,0.000,85.200,57.642,931.000
Right-arm medium-fast,27014,0.000,122.600,81.711,"1,305.000"
Right-arm Fast,26637,0.000,127.100,84.056,"1,339.600"
Right-arm medium,23919,0.000,118.100,74.744,"1,229.700"
Slow left-arm orthodox,23582,-1.000,86.950,55.588,967.000
Right-arm offbreak,20948,0.000,86.900,54.632,999.000
Right-arm fast-medium,15380,0.000,122.400,79.347,"1,410.000"
Left-arm medium-fast,12492,0.000,127.000,86.692,153.000
Left-arm fast-medium,7780,0.000,122.300,77.439,151.800



3. DUPLICATE TIMESTAMP INVESTIGATION
Rows involved in repeated timestamp values: 311
Distinct repeated timestamp values: 138

Repeated timestamps within the same match:


' match_id       delivery_ist_time  records\n    13511 2022-01-23 00:00:00 UTC       34\n    16063 2022-08-02 01:51:11 UTC        2\n    17737 2022-10-07 00:00:00 UTC        2\n    24354 2023-08-13 00:00:00 UTC        5\n    45489 2026-09-15 18:57:34 UTC        2'

#### Targeted anomaly investigation

In [36]:
# 1. DATE MISMATCHES BY MATCH
print("1. MATCHES WITH DATE DIFFERENCES")

date_mismatch_summary = (
    date_check[
        date_check["date_difference_days"] != 0
    ]
    .groupby(
        ["match_id", "match_date_only", "date_difference_days"]
    )
    .agg(
        delivery_rows=("innings_ball_num", "size"),
        first_delivery=("delivery_time", "min"),
        last_delivery=("delivery_time", "max")
    )
    .reset_index()
    .sort_values(
        ["date_difference_days", "delivery_rows"],
        ascending=[False, False]
    )
)

display(date_mismatch_summary.to_string(index=False))


# 2. MATCH 38626 DETAILS
print("\n2. MATCH 38626 INVESTIGATION")

display(
    matches[matches["match_id"] == 38626].T
)

display(
    ball_by_ball[
        ball_by_ball["match_id"] == 38626
    ][[
        "innings", "innings_ball_num",
        "overid", "over_ball_num",
        "delivery_ist_time",
        "striker_name", "bowler_name",
        "runs", "extras_type"
    ]].to_string(index=False)
)


# 3. FOUR CHRONOLOGICAL REVERSALS WITH NEIGHBOURS
print("\n3. CHRONOLOGICAL REVERSAL CONTEXT")

reversal_keys = (
    time_audit.loc[
        time_audit["time_reversal"],
        ["match_id", "innings", "innings_ball_num"]
    ]
    .to_records(index=False)
)

for match_id, innings, sequence in reversal_keys:
    print(
        f"\nMatch {match_id}, innings {innings}, "
        f"sequence {sequence}"
    )

    context = ball_by_ball[
        (ball_by_ball["match_id"] == match_id) &
        (ball_by_ball["innings"] == innings) &
        (ball_by_ball["innings_ball_num"].between(
            sequence - 2, sequence + 2
        ))
    ][[
        "innings_ball_num", "overid", "over_ball_num",
        "delivery_ist_time",
        "striker_name", "bowler_name", "runs"
    ]].sort_values("innings_ball_num")

    display(context.to_string(index=False))


# 4. BOWLING SPEED OUTLIERS WITH DELIVERY CONTEXT
print("\n4. EXTREME BOWLING SPEED RECORDS")

speed_context = ball_by_ball[
    ["match_id", "innings", "overid", "over_ball_num",
     "delivery_ist_time", "bowler_id", "bowler_name",
     "bowling_type", "bowl_speed", "extras_type",
     "delivery_style"]
].copy()

print("Negative speed records:")
display(
    speed_context[
        speed_context["bowl_speed"] < 0
    ].to_string(index=False)
)

print("\nSpeeds above 200:")
display(
    speed_context[
        speed_context["bowl_speed"] > 200
    ].sort_values("bowl_speed", ascending=False)
    .to_string(index=False)
)

print("\nSpeed frequency above 150:")
display(
    speed_context[
        speed_context["bowl_speed"] > 150
    ]["bowl_speed"]
    .value_counts()
    .head(30)
    .to_string()
)


# 5. ZERO SPEED BY BOWLING TYPE
print("\n5. ZERO SPEED DISTRIBUTION")

zero_speed_summary = (
    ball_by_ball.assign(
        is_zero_speed=ball_by_ball["bowl_speed"] == 0
    )
    .groupby("bowling_type")
    .agg(
        total_records=("bowl_speed", "size"),
        zero_speed_records=("is_zero_speed", "sum")
    )
)

zero_speed_summary["zero_speed_percentage"] = (
    zero_speed_summary["zero_speed_records"] /
    zero_speed_summary["total_records"] * 100
)

display(
    zero_speed_summary.sort_values(
        "zero_speed_percentage",
        ascending=False
    )
)

1. MATCHES WITH DATE DIFFERENCES


' match_id match_date_only  date_difference_days  delivery_rows            first_delivery             last_delivery\n    38626      2024-12-09                     2            254 2024-12-11 11:35:44+00:00 2024-12-11 15:03:13+00:00\n    38628      2024-12-12                     2            253 2024-12-14 11:33:14+00:00 2024-12-14 15:27:04+00:00\n    38627      2024-12-11                     2            233 2024-12-13 11:34:14+00:00 2024-12-13 14:50:36+00:00\n    41510      2025-07-31                     1            256 2025-08-01 00:02:21+00:00 2025-08-01 03:45:38+00:00\n    41512      2025-08-03                     1            255 2025-08-04 00:01:05+00:00 2025-08-04 03:37:14+00:00\n    13515      2022-01-30                     1            254 2022-01-31 01:30:27+00:00 2022-01-31 05:01:58+00:00\n    13514      2022-01-29                     1            253 2022-01-30 01:30:15+00:00 2022-01-30 04:51:20+00:00\n    13512      2022-01-23                     1            252 2022-01-


2. MATCH 38626 INVESTIGATION


,601
match_id,38626
comp_id,2753
match_number,1
match_date,2024-12-09
stage,League
venue_id,59
venue_country,Zimbabwe
home_team_id,72
home_team,Zimbabwe
away_team_id,78


' innings  innings_ball_num  overid  over_ball_num       delivery_ist_time         striker_name          bowler_name  runs extras_type\n       1                 1       1              1 2024-12-11 11:35:44 UTC   Rahmanullah Gurbaz      Richard Ngarava     0         NaN\n       1                 2       1              2 2024-12-11 11:36:11 UTC   Rahmanullah Gurbaz      Richard Ngarava     0         NaN\n       1                 3       1              3 2024-12-11 11:36:39 UTC   Rahmanullah Gurbaz      Richard Ngarava     0         NaN\n       1                 4       1              4 2024-12-11 11:38:18 UTC    Hazratullah Zazai      Richard Ngarava     0         NaN\n       1                 5       1              5 2024-12-11 11:38:43 UTC    Hazratullah Zazai      Richard Ngarava     4         NaN\n       1                 6       1              6 2024-12-11 11:39:11 UTC    Hazratullah Zazai      Richard Ngarava     1         NaN\n       1                 7       2              1 2024


3. CHRONOLOGICAL REVERSAL CONTEXT

Match 14104, innings 1, sequence 8


... (11 outputs left out)


Speed frequency above 150:


'bowl_speed\n151.000    29\n150.100    17\n150.400    16\n150.300    13\n150.500    11\n150.900    10\n153.000    10\n152.000     9\n151.200     9\n154.000     8\n150.600     8\n152.800     7\n151.800     7\n150.200     7\n152.400     6\n150.700     6\n151.100     6\n150.800     6\n152.100     6\n152.600     6\n151.400     6\n152.300     5\n151.300     5\n153.600     5\n152.200     5\n153.200     4\n153.900     4\n151.700     4\n153.100     3\n153.300     3'


5. ZERO SPEED DISTRIBUTION


,total_records,zero_speed_records,zero_speed_percentage
bowling_type,,,
Left-arm Fast,4453,1924,43.207
Right-arm medium,23919,10231,42.774
Right-arm offbreak,20948,8751,41.775
Left-arm fast-medium,7780,3191,41.015
Slow left-arm orthodox,23582,9456,40.098
Right-arm fast-medium,15380,6140,39.922
Right-arm Fast,26637,10013,37.591
Right-arm medium-fast,27014,10107,37.414
Right-arm Leg-Break,28027,10169,36.283


#### Final investigation before classifying the fields

In [37]:
# 1. COMPLETE MATCH DATE OFFSET SUMMARY
print("1. ALL MATCHES WITH DATE OFFSETS")

offset_summary = (
    date_check[date_check["date_difference_days"] != 0]
    .groupby(
        ["match_id", "match_date_only", "date_difference_days"]
    )
    .agg(
        delivery_rows=("innings_ball_num", "size"),
        first_delivery=("delivery_time", "min"),
        last_delivery=("delivery_time", "max")
    )
    .reset_index()
    .sort_values(["date_difference_days", "match_id"])
)

display(offset_summary.to_string(index=False))

print("\nOffset counts by match:")
display(
    offset_summary.groupby("date_difference_days")
    .agg(
        affected_matches=("match_id", "nunique"),
        affected_delivery_rows=("delivery_rows", "sum")
    )
)


# 2. INSPECT REPEATED MIDNIGHT TIMESTAMPS
print("\n2. MIDNIGHT TIMESTAMP RECORDS")

midnight_records = ball_by_ball[
    ball_by_ball["delivery_ist_time"].str.contains(
        r"00:00:00 UTC$", na=False
    )
].copy()

print("Total midnight timestamp records:", len(midnight_records))

display(
    midnight_records.groupby("match_id")
    .agg(
        records=("innings_ball_num", "size"),
        innings=("innings", "nunique"),
        first_sequence=("innings_ball_num", "min"),
        last_sequence=("innings_ball_num", "max")
    )
    .sort_values("records", ascending=False)
    .to_string()
)


# 3. BOWLING SPEED COUNTS BY THRESHOLD
print("\n3. BOWLING SPEED THRESHOLD COUNTS")

speed_thresholds = pd.DataFrame({
    "condition": [
        "Negative",
        "Exactly zero",
        "Between 0 and 50",
        "Between 50 and 100",
        "Between 100 and 150",
        "Between 150 and 160",
        "Between 160 and 200",
        "Above 200",
        "Above 500",
        "Above 1000"
    ],
    "records": [
        (ball_by_ball["bowl_speed"] < 0).sum(),
        (ball_by_ball["bowl_speed"] == 0).sum(),
        ((ball_by_ball["bowl_speed"] > 0) &
         (ball_by_ball["bowl_speed"] < 50)).sum(),
        ((ball_by_ball["bowl_speed"] >= 50) &
         (ball_by_ball["bowl_speed"] < 100)).sum(),
        ((ball_by_ball["bowl_speed"] >= 100) &
         (ball_by_ball["bowl_speed"] <= 150)).sum(),
        ((ball_by_ball["bowl_speed"] > 150) &
         (ball_by_ball["bowl_speed"] <= 160)).sum(),
        ((ball_by_ball["bowl_speed"] > 160) &
         (ball_by_ball["bowl_speed"] <= 200)).sum(),
        (ball_by_ball["bowl_speed"] > 200).sum(),
        (ball_by_ball["bowl_speed"] > 500).sum(),
        (ball_by_ball["bowl_speed"] > 1000).sum()
    ]
})

display(speed_thresholds)


# 4. SPEED OUTLIERS BY MATCH AND BOWLER
print("\n4. SPEED OUTLIERS ABOVE 200")

extreme_speeds = ball_by_ball[
    ball_by_ball["bowl_speed"] > 200
].copy()

display(
    extreme_speeds[[
        "match_id", "innings", "overid", "over_ball_num",
        "delivery_ist_time", "bowler_id", "bowler_name",
        "bowling_type", "bowl_speed"
    ]].sort_values(
        "bowl_speed", ascending=False
    ).to_string(index=False)
)


# 5. SPEED DISTRIBUTION WITHOUT ZERO AND NEGATIVE VALUES
print("\n5. POSITIVE SPEED DISTRIBUTION")

positive_speeds = ball_by_ball.loc[
    ball_by_ball["bowl_speed"] > 0,
    "bowl_speed"
]

print(positive_speeds.describe(
    percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
))

1. ALL MATCHES WITH DATE OFFSETS


' match_id match_date_only  date_difference_days  delivery_rows            first_delivery             last_delivery\n    37349      2024-11-10                    -1            229 2024-11-09 20:00:15+00:00 2024-11-09 23:56:50+00:00\n    37350      2024-11-11                    -1            226 2024-11-10 20:01:28+00:00 2024-11-10 23:12:03+00:00\n    37351      2024-11-15                    -1            197 2024-11-14 20:54:18+00:00 2024-11-14 23:59:52+00:00\n    37353      2024-11-18                    -1             30 2024-11-17 20:01:45+00:00 2024-11-17 20:23:07+00:00\n    41851      2026-02-01                    -1            121 2026-01-31 17:16:20+00:00 2026-01-31 20:17:54+00:00\n    46941      2025-10-30                    -1            247 2025-10-29 12:00:40+00:00 2025-10-29 15:40:02+00:00\n    46942      2025-11-01                    -1            227 2025-10-31 12:01:33+00:00 2025-10-31 15:25:13+00:00\n    48514      2026-01-12                    -1            157 2026-01-


Offset counts by match:


,affected_matches,affected_delivery_rows
date_difference_days,,
-1,9,1681
1,61,8009
2,3,740



2. MIDNIGHT TIMESTAMP RECORDS
Total midnight timestamp records: 42


'          records  innings  first_sequence  last_sequence\nmatch_id                                                 \n13511          34        1               1             34\n24354           5        1             102            106\n17737           2        1               7              8\n25158           1        1             120            120'


3. BOWLING SPEED THRESHOLD COUNTS


,condition,records
0,Negative,1
1,Exactly zero,77861
2,Between 0 and 50,22
3,Between 50 and 100,42978
4,Between 100 and 150,79838
5,Between 150 and 160,279
6,Between 160 and 200,1
7,Above 200,19
8,Above 500,18
9,Above 1000,12



4. SPEED OUTLIERS ABOVE 200


' match_id  innings  overid  over_ball_num       delivery_ist_time  bowler_id        bowler_name           bowling_type  bowl_speed\n    14144        2       4              1 2022-05-05 21:50:29 UTC       4816      Khaleel Ahmed        Left-arm medium   1,456.100\n    19960        1      20              6 2023-03-26 19:02:14 UTC       4235     Sisanda Magala  Right-arm fast-medium   1,410.000\n    16259        2       2              6 2022-07-20 22:29:17 UTC      22920        Jacob Duffy  Right-arm fast-medium   1,351.000\n    14116        2      19              4 2022-04-13 23:16:45 UTC       3628      Kagiso Rabada         Right-arm Fast   1,339.600\n    14122        2      19              6 2022-04-17 23:03:51 UTC        161       Dwayne Bravo  Right-arm medium-fast   1,305.000\n    21178        1       6              4 2023-04-08 06:58:36 UTC       6515    Benjamin Lister        Left-arm medium   1,257.000\n    17764        1      19              6 2022-10-12 15:15:26 UTC        82


5. POSITIVE SPEED DISTRIBUTION
count   123,137.000
mean        116.570
std          24.796
min           1.320
1%           79.700
5%           84.600
25%          94.300
50%         123.800
75%         135.500
95%         142.800
99%         147.200
max       1,456.100
Name: bowl_speed, dtype: float64


#### Inspect remaining suspicious records

In [38]:
# 1. POSITIVE SPEEDS BELOW 50
print("1. POSITIVE SPEEDS BELOW 50")

low_speeds = ball_by_ball[
    (ball_by_ball["bowl_speed"] > 0) &
    (ball_by_ball["bowl_speed"] < 50)
][[
    "match_id", "innings", "overid", "over_ball_num",
    "delivery_ist_time", "bowler_id", "bowler_name",
    "bowling_type", "bowl_speed", "delivery_style"
]]

display(low_speeds.to_string(index=False))


# 2. ALL SPEEDS ABOVE 200
print("\n2. ALL SPEEDS ABOVE 200")

extreme_speeds = ball_by_ball[
    ball_by_ball["bowl_speed"] > 200
][[
    "match_id", "innings", "overid", "over_ball_num",
    "delivery_ist_time", "bowler_id", "bowler_name",
    "bowling_type", "bowl_speed", "delivery_style"
]].sort_values("bowl_speed", ascending=False)

display(extreme_speeds.to_string(index=False))


# 3. THE SINGLE VALUE BETWEEN 160 AND 200
print("\n3. SPEED BETWEEN 160 AND 200")

display(
    ball_by_ball[
        (ball_by_ball["bowl_speed"] > 160) &
        (ball_by_ball["bowl_speed"] <= 200)
    ][[
        "match_id", "innings", "overid", "over_ball_num",
        "delivery_ist_time", "bowler_name",
        "bowling_type", "bowl_speed"
    ]].to_string(index=False)
)


# 4. MIDNIGHT TIMESTAMP RECORDS WITH MATCH CONTEXT
print("\n4. MIDNIGHT TIMESTAMP RECORDS")

midnight_details = ball_by_ball[
    ball_by_ball["delivery_ist_time"].str.contains(
        r"00:00:00 UTC$", na=False
    )
][[
    "match_id", "innings", "innings_ball_num",
    "overid", "over_ball_num", "delivery_ist_time",
    "striker_name", "bowler_name"
]].copy()

midnight_details = midnight_details.merge(
    matches[[
        "match_id", "match_date", "venue_country",
        "home_team", "away_team"
    ]],
    on="match_id",
    how="left"
)

display(
    midnight_details.sort_values(
        ["match_id", "innings_ball_num"]
    ).to_string(index=False)
)


# 5. MATCH 25158: INSPECT ITS TIMESTAMP CONTEXT
print("\n5. MATCH 25158")

display(
    matches[matches["match_id"] == 25158].T
)

display(
    ball_by_ball[
        ball_by_ball["match_id"] == 25158
    ][[
        "innings", "innings_ball_num",
        "overid", "over_ball_num",
        "delivery_ist_time",
        "striker_name", "bowler_name"
    ]].to_string(index=False)
)

1. POSITIVE SPEEDS BELOW 50


' match_id  innings  overid  over_ball_num       delivery_ist_time  bowler_id          bowler_name           bowling_type  bowl_speed    delivery_style\n    13513        2      13              6 2022-01-27 04:59:29 UTC        494       Kieron Pollard  Right-arm medium-fast      10.000        Off Cutter\n    13524        1      14              2 2022-02-20 19:58:58 UTC       6797       Dominic Drakes   Left-arm medium-fast      12.400          Straight\n    13614        1       5              4 2022-02-15 14:04:27 UTC        830       Josh Hazlewood  Right-arm fast-medium       6.310           Inswing\n    14111        2       9              6 2022-04-09 22:22:51 UTC       1071       Jaydev Unadkat   Left-arm medium-fast      11.700        Off Cutter\n    14114        1       3              5 2022-04-11 19:44:44 UTC        360    Bhuvneshwar Kumar       Right-arm medium      29.610          Straight\n    14160        2       5              4 2022-05-18 21:50:42 UTC       2633         Ja


2. ALL SPEEDS ABOVE 200


' match_id  innings  overid  over_ball_num       delivery_ist_time  bowler_id        bowler_name           bowling_type  bowl_speed   delivery_style\n    14144        2       4              1 2022-05-05 21:50:29 UTC       4816      Khaleel Ahmed        Left-arm medium   1,456.100        Out Swing\n    19960        1      20              6 2023-03-26 19:02:14 UTC       4235     Sisanda Magala  Right-arm fast-medium   1,410.000        Full Toss\n    16259        2       2              6 2022-07-20 22:29:17 UTC      22920        Jacob Duffy  Right-arm fast-medium   1,351.000          Inswing\n    14116        2      19              4 2022-04-13 23:16:45 UTC       3628      Kagiso Rabada         Right-arm Fast   1,339.600        Full Toss\n    14122        2      19              6 2022-04-17 23:03:51 UTC        161       Dwayne Bravo  Right-arm medium-fast   1,305.000 Reverse Swing In\n    21178        1       6              4 2023-04-08 06:58:36 UTC       6515    Benjamin Lister        Le


3. SPEED BETWEEN 160 AND 200


' match_id  innings  overid  over_ball_num       delivery_ist_time bowler_name   bowling_type  bowl_speed\n    28963        1      18              4 2024-01-12 13:25:09 UTC  Haris Rauf Right-arm Fast     177.100'


4. MIDNIGHT TIMESTAMP RECORDS


' match_id  innings  innings_ball_num  overid  over_ball_num       delivery_ist_time     striker_name      bowler_name match_date  venue_country   home_team   away_team\n    13511        1                 1       1              1 2022-01-23 00:00:00 UTC        Jason Roy Sheldon Cottrell 2022-01-22    West Indies West Indies     England\n    13511        1                 2       1              2 2022-01-23 00:00:00 UTC        Jason Roy Sheldon Cottrell 2022-01-22    West Indies West Indies     England\n    13511        1                 3       1              3 2022-01-23 00:00:00 UTC        Jason Roy Sheldon Cottrell 2022-01-22    West Indies West Indies     England\n    13511        1                 4       1              4 2022-01-23 00:00:00 UTC        Jason Roy Sheldon Cottrell 2022-01-22    West Indies West Indies     England\n    13511        1                 5       1              5 2022-01-23 00:00:00 UTC        Jason Roy Sheldon Cottrell 2022-01-22    West Indies West Indie


5. MATCH 25158


,295
match_id,25158
comp_id,1999
match_number,1
match_date,2023-08-30
stage,League
venue_id,66
venue_country,United Kingdom
home_team_id,22
home_team,England
away_team_id,17


' innings  innings_ball_num  overid  over_ball_num       delivery_ist_time     striker_name      bowler_name\n       1                 1       1              1 2023-08-30 22:33:25 UTC       Finn Allen        Luke Wood\n       1                 2       1              2 2023-08-30 22:33:58 UTC       Finn Allen        Luke Wood\n       1                 3       1              3 2023-08-30 22:34:17 UTC       Finn Allen        Luke Wood\n       1                 4       1              4 2023-08-30 22:35:28 UTC       Finn Allen        Luke Wood\n       1                 5       1              5 2023-08-30 22:36:27 UTC       Finn Allen        Luke Wood\n       1                 6       1              6 2023-08-30 22:37:46 UTC       Finn Allen        Luke Wood\n       1                 7       2              1 2023-08-30 22:39:14 UTC     Devon Conway       Sam Curran\n       1                 8       2              2 2023-08-30 22:39:43 UTC     Devon Conway       Sam Curran\n       1          

#### Inspect all speed and timestamp exceptions

In [39]:
# 1. All positive speeds below 50
print("POSITIVE SPEEDS BELOW 50")
display(
    ball_by_ball.loc[
        (ball_by_ball["bowl_speed"] > 0) &
        (ball_by_ball["bowl_speed"] < 50),
        ["match_id", "innings", "innings_ball_num", "bowler_name",
         "bowling_type", "bowl_speed", "delivery_style"]
    ].sort_values("bowl_speed").to_string(index=False)
)

# 2. All speeds above 200
print("\nALL SPEEDS ABOVE 200")
display(
    ball_by_ball.loc[
        ball_by_ball["bowl_speed"] > 200,
        ["match_id", "innings", "innings_ball_num", "bowler_name",
         "bowling_type", "bowl_speed", "delivery_style"]
    ].sort_values("bowl_speed", ascending=False).to_string(index=False)
)

# 3. The single speed between 160 and 200
print("\nSPEEDS BETWEEN 160 AND 200")
display(
    ball_by_ball.loc[
        ball_by_ball["bowl_speed"].between(160, 200),
        ["match_id", "innings", "innings_ball_num", "bowler_name",
         "bowling_type", "bowl_speed", "delivery_style"]
    ].to_string(index=False)
)

# 4. All midnight timestamp records, with match context
print("\nALL MIDNIGHT TIMESTAMP RECORDS")
midnight_mask = ball_by_ball["delivery_ist_time"].str.endswith(
    "00:00:00 UTC", na=False
)

midnight_details = ball_by_ball.loc[
    midnight_mask,
    ["match_id", "innings", "innings_ball_num", "overid",
     "over_ball_num", "delivery_ist_time", "striker_name", "bowler_name"]
].merge(
    matches[["match_id", "match_date", "venue_country", "home_team", "away_team"]],
    on="match_id",
    how="left"
)

display(midnight_details.sort_values(
    ["match_id", "innings", "innings_ball_num"]
).to_string(index=False))

# 5. Inspect the midnight record in match 25158
print("\nMATCH 25158 — MIDNIGHT RECORD")
display(
    ball_by_ball.loc[
        (ball_by_ball["match_id"] == 25158) &
        ball_by_ball["delivery_ist_time"].str.endswith("00:00:00 UTC", na=False)
    ].to_string(index=False)
)

POSITIVE SPEEDS BELOW 50


' match_id  innings  innings_ball_num          bowler_name           bowling_type  bowl_speed    delivery_style\n    16317        1                99           Sam Curran   Left-arm medium-fast       1.320 Reverse Swing Out\n    13614        1                30       Josh Hazlewood  Right-arm fast-medium       6.310           Inswing\n    20214        1                40      Mayank Markande    Right-arm Leg-Break       6.500            Googly\n    17740        1                46     Mitchell Santner Slow left-arm orthodox       8.500          Orthodox\n    32134        1                83 Wellington Masakadza Slow left-arm orthodox       9.360          Orthodox\n    20201        1                37        Harpreet Brar Slow left-arm orthodox       9.900          Orthodox\n    13513        2                84       Kieron Pollard  Right-arm medium-fast      10.000        Off Cutter\n    16316        1               111         Chris Jordan  Right-arm fast-medium      10.000          S


ALL SPEEDS ABOVE 200


' match_id  innings  innings_ball_num        bowler_name           bowling_type  bowl_speed   delivery_style\n    14144        2                20      Khaleel Ahmed        Left-arm medium   1,456.100        Out Swing\n    19960        1               124     Sisanda Magala  Right-arm fast-medium   1,410.000        Full Toss\n    16259        2                12        Jacob Duffy  Right-arm fast-medium   1,351.000          Inswing\n    14116        2               116      Kagiso Rabada         Right-arm Fast   1,339.600        Full Toss\n    14122        2               115       Dwayne Bravo  Right-arm medium-fast   1,305.000 Reverse Swing In\n    21178        1                35    Benjamin Lister        Left-arm medium   1,257.000        Seamed In\n    17764        1               119     Mitchell Starc          Left-arm Fast   1,246.000       Off Cutter\n    21176        2                94      Dasun Shanaka       Right-arm medium   1,229.700         Straight\n    13524        1


SPEEDS BETWEEN 160 AND 200


' match_id  innings  innings_ball_num bowler_name   bowling_type  bowl_speed delivery_style\n    28963        1               111  Haris Rauf Right-arm Fast     177.100     Off Cutter'


ALL MIDNIGHT TIMESTAMP RECORDS


' match_id  innings  innings_ball_num  overid  over_ball_num       delivery_ist_time     striker_name      bowler_name match_date  venue_country   home_team   away_team\n    13511        1                 1       1              1 2022-01-23 00:00:00 UTC        Jason Roy Sheldon Cottrell 2022-01-22    West Indies West Indies     England\n    13511        1                 2       1              2 2022-01-23 00:00:00 UTC        Jason Roy Sheldon Cottrell 2022-01-22    West Indies West Indies     England\n    13511        1                 3       1              3 2022-01-23 00:00:00 UTC        Jason Roy Sheldon Cottrell 2022-01-22    West Indies West Indies     England\n    13511        1                 4       1              4 2022-01-23 00:00:00 UTC        Jason Roy Sheldon Cottrell 2022-01-22    West Indies West Indies     England\n    13511        1                 5       1              5 2022-01-23 00:00:00 UTC        Jason Roy Sheldon Cottrell 2022-01-22    West Indies West Indie


MATCH 25158 — MIDNIGHT RECORD


' match_id  innings  overid  over_ball_num  innings_ball_num  batting_team_id  bowling_team_id  striker_id striker_name  non_striker_id non_striker_name  bowler_id  bowler_name batting_type   bowling_type  batting_order  runs  is_four  is_six  is_wicket  dismissal_type_id dismissal_type  dismissed_player_id extras_type  extras1_runs  extras_2_runs pace_or_spin  bowl_speed      stroke delivery_style over_or_around on_or_off       delivery_ist_time\n    25158        1      20              5               120               17               22         278  Tim Southee            2526        Ish Sodhi       6097 Brydon Carse          RHB Right-arm Fast             10     1    False   False      False                NaN            NaN                  NaN         NaN             6              0         Pace     123.000 Cover Drive     Off Cutter           Over  Off-side 2023-08-31 00:00:00 UTC'

### Check whether bowling-speed anomalies are isolated or systematic

In [40]:
speed_by_bowler = (
    ball_by_ball.groupby(["bowler_id", "bowler_name", "bowling_type"])
    .agg(
        deliveries=("bowl_speed", "size"),
        zero_speed=("bowl_speed", lambda x: (x == 0).sum()),
        low_speed=("bowl_speed", lambda x: ((x > 0) & (x < 50)).sum()),
        normal_range=("bowl_speed", lambda x: x.between(50, 160).sum()),
        high_speed=("bowl_speed", lambda x: (x > 160).sum()),
        median_speed=("bowl_speed", "median"),
        min_speed=("bowl_speed", "min"),
        max_speed=("bowl_speed", "max")
    )
    .reset_index()
)

speed_by_bowler["zero_pct"] = (
    speed_by_bowler["zero_speed"] / speed_by_bowler["deliveries"] * 100
).round(1)

speed_by_bowler["anomaly_count"] = (
    speed_by_bowler["low_speed"] + speed_by_bowler["high_speed"]
)

print("BOWLERS WITH AT LEAST ONE NON-ZERO SPEED OUTSIDE 50–160")
display(
    speed_by_bowler[speed_by_bowler["anomaly_count"] > 0]
    .sort_values(["anomaly_count", "deliveries"], ascending=False)
    .to_string(index=False)
)

print("\nOVERALL SPEED COVERAGE BY BOWLING TYPE")
display(
    ball_by_ball.groupby("bowling_type")
    .agg(
        deliveries=("bowl_speed", "size"),
        zero_speed=("bowl_speed", lambda x: (x == 0).sum()),
        low_speed=("bowl_speed", lambda x: ((x > 0) & (x < 50)).sum()),
        high_speed=("bowl_speed", lambda x: (x > 160).sum())
    )
    .to_string()
)

BOWLERS WITH AT LEAST ONE NON-ZERO SPEED OUTSIDE 50–160


' bowler_id          bowler_name           bowling_type  deliveries  zero_speed  low_speed  normal_range  high_speed  median_speed  min_speed  max_speed  zero_pct  anomaly_count\n      2436        Dasun Shanaka       Right-arm medium         557         181          0           373           3       113.000      0.000  1,229.700    32.500              3\n       360    Bhuvneshwar Kumar       Right-arm medium        2314         632          1          1680           1       129.900      0.000    201.000    27.300              2\n      4373    Mustafizur Rahman        Left-arm medium         927         287          2           638           0       122.700      0.000    143.300    31.000              2\n      4823    Washington Sundar     Right-arm offbreak         806         237          0           567           2        92.100      0.000    999.000    29.400              2\n      6464       Arshdeep Singh   Left-arm medium-fast        3432         959          1          2472      


OVERALL SPEED COVERAGE BY BOWLING TYPE


'                        deliveries  zero_speed  low_speed  high_speed\nbowling_type                                                         \nLeft-arm Chinaman             4685        1606          0           0\nLeft-arm Fast                 4453        1924          1           1\nLeft-arm fast-medium          7780        3191          1           0\nLeft-arm medium               6082        2119          2           2\nLeft-arm medium-fast         12492        4154          5           0\nRight-arm Fast               26637       10013          1           4\nRight-arm Leg-Break          28027       10169          1           1\nRight-arm fast-medium        15380        6140          3           2\nRight-arm medium             23919       10231          2           4\nRight-arm medium-fast        27014       10107          3           1\nRight-arm offbreak           20948        8751          0           3\nSlow left-arm orthodox       23582        9456          3           2'

### Check match-level delivery completeness

In [41]:
# Match-innings delivery coverage
innings_audit = (
    ball_by_ball.groupby(["match_id", "innings"])
    .agg(
        delivery_records=("innings_ball_num", "size"),
        last_delivery_number=("innings_ball_num", "max"),
        distinct_batters=("striker_id", "nunique"),
        distinct_bowlers=("bowler_id", "nunique"),
        total_runs_recorded=("runs", "sum"),
        wickets_recorded=("is_wicket", "sum")
    )
    .reset_index()
)

# Add match context
innings_audit = innings_audit.merge(
    matches[[
        "match_id", "comp_id", "match_number", "match_date",
        "home_team", "away_team", "result"
    ]],
    on="match_id",
    how="left"
)

print("INNINGS DELIVERY RECORD COUNT DISTRIBUTION")
display(innings_audit["delivery_records"].describe())

print("\nINNINGS WITH FEWER THAN 30 DELIVERY RECORDS")
display(
    innings_audit[innings_audit["delivery_records"] < 30]
    .sort_values(["delivery_records", "match_id"])
    .to_string(index=False)
)

print("\nINNINGS WITH MORE THAN 150 DELIVERY RECORDS")
display(
    innings_audit[innings_audit["delivery_records"] > 150]
    .sort_values("delivery_records", ascending=False)
    .to_string(index=False)
)

INNINGS DELIVERY RECORD COUNT DISTRIBUTION


,delivery_records
count,"1,724.000"
mean,116.589
std,19.697
min,1.000
25%,117.000
50%,123.000
75%,126.000
max,136.000



INNINGS WITH FEWER THAN 30 DELIVERY RECORDS


' match_id  innings  delivery_records  last_delivery_number  distinct_batters  distinct_bowlers  total_runs_recorded  wickets_recorded  comp_id  match_number match_date             home_team             away_team                                                            result\n    34429        4                 1                     1                 1                 1                    4                 0     2542             3 2024-07-30             Sri Lanka                 India  India Tied with Srilanka (India Won the Super Over by 2 wickets)\n    45512        4                 1                     1                 1                 1                    3                 0     3133            18 2025-09-26                 India             Sri Lanka                              India won the Super Over(Match Tied)\n    50359        4                 1                     1                 1                 1                    4                 0     3298            38 2026-


INNINGS WITH MORE THAN 150 DELIVERY RECORDS


'Empty DataFrame\nColumns: [match_id, innings, delivery_records, last_delivery_number, distinct_batters, distinct_bowlers, total_runs_recorded, wickets_recorded, comp_id, match_number, match_date, home_team, away_team, result]\nIndex: []'

In [42]:
# Inspect innings with fewer than 30 records, with all match innings shown
short_ids = innings_audit.loc[
    innings_audit["delivery_records"] < 30, "match_id"
].unique()

print("ALL INNINGS FOR MATCHES WITH SHORT INNINGS")
display(
    innings_audit[innings_audit["match_id"].isin(short_ids)]
    .sort_values(["match_id", "innings"])
    .to_string(index=False)
)

print("\nMATCH METADATA FOR THESE MATCHES")
display(
    matches[matches["match_id"].isin(short_ids)][[
        "match_id", "match_date", "home_team", "away_team",
        "first_inn_revised_overs", "second_inn_revised_overs",
        "revised_target", "result"
    ]].sort_values("match_id").to_string(index=False)
)

ALL INNINGS FOR MATCHES WITH SHORT INNINGS


' match_id  innings  delivery_records  last_delivery_number  distinct_batters  distinct_bowlers  total_runs_recorded  wickets_recorded  comp_id  match_number match_date             home_team             away_team                                                            result\n    15410        1                22                    22                 4                 3                   26                 2     1439             5 2022-06-19                 India          South Africa                                                   Match Abandoned\n    17765        1                75                    75                 4                 5                  105                 2     1611             3 2022-10-14             Australia               England                                                         No Result\n    17765        2                25                    25                 5                 3                   23                 3     1611             3 2022-


MATCH METADATA FOR THESE MATCHES


' match_id match_date             home_team             away_team  first_inn_revised_overs  second_inn_revised_overs  revised_target                                                            result\n    15410 2022-06-19                 India          South Africa                  114.000                       NaN             NaN                                                   Match Abandoned\n    17765 2022-10-14             Australia               England                   72.000                    72.000         130.000                                                         No Result\n    31707 2024-04-18              Pakistan           New Zealand                   30.000                       NaN             NaN                                                         No Result\n    34429 2024-07-30             Sri Lanka                 India                      NaN                       NaN             NaN  India Tied with Srilanka (India Won the Super Over by 2 wickets)\n    

#### Classify short innings

In [44]:
# Rebuild short innings with match metadata
short_innings = innings_audit[
    innings_audit["delivery_records"] < 30
].copy()

short_innings = short_innings.merge(
    matches[[
        "match_id",
        "first_inn_revised_overs",
        "second_inn_revised_overs",
        "revised_target"
    ]],
    on="match_id",
    how="left"
)

short_innings["match_result_lower"] = (
    short_innings["result"].fillna("").str.lower()
)

short_innings["short_innings_category"] = np.select(
    [
        short_innings["innings"].isin([3, 4]),
        short_innings["match_result_lower"].str.contains(
            "abandoned|no result"
        ),
        short_innings["first_inn_revised_overs"].notna() |
        short_innings["second_inn_revised_overs"].notna()
    ],
    [
        "Super Over / extra innings",
        "Abandoned or No Result",
        "Revised overs recorded"
    ],
    default="Unexplained short innings"
)

display(
    short_innings[[
        "match_id", "innings", "delivery_records",
        "match_date", "home_team", "away_team",
        "result", "first_inn_revised_overs",
        "second_inn_revised_overs", "revised_target",
        "short_innings_category"
    ]].sort_values(["short_innings_category", "match_id"])
    .to_string(index=False)
)

print("\nCATEGORY COUNTS")
display(short_innings["short_innings_category"].value_counts())

' match_id  innings  delivery_records match_date             home_team             away_team                                                            result  first_inn_revised_overs  second_inn_revised_overs  revised_target     short_innings_category\n    15410        1                22 2022-06-19                 India          South Africa                                                   Match Abandoned                  114.000                       NaN             NaN     Abandoned or No Result\n    17765        2                25 2022-10-14             Australia               England                                                         No Result                   72.000                    72.000         130.000     Abandoned or No Result\n    31707        1                 2 2024-04-18              Pakistan           New Zealand                                                         No Result                   30.000                       NaN             NaN     Abandoned o


CATEGORY COUNTS


,count
short_innings_category,
Super Over / extra innings,12
Abandoned or No Result,8


### Check innings numbering and match structure

In [46]:
# Identify matches whose innings are not exactly [1, 2]
unusual_matches = match_innings_count[
    ~match_innings_count["innings_list"].apply(lambda x: x == [1, 2])
].copy()

print("UNUSUAL INNINGS STRUCTURE — MATCH DETAILS")

display(
    unusual_matches[
        ["match_id", "innings_count", "innings_list",
         "match_date", "home_team", "away_team", "result"]
    ].sort_values("match_id").to_string(index=False)
)

print("\nDISTRIBUTION OF UNUSUAL INNINGS PATTERNS")

display(
    unusual_matches["innings_list"]
    .astype(str)
    .value_counts()
)

UNUSUAL INNINGS STRUCTURE — MATCH DETAILS


' match_id  innings_count innings_list match_date             home_team                away_team                                                            result\n    15410              1          [1] 2022-06-19                 India             South Africa                                                   Match Abandoned\n    15546              1          [1] 2022-07-02           West Indies               Bangladesh                                                         No Result\n    20226              1          [1] 2023-05-03  Lucknow Super Giants      Chennai Super Kings                                                   Match Abandoned\n    21406              1          [1] 2023-04-20              Pakistan              New Zealand                                                   Match Abandoned\n    28333              1          [1] 2023-12-29           New Zealand               Bangladesh                                                         No Result\n    31707            


DISTRIBUTION OF UNUSUAL INNINGS PATTERNS


,count
innings_list,
[1],22
"[1, 2, 3, 4]",6


#### Verify the 22 one-innings matches

In [48]:
# Select the 22 matches with only innings 1
one_innings_matches = innings_audit[
    innings_audit["match_id"].isin(
        match_innings_count.loc[
            match_innings_count["innings_list"].apply(lambda x: x == [1]),
            "match_id"
        ]
    )
].copy()

# Add match metadata
one_innings_matches = one_innings_matches.merge(
    matches[[
        "match_id", "first_inn_revised_overs",
        "second_inn_revised_overs", "revised_target"
    ]],
    on="match_id",
    how="left"
)

print("ALL 22 MATCHES WITH ONLY ONE RECORDED INNINGS")

display(
    one_innings_matches[[
        "match_id", "match_date", "home_team", "away_team",
        "delivery_records", "result",
        "first_inn_revised_overs", "second_inn_revised_overs",
        "revised_target"
    ]].sort_values("match_id").to_string(index=False)
)

print("\nRESULT CATEGORY COUNTS")
display(one_innings_matches["result"].value_counts(dropna=False))

ALL 22 MATCHES WITH ONLY ONE RECORDED INNINGS


' match_id match_date             home_team                away_team  delivery_records          result  first_inn_revised_overs  second_inn_revised_overs  revised_target\n    15410 2022-06-19                 India             South Africa                22 Match Abandoned                  114.000                       NaN             NaN\n    15546 2022-07-02           West Indies               Bangladesh                82       No Result                   84.000                       NaN             NaN\n    20226 2023-05-03  Lucknow Super Giants      Chennai Super Kings               118 Match Abandoned                      NaN                       NaN             NaN\n    21406 2023-04-20              Pakistan              New Zealand               118 Match Abandoned                      NaN                       NaN             NaN\n    28333 2023-12-29           New Zealand               Bangladesh                66       No Result                      NaN                       


RESULT CATEGORY COUNTS


,count
result,
No Result,19
Match Abandoned,3


### Final delivery-level consistency check

In [49]:
# Inspect runs and extras distributions by innings
runs_extras_audit = (
    ball_by_ball.groupby(["innings", "extras_type"], dropna=False)
    .agg(
        deliveries=("runs", "size"),
        runs_sum=("runs", "sum"),
        extras2_sum=("extras_2_runs", "sum")
    )
    .reset_index()
)

print("RUNS AND EXTRAS BY INNINGS AND EXTRAS TYPE")
display(runs_extras_audit.to_string(index=False))

print("\nRUNS DISTRIBUTION FOR DELIVERIES WITH NO EXTRAS")
display(
    ball_by_ball.loc[
        ball_by_ball["extras_type"].isna(), "runs"
    ].value_counts().sort_index()
)

RUNS AND EXTRAS BY INNINGS AND EXTRAS TYPE


' innings extras_type  deliveries  runs_sum  extras2_sum\n       1    Overstep         310       475           29\n       1        byes         301         0            0\n       1     legbyes        1459         1            0\n       1       other         139       225           17\n       1        wide        4088         0            0\n       1         NaN       99430    142352            1\n       2    Overstep         266       377           19\n       2     Penalty           1         0            0\n       2        byes         228         0            0\n       2     legbyes        1288         0            0\n       2       other         124       183           18\n       2        wide        3622         0            0\n       2         NaN       89705    125893            0\n       3       other           1         4            0\n       3        wide           3         0            0\n       3         NaN          21        10            0\n       4     legbyes          


RUNS DISTRIBUTION FOR DELIVERIES WITH NO EXTRAS


,count
runs,
0,68029
1,72043
2,12496
3,653
4,23180
5,25
6,12741


#### Inspect actual delivery records

In [50]:
# Inspect deliveries with extras and unusual run combinations

print("OVERSTEP RECORDS")
display(
    ball_by_ball[ball_by_ball["extras_type"] == "Overstep"][
        ["match_id", "innings", "striker_name", "bowler_name",
         "runs", "extras1_runs", "extras_2_runs",
         "extras_type", "is_four", "is_six"]
    ].head(15).to_string(index=False)
)

print("\nOTHER EXTRAS RECORDS")
display(
    ball_by_ball[ball_by_ball["extras_type"] == "other"][
        ["match_id", "innings", "striker_name", "bowler_name",
         "runs", "extras1_runs", "extras_2_runs",
         "extras_type", "is_four", "is_six"]
    ].head(15).to_string(index=False)
)

print("\nDELIVERIES WITH 5 RUNS AND NO EXTRAS")
display(
    ball_by_ball[
        ball_by_ball["runs"].eq(5) &
        ball_by_ball["extras_type"].isna()
    ][
        ["match_id", "innings", "striker_name", "bowler_name",
         "runs", "extras_2_runs", "is_four", "is_six",
         "stroke", "delivery_style"]
    ].to_string(index=False)
)

OVERSTEP RECORDS


' match_id  innings     striker_name       bowler_name  runs  extras1_runs  extras_2_runs extras_type  is_four  is_six\n    13513        1  Nicholas Pooran     George Garton     0             1              0    Overstep    False   False\n    13515        2     Sam Billings      Jason Holder     1             1              0    Overstep    False   False\n    13523        1     Ishan Kishan      Akeal Hosein     0             1              4    Overstep    False   False\n    13523        1     Ishan Kishan      Akeal Hosein     1             1              0    Overstep    False   False\n    14094        1  Ruturaj Gaikwad       Umesh Yadav     0             1              0    Overstep    False   False\n    14098        1      Jos Buttler Bhuvneshwar Kumar     0             1              0    Overstep    False   False\n    14098        1 Yashasvi Jaiswal Bhuvneshwar Kumar     0             1              0    Overstep    False   False\n    14098        1      Jos Buttler       Umran


OTHER EXTRAS RECORDS


' match_id  innings         striker_name         bowler_name  runs  extras1_runs  extras_2_runs extras_type  is_four  is_six\n    13513        1        Rovman Powell         Tymal Mills     0             5              0       other    False   False\n    13515        1        Rovman Powell       Saqib Mahmood     0             5              0       other    False   False\n    13515        2         Sam Billings    Sheldon Cottrell     2             5              0       other    False   False\n    13614        1        Dasun Shanaka      Mitchell Starc     0             5              4       other    False   False\n    13614        2          Ashton Agar      Nuwan Thushara     0             5              0       other    False   False\n    14094        1 Mahendra Singh Dhoni         Shivam Mavi     6             5              0       other    False    True\n    14098        2    Bhuvneshwar Kumar Nathan Coulter Nile     0             5              0       other    False   False\


DELIVERIES WITH 5 RUNS AND NO EXTRAS


' match_id  innings         striker_name        bowler_name  runs  extras_2_runs  is_four  is_six              stroke delivery_style\n    16317        2            Jason Roy      Anrich Nortje     5              0    False   False               Steer      Out Swing\n    17557        2           Sam Curran        Shadab Khan     5              0    False   False              Worked         Googly\n    17738        1      Kane Williamson         Haris Rauf     5              0    False   False                Push        Inswing\n    20186        2          Kyle Mayers          Moeen Ali     5              0    False   False         Cover Drive      Off Break\n    28960        2          Shivam Dube      Mohammad Nabi     5              0    False   False          Leg Glance      Off Break\n    29924        2       Ibrahim Zadran    Binura Fernando     5              0    False   False         Cover Drive      Full Toss\n    30076        1       Marcus Stoinis        Umesh Yadav     5    

#### Inspect the five-run deliveries completely

In [51]:
five_run_records = ball_by_ball[
    ball_by_ball["runs"].eq(5) &
    ball_by_ball["extras_type"].isna()
]

print("COUNT:", len(five_run_records))

display(
    five_run_records[[
        "match_id", "innings", "overid", "over_ball_num",
        "striker_name", "non_striker_name", "bowler_name",
        "runs", "extras1_runs", "extras_2_runs",
        "extras_type", "is_four", "is_six",
        "stroke", "delivery_style"
    ]].to_string(index=False)
)

COUNT: 25


' match_id  innings  overid  over_ball_num         striker_name     non_striker_name        bowler_name  runs  extras1_runs  extras_2_runs extras_type  is_four  is_six              stroke delivery_style\n    16317        2       2              4            Jason Roy          Jos Buttler      Anrich Nortje     5             6              0         NaN    False   False               Steer      Out Swing\n    17557        2      13              2           Sam Curran            Moeen Ali        Shadab Khan     5             6              0         NaN    False   False              Worked         Googly\n    17738        1       6              6      Kane Williamson         Devon Conway         Haris Rauf     5             6              0         NaN    False   False                Push        Inswing\n    20186        2       6              1          Kyle Mayers         Lokesh Rahul          Moeen Ali     5             6              0         NaN    False   False         Cover Drive 

In [52]:
print("MATCHES DATASET COLUMNS")
for col in matches.columns:
    print(col)

MATCHES DATASET COLUMNS
match_id
comp_id
match_number
match_date
stage
venue_id
venue_country
home_team_id
home_team
away_team_id
away_team
home_team_captain_id
away_team_captain_id
toss_win_team_id
toss_opted
man_of_the_match
first_inn_revised_overs
second_inn_revised_overs
revised_target
knock_out
win_team_id
result


# Playing-XI audit

### Playing-XI field distributions

In [61]:
print("PLAYING XI SHAPE:", playing_eleven.shape)

print("\nMISSING VALUES")
display(
    playing_eleven.isna().sum().to_frame("missing_count")
)

print("\nBOOLEAN / FLAG DISTRIBUTIONS")

for col in ["is_played", "is_captain", "is_wk",
            "is_impact", "is_impact_off"]:
    print(f"\n{col}")
    display(
        playing_eleven[col]
        .value_counts(dropna=False)
        .to_frame("record_count")
    )

print("\nBATTING ORDER DISTRIBUTION")
display(
    playing_eleven["batting_order"]
    .value_counts(dropna=False)
    .sort_index()
    .to_frame("record_count")
)

PLAYING XI SHAPE: (28381, 9)

MISSING VALUES


,missing_count
match_id,0
team_id,0
player_id,0
batting_order,8678
is_played,0
is_captain,8667
is_wk,8667
is_impact,0
is_impact_off,0



BOOLEAN / FLAG DISTRIBUTIONS

is_played


,record_count
is_played,
True,19714
False,8667



is_captain


,record_count
is_captain,
False,18031
NaN,8667
True,1683


... (4 outputs left out)


is_impact_off


,record_count
is_impact_off,
0,27823
1,558



BATTING ORDER DISTRIBUTION


,record_count
batting_order,
1.000,1743
2.000,1743
3.000,1744
4.000,1742
5.000,1741
6.000,1741
7.000,1735
8.000,1736
9.000,1736


### Validate playing status, impact flags and batting order

In [62]:
# 1. Relationship between is_played and missing batting order
print("IS_PLAYED vs BATTING ORDER MISSING")

display(pd.crosstab(
    playing_eleven["is_played"],
    playing_eleven["batting_order"].isna(),
    margins=True
))

# 2. Relationship between impact flags
print("\nIS_IMPACT vs IS_IMPACT_OFF")

display(pd.crosstab(
    playing_eleven["is_impact"],
    playing_eleven["is_impact_off"],
    margins=True
))

print(
    "\nRows where impact flags differ:",
    (playing_eleven["is_impact"] !=
     playing_eleven["is_impact_off"]).sum()
)

# 3. Impact flag vs playing status
print("\nIMPACT FLAGS vs PLAYING STATUS")

display(pd.crosstab(
    playing_eleven["is_played"],
    playing_eleven["is_impact"],
    margins=True
))

# 4. Inspect batting orders 12 and 13
print("\nRECORDS WITH BATTING ORDER 12 OR 13")

display(
    playing_eleven[
        playing_eleven["batting_order"].isin([12, 13])
    ].sort_values(
        ["match_id", "team_id", "batting_order"]
    ).to_string(index=False)
)

# 5. Inspect played records with missing batting order
print("\nPLAYED RECORDS WITH MISSING BATTING ORDER")

display(
    playing_eleven[
        (playing_eleven["is_played"] == True) &
        playing_eleven["batting_order"].isna()
    ].to_string(index=False)
)

IS_PLAYED vs BATTING ORDER MISSING


batting_order,False,True,All
is_played,,,
False,0,8667,8667
True,19703,11,19714
All,19703,8678,28381



IS_IMPACT vs IS_IMPACT_OFF


is_impact_off,0,1,All
is_impact,,,
0,27265,558,27823
1,558,0,558
All,27823,558,28381



Rows where impact flags differ: 1116

IMPACT FLAGS vs PLAYING STATUS


is_impact,0,1,All
is_played,,,
False,8667,0,8667
True,19156,558,19714
All,27823,558,28381



RECORDS WITH BATTING ORDER 12 OR 13


' match_id  team_id  player_id  batting_order  is_played is_captain is_wk  is_impact  is_impact_off\n    14147        8      29888         12.000       True      False False          0              0\n    20181        1       6806         12.000       True      False False          1              0\n    20181     3508       1566         12.000       True      False False          0              0\n    20182        4       1617         12.000       True      False False          1              0\n    20182        8        278         12.000       True      False False          0              0\n    20183        7       4816         12.000       True      False False          0              1\n    20183     3509       2985         12.000       True      False False          0              0\n    20184        3       4288         12.000       True      False False          1              0\n    20184      242       7240         12.000       True      False False          0              1\


PLAYED RECORDS WITH MISSING BATTING ORDER


' match_id  team_id  player_id  batting_order  is_played is_captain is_wk  is_impact  is_impact_off\n    48512       25       7690            NaN       True      False False          0              0\n    48512       25       3091            NaN       True      False False          0              0\n    48512       25      20327            NaN       True      False  True          0              0\n    48512       25       7696            NaN       True      False False          0              0\n    48512       25      20704            NaN       True      False False          0              0\n    48512       25       2921            NaN       True      False False          0              0\n    48512       25       2490            NaN       True      False False          0              0\n    48512       25       5979            NaN       True      False False          0              0\n    48512       25       4926            NaN       True      False False          0              0\

### Investigate impact substitutions and unusual batting orders

In [63]:
# 1. Inspect the 11 played records with missing batting order
print("PLAYED RECORDS WITH MISSING BATTING ORDER")

missing_order = playing_eleven[
    (playing_eleven["is_played"] == True) &
    playing_eleven["batting_order"].isna()
]

display(
    missing_order.merge(
        matches[["match_id", "match_date", "home_team", "away_team", "result"]],
        on="match_id",
        how="left"
    ).to_string(index=False)
)

# 2. Inspect every impact-related record
print("\nALL IMPACT-RELATED RECORDS")

impact_records = playing_eleven[
    (playing_eleven["is_impact"] == 1) |
    (playing_eleven["is_impact_off"] == 1)
].copy()

impact_records = impact_records.merge(
    matches[["match_id", "match_date", "home_team", "away_team", "result"]],
    on="match_id",
    how="left"
)

display(
    impact_records.sort_values(
        ["match_id", "team_id", "is_impact", "is_impact_off"]
    ).to_string(index=False)
)

# 3. Count impact flags by match and team
print("\nMATCH-TEAM IMPACT FLAG COUNTS")

impact_team_summary = (
    playing_eleven.groupby(["match_id", "team_id"])
    .agg(
        impact_on=("is_impact", "sum"),
        impact_off=("is_impact_off", "sum"),
        played=("is_played", "sum"),
        squad_records=("player_id", "size")
    )
    .reset_index()
)

display(
    impact_team_summary[
        (impact_team_summary["impact_on"] > 0) |
        (impact_team_summary["impact_off"] > 0)
    ].to_string(index=False)
)

# 4. Inspect batting order 13
print("\nBATTING ORDER 13 RECORDS")

display(
    playing_eleven[
        playing_eleven["batting_order"] == 13
    ].merge(
        matches[["match_id", "match_date", "home_team", "away_team", "result"]],
        on="match_id",
        how="left"
    ).to_string(index=False)
)

PLAYED RECORDS WITH MISSING BATTING ORDER


' match_id  team_id  player_id  batting_order  is_played is_captain is_wk  is_impact  is_impact_off match_date home_team away_team                    result\n    48512       25       7690            NaN       True      False False          0              0 2026-01-07 Sri Lanka  Pakistan Pakistan Won by 6 wickets\n    48512       25       3091            NaN       True      False False          0              0 2026-01-07 Sri Lanka  Pakistan Pakistan Won by 6 wickets\n    48512       25      20327            NaN       True      False  True          0              0 2026-01-07 Sri Lanka  Pakistan Pakistan Won by 6 wickets\n    48512       25       7696            NaN       True      False False          0              0 2026-01-07 Sri Lanka  Pakistan Pakistan Won by 6 wickets\n    48512       25      20704            NaN       True      False False          0              0 2026-01-07 Sri Lanka  Pakistan Pakistan Won by 6 wickets\n    48512       25       2921            NaN       True  


ALL IMPACT-RELATED RECORDS


' match_id  team_id  player_id  batting_order  is_played is_captain is_wk  is_impact  is_impact_off match_date                   home_team                   away_team                                                            result\n    20181        1        535          5.000       True      False False          0              1 2023-03-31              Gujarat Titans         Chennai Super Kings                                   Gujarat Titans Won by 5 wickets\n    20181        1       6806         12.000       True      False False          1              0 2023-03-31              Gujarat Titans         Chennai Super Kings                                   Gujarat Titans Won by 5 wickets\n    20181     3508       1568          8.000       True      False False          0              1 2023-03-31              Gujarat Titans         Chennai Super Kings                                   Gujarat Titans Won by 5 wickets\n    20181     3508      23450          3.000       True      False 


MATCH-TEAM IMPACT FLAG COUNTS


' match_id  team_id  impact_on  impact_off  played  squad_records\n    20181        1          1           1      12             12\n    20181     3508          1           1      12             12\n    20182        4          1           1      12             12\n    20182        8          1           1      12             12\n    20183        7          1           1      12             12\n    20183     3509          1           1      12             12\n    20184        3          1           1      12             12\n    20184      242          1           1      12             12\n    20185        2          1           1      12             12\n    20186        1          1           1      12             12\n    20186     3509          1           1      12             12\n    20187        7          1           1      12             12\n    20187     3508          1           1      12             12\n    20188        3          1           1      12             12\n    20188


BATTING ORDER 13 RECORDS


' match_id  team_id  player_id  batting_order  is_played is_captain is_wk  is_impact  is_impact_off match_date             home_team             away_team                                       result\n    21585        2       8177         13.000       True      False False          0              1 2023-05-26        Gujarat Titans        Mumbai Indians                Gujarat Titans won by 62 runs\n    31272     3509       6287         13.000       True      False False          0              1 2024-05-05  Lucknow Super Giants Kolkata Knight Riders         Kolkata Knight Riders Won by 98 runs\n    40749        2       8634         13.000       True      False False          0              0 2025-05-06        Mumbai Indians        Gujarat Titans Gujarat Titans Won by 3 wickets (DLS Method)\n    48029     3093      24767         13.000       True      False False          0              0 2025-11-23          Bangladesh A            Pakistan A                Pakistan A Won the super over\

### Resolve playing-XI exceptions and test impact-flag patterns

In [64]:
# Player names for readable output
player_names = (
    competition_team_player_mapping[["player_id", "player_name"]]
    .drop_duplicates()
    .drop_duplicates(subset="player_id")
)

# 1. All played records with missing batting order
missing_order = playing_eleven[
    (playing_eleven["is_played"] == True) &
    playing_eleven["batting_order"].isna()
].merge(player_names, on="player_id", how="left")

print("ALL PLAYED RECORDS WITH MISSING BATTING ORDER")
display(missing_order.sort_values(["match_id", "team_id", "player_id"]).to_string(index=False))

# 2. Full playing-XI records for match 48512
match_48512_xi = playing_eleven[
    playing_eleven["match_id"] == 48512
].merge(player_names, on="player_id", how="left")

print("\nFULL PLAYING-XI RECORDS FOR MATCH 48512")
display(match_48512_xi.sort_values(["team_id", "batting_order", "player_id"]).to_string(index=False))

# 3. Summarize impact flags by match and team
impact_summary = playing_eleven.groupby(["match_id", "team_id"]).agg(
    impact_on=("is_impact", "sum"),
    impact_off=("is_impact_off", "sum"),
    played=("is_played", "sum"),
    squad_records=("player_id", "size")
).reset_index()

print("\nIMPACT FLAG PAIRING PATTERNS")
display(
    impact_summary[
        (impact_summary["impact_on"] > 0) |
        (impact_summary["impact_off"] > 0)
    ].groupby(["impact_on", "impact_off"]).size()
    .reset_index(name="match_team_count")
    .sort_values(["impact_on", "impact_off"])
    .to_string(index=False)
)

print("\nMATCH-TEAM GROUPS WITH UNPAIRED OR MULTIPLE IMPACT FLAGS")
display(
    impact_summary[
        ((impact_summary["impact_on"] > 0) |
         (impact_summary["impact_off"] > 0)) &
        (
            (impact_summary["impact_on"] != 1) |
            (impact_summary["impact_off"] != 1)
        )
    ].to_string(index=False)
)

# 4. Show impact-flagged players with names and match context
impact_detail = playing_eleven[
    (playing_eleven["is_impact"] == 1) |
    (playing_eleven["is_impact_off"] == 1)
].merge(player_names, on="player_id", how="left").merge(
    matches[["match_id", "match_date", "home_team", "away_team", "result"]],
    on="match_id", how="left"
)

print("\nIMPACT-FLAGGED PLAYER RECORDS — FIRST 40")
display(
    impact_detail.sort_values(["match_id", "team_id", "is_impact", "is_impact_off"])
    .head(40).to_string(index=False)
)

ALL PLAYED RECORDS WITH MISSING BATTING ORDER


' match_id  team_id  player_id  batting_order  is_played is_captain is_wk  is_impact  is_impact_off      player_name\n    48512       25       2490            NaN       True      False False          0              0   Mohammad Nawaz\n    48512       25       2921            NaN       True      False False          0              0     Fakhar Zaman\n    48512       25       3091            NaN       True      False False          0              0 Sahibzada Farhan\n    48512       25       3489            NaN       True       True False          0              0      Salman Agha\n    48512       25       4926            NaN       True      False False          0              0      Shadab Khan\n    48512       25       5189            NaN       True      False False          0              0    Faheem Ashraf\n    48512       25       5979            NaN       True      False False          0              0      Abrar Ahmed\n    48512       25       7690            NaN       True      Fa


FULL PLAYING-XI RECORDS FOR MATCH 48512


' match_id  team_id  player_id  batting_order  is_played is_captain is_wk  is_impact  is_impact_off         player_name\n    48512       25       2490            NaN       True      False False          0              0      Mohammad Nawaz\n    48512       25       2921            NaN       True      False False          0              0        Fakhar Zaman\n    48512       25       3091            NaN       True      False False          0              0    Sahibzada Farhan\n    48512       25       3489            NaN       True       True False          0              0         Salman Agha\n    48512       25       4926            NaN       True      False False          0              0         Shadab Khan\n    48512       25       5189            NaN       True      False False          0              0       Faheem Ashraf\n    48512       25       5979            NaN       True      False False          0              0         Abrar Ahmed\n    48512       25       6922          


IMPACT FLAG PAIRING PATTERNS


' impact_on  impact_off  match_team_count\n         1           1               558'


MATCH-TEAM GROUPS WITH UNPAIRED OR MULTIPLE IMPACT FLAGS


'Empty DataFrame\nColumns: [match_id, team_id, impact_on, impact_off, played, squad_records]\nIndex: []'


IMPACT-FLAGGED PLAYER RECORDS — FIRST 40


' match_id  team_id  player_id  batting_order  is_played is_captain is_wk  is_impact  is_impact_off        player_name match_date                   home_team                   away_team                                       result\n    20181        1        535          5.000       True      False False          0              1      Ambati Rayudu 2023-03-31              Gujarat Titans         Chennai Super Kings              Gujarat Titans Won by 5 wickets\n    20181        1       6806         12.000       True      False False          1              0   Tushar Deshpande 2023-03-31              Gujarat Titans         Chennai Super Kings              Gujarat Titans Won by 5 wickets\n    20181     3508       1568          8.000       True      False False          0              1    Kane Williamson 2023-03-31              Gujarat Titans         Chennai Super Kings              Gujarat Titans Won by 5 wickets\n    20181     3508      23450          3.000       True      False False   

### Impact-player participation audit

In [65]:
# Count each player's delivery involvement by match
striker_counts = (
    ball_by_ball.groupby(["match_id", "striker_id"])
    .size().rename("as_striker").reset_index()
    .rename(columns={"striker_id": "player_id"})
)

non_striker_counts = (
    ball_by_ball.groupby(["match_id", "non_striker_id"])
    .size().rename("as_non_striker").reset_index()
    .rename(columns={"non_striker_id": "player_id"})
)

bowler_counts = (
    ball_by_ball.groupby(["match_id", "bowler_id"])
    .size().rename("as_bowler").reset_index()
    .rename(columns={"bowler_id": "player_id"})
)

# Combine participation counts
participation = striker_counts.merge(
    non_striker_counts, on=["match_id", "player_id"], how="outer"
).merge(
    bowler_counts, on=["match_id", "player_id"], how="outer"
)

role_cols = ["as_striker", "as_non_striker", "as_bowler"]
participation[role_cols] = participation[role_cols].fillna(0).astype(int)
participation["total_delivery_involvement"] = participation[role_cols].sum(axis=1)

# Attach names, flags, and match context
impact_detail = playing_eleven[
    (playing_eleven["is_impact"] == 1) |
    (playing_eleven["is_impact_off"] == 1)
].merge(player_names, on="player_id", how="left").merge(
    participation, on=["match_id", "player_id"], how="left"
).merge(
    matches[["match_id", "match_date", "home_team", "away_team"]],
    on="match_id", how="left"
)

impact_detail[role_cols + ["total_delivery_involvement"]] = (
    impact_detail[role_cols + ["total_delivery_involvement"]]
    .fillna(0).astype(int)
)

impact_detail["has_delivery_involvement"] = (
    impact_detail["total_delivery_involvement"] > 0
)

# Summary by flag
print("DELIVERY PARTICIPATION BY IMPACT FLAG")
display(
    impact_detail.groupby(
        ["is_impact", "is_impact_off"]
    ).agg(
        players=("player_id", "size"),
        with_delivery_involvement=("has_delivery_involvement", "sum"),
        without_delivery_involvement=(
            "has_delivery_involvement",
            lambda x: (~x).sum()
        ),
        striker_rows=("as_striker", "sum"),
        non_striker_rows=("as_non_striker", "sum"),
        bowler_rows=("as_bowler", "sum")
    ).reset_index().to_string(index=False)
)

# Pair both flagged players into one row per match-team
incoming = impact_detail[impact_detail["is_impact"] == 1].copy()
outgoing = impact_detail[impact_detail["is_impact_off"] == 1].copy()

pair_audit = incoming.merge(
    outgoing,
    on=["match_id", "team_id"],
    suffixes=("_impact", "_impact_off")
)

pair_columns = [
    "match_id", "team_id",
    "player_id_impact", "player_name_impact",
    "total_delivery_involvement_impact",
    "player_id_impact_off", "player_name_impact_off",
    "total_delivery_involvement_impact_off",
    "match_date_impact", "home_team_impact", "away_team_impact"
]

print("\nIMPACT PAIR PARTICIPATION — FIRST 30 PAIRS")
display(pair_audit[pair_columns].head(30).to_string(index=False))

print("\nPAIR PARTICIPATION SUMMARY")
display(
    pair_audit.assign(
        impact_has_deliveries=(
            pair_audit["total_delivery_involvement_impact"] > 0
        ),
        impact_off_has_deliveries=(
            pair_audit["total_delivery_involvement_impact_off"] > 0
        )
    ).groupby(
        ["impact_has_deliveries", "impact_off_has_deliveries"]
    ).size().reset_index(name="pair_count").to_string(index=False)
)

# Confirm exact missing batting-order count and match-team distribution
print("\nPLAYED RECORDS WITH MISSING BATTING ORDER")
display(
    missing_order.groupby(["match_id", "team_id"])
    .size().reset_index(name="records")
    .to_string(index=False)
)
print("Total:", len(missing_order))

DELIVERY PARTICIPATION BY IMPACT FLAG


' is_impact  is_impact_off  players  with_delivery_involvement  without_delivery_involvement  striker_rows  non_striker_rows  bowler_rows\n         0              1      558                        543                            15          5549              5508         6068\n         1              0      558                        535                            23          4805              4582         4710'


IMPACT PAIR PARTICIPATION — FIRST 30 PAIRS


' match_id  team_id  player_id_impact    player_name_impact  total_delivery_involvement_impact  player_id_impact_off player_name_impact_off  total_delivery_involvement_impact_off match_date_impact            home_team_impact            away_team_impact\n    20181        1              6806      Tushar Deshpande                                 22                   535          Ambati Rayudu                                     31        2023-03-31              Gujarat Titans         Chennai Super Kings\n    20181     3508             23450         Sai Sudharsan                                 35                  1568        Kane Williamson                                      0        2023-03-31              Gujarat Titans         Chennai Super Kings\n    20182        4              1617          Rishi Dhawan                                  6                  2003      Bhanuka Rajapaksa                                     56        2023-04-01                Punjab Kings       Kolkata Kn


PAIR PARTICIPATION SUMMARY


' impact_has_deliveries  impact_off_has_deliveries  pair_count\n                 False                       True          23\n                  True                      False          15\n                  True                       True         520'


PLAYED RECORDS WITH MISSING BATTING ORDER


' match_id  team_id  records\n    48512       25       11'

Total: 11


### Batting-order and roster-structure audit

In [66]:
# 1. Batting-order distribution, restricted to played records
played_xi = playing_eleven[playing_eleven["is_played"] == True].copy()

print("BATTING-ORDER DISTRIBUTION AMONG PLAYED RECORDS")
display(
    played_xi["batting_order"]
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("batting_order")
    .reset_index(name="records")
    .to_string(index=False)
)

# 2. Duplicate batting positions within a match-team
position_counts = (
    played_xi.dropna(subset=["batting_order"])
    .groupby(["match_id", "team_id", "batting_order"])
    .size()
    .reset_index(name="players_at_position")
)

duplicate_positions = position_counts[
    position_counts["players_at_position"] > 1
]

print("\nDUPLICATE BATTING POSITIONS WITHIN MATCH-TEAM")
print("Duplicate position groups:", len(duplicate_positions))
display(duplicate_positions.head(50).to_string(index=False))

# 3. Match-team roster sizes and played-player counts
roster_summary = playing_eleven.groupby(
    ["match_id", "team_id"]
).agg(
    total_records=("player_id", "size"),
    played_records=("is_played", "sum"),
    not_played_records=("is_played", lambda x: (~x).sum()),
    distinct_players=("player_id", "nunique"),
    batting_positions_present=("batting_order", "count"),
    impact_on=("is_impact", "sum"),
    impact_off=("is_impact_off", "sum")
).reset_index()

print("\nROSTER SIZE DISTRIBUTION")
display(
    roster_summary.groupby(
        ["total_records", "played_records"]
    ).size().reset_index(name="match_team_groups")
    .sort_values(["total_records", "played_records"])
    .to_string(index=False)
)

print("\nROSTER GROUPS WITH UNUSUAL COUNTS")
display(
    roster_summary[
        (roster_summary["total_records"] != 12) |
        (roster_summary["played_records"] > 12)
    ].to_string(index=False)
)

# 4. Inspect positions 12 and 13 with player names and match context
positions_12_13 = played_xi[
    played_xi["batting_order"].isin([12, 13])
].merge(
    player_names, on="player_id", how="left"
).merge(
    matches[["match_id", "match_date", "home_team", "away_team", "result"]],
    on="match_id", how="left"
)

print("\nPLAYED RECORDS AT BATTING POSITIONS 12 AND 13")
display(
    positions_12_13.sort_values(
        ["match_id", "team_id", "batting_order"]
    ).to_string(index=False)
)

print("\nPOSITION 12/13 FLAG SUMMARY")
display(
    positions_12_13.groupby(
        ["batting_order", "is_impact", "is_impact_off"]
    ).size().reset_index(name="records")
    .to_string(index=False)
)

BATTING-ORDER DISTRIBUTION AMONG PLAYED RECORDS


' batting_order  records\n         1.000     1743\n         2.000     1743\n         3.000     1744\n         4.000     1742\n         5.000     1741\n         6.000     1741\n         7.000     1735\n         8.000     1736\n         9.000     1736\n        10.000     1735\n        11.000     1736\n        12.000      564\n        13.000        7\n           NaN       11'


DUPLICATE BATTING POSITIONS WITHIN MATCH-TEAM
Duplicate position groups: 35


' match_id  team_id  batting_order  players_at_position\n    13613       20          1.000                    2\n    13613       20          2.000                    2\n    13613       20          3.000                    2\n    13613       20          4.000                    2\n    13613       20          5.000                    2\n    13613       20          6.000                    2\n    13613       27          1.000                    2\n    13613       27          2.000                    2\n    13613       27          3.000                    2\n    13613       27          4.000                    2\n    13613       27          5.000                    2\n    13613       27          6.000                    2\n    21176       17          1.000                    2\n    21176       17          2.000                    2\n    21176       17          3.000                    2\n    21176       17          4.000                    2\n    21176       17          5.000              


ROSTER SIZE DISTRIBUTION


' total_records  played_records  match_team_groups\n            10              10                  1\n            11              11                569\n            12              12                141\n            13              11                  6\n            13              13                  1\n            14               0                  1\n            14              11                 33\n            15               0                 10\n            15              11                205\n            16               0                  2\n            16              11                158\n            16              12                  1\n            16              16                  1\n            17               0                  1\n            17              11                 87\n            17              17                  3\n            18              11                 31\n            19              11                 16\n            19              12

... (2 outputs left out)


PLAYED RECORDS AT BATTING POSITIONS 12 AND 13


' match_id  team_id  player_id  batting_order  is_played is_captain is_wk  is_impact  is_impact_off           player_name match_date                   home_team                   away_team                                                            result\n    14147        8      29888         12.000       True      False False          0              0          Harshit Rana 2022-05-07        Lucknow Super Giants       Kolkata Knight Riders                               Lucknow Super Giants won by 75 runs\n    20181        1       6806         12.000       True      False False          1              0      Tushar Deshpande 2023-03-31              Gujarat Titans         Chennai Super Kings                                   Gujarat Titans Won by 5 wickets\n    20181     3508       1566         12.000       True      False False          0              0        Mohammed Shami 2023-03-31              Gujarat Titans         Chennai Super Kings                                   Gujarat Tita


POSITION 12/13 FLAG SUMMARY


' batting_order  is_impact  is_impact_off  records\n        12.000          0              0      163\n        12.000          0              1      170\n        12.000          1              0      231\n        13.000          0              0        4\n        13.000          0              1        3'

### Investigate duplicate batting positions and large rosters

In [67]:
# 1. Expand all duplicate batting-position groups
duplicate_details = played_xi.merge(
    duplicate_positions[
        ["match_id", "team_id", "batting_order"]
    ],
    on=["match_id", "team_id", "batting_order"],
    how="inner"
).merge(
    player_names, on="player_id", how="left"
).merge(
    matches[["match_id", "match_date", "home_team", "away_team"]],
    on="match_id", how="left"
)

print("ALL DUPLICATE BATTING-POSITION RECORDS")
display(
    duplicate_details.sort_values(
        ["match_id", "team_id", "batting_order", "player_id"]
    ).to_string(index=False)
)

# 2. Check whether duplicate positions involve impact flags
print("\nDUPLICATE POSITION GROUPS BY IMPACT-FLAG COMBINATION")
display(
    duplicate_details.groupby(
        ["is_impact", "is_impact_off"]
    ).size().reset_index(name="records").to_string(index=False)
)

# 3. Inspect the largest roster groups
largest_rosters = roster_summary[
    roster_summary["total_records"] >= 17
].copy()

largest_rosters = largest_rosters.merge(
    matches[["match_id", "match_date", "home_team", "away_team", "result"]],
    on="match_id", how="left"
)

print("\nROSTER GROUPS WITH 17 OR MORE RECORDS")
display(
    largest_rosters.sort_values(
        ["total_records", "match_id"], ascending=[False, True]
    ).to_string(index=False)
)

# 4. Inspect player-level records for those large rosters
large_roster_details = playing_eleven.merge(
    largest_rosters[["match_id", "team_id"]],
    on=["match_id", "team_id"],
    how="inner"
).merge(
    player_names, on="player_id", how="left"
)

print("\nPLAYER RECORDS IN LARGE ROSTERS")
display(
    large_roster_details.sort_values(
        ["match_id", "team_id", "is_played", "batting_order"],
        ascending=[True, True, False, True]
    ).to_string(index=False)
)

# 5. Check whether any player ID is repeated within a match-team
repeated_player_ids = (
    playing_eleven.groupby(["match_id", "team_id", "player_id"])
    .size().reset_index(name="records")
)

repeated_player_ids = repeated_player_ids[
    repeated_player_ids["records"] > 1
]

print("\nREPEATED PLAYER IDS WITHIN MATCH-TEAM")
print("Repeated player-ID groups:", len(repeated_player_ids))
display(repeated_player_ids.to_string(index=False))

ALL DUPLICATE BATTING-POSITION RECORDS


' match_id  team_id  player_id  batting_order  is_played is_captain is_wk  is_impact  is_impact_off          player_name match_date   home_team   away_team\n    13613       20        838          1.000       True      False False          0              0        Glenn Maxwell 2022-02-13   Australia   Sri Lanka\n    13613       20       3539          1.000       True      False False          0              0        Ben McDermott 2022-02-13   Australia   Sri Lanka\n    13613       20        250          2.000       True       True False          0              0          Aaron Finch 2022-02-13   Australia   Sri Lanka\n    13613       20       2639          2.000       True      False False          0              0       Marcus Stoinis 2022-02-13   Australia   Sri Lanka\n    13613       20       3539          3.000       True      False False          0              0        Ben McDermott 2022-02-13   Australia   Sri Lanka\n    13613       20       4736          3.000       True      Fa


DUPLICATE POSITION GROUPS BY IMPACT-FLAG COMBINATION


' is_impact  is_impact_off  records\n         0              0       75'


ROSTER GROUPS WITH 17 OR MORE RECORDS


' match_id  team_id  total_records  played_records  not_played_records  distinct_players  batting_positions_present  impact_on  impact_off match_date                   home_team                   away_team                                                            result\n    40756        1             27              12                  15                27                         12          1           1 2025-05-20         Chennai Super Kings            Rajasthan Royals                                 Rajasthan Royals Won by 6 wickets\n    31243        2             26              12                  14                26                         12          1           1 2024-04-11              Mumbai Indians Royal Challengers Bengaluru                                   Mumbai Indians Won by 7 wickets\n    40754        2             26              12                  14                26                         12          1           1 2025-05-26                Punjab Kings       


PLAYER RECORDS IN LARGE ROSTERS


' match_id  team_id  player_id  batting_order  is_played is_captain is_wk  is_impact  is_impact_off             player_name\n    13613       20       3539          1.000       True      False False          0              0           Ben McDermott\n    13613       20        838          1.000       True      False False          0              0           Glenn Maxwell\n    13613       20        250          2.000       True       True False          0              0             Aaron Finch\n    13613       20       2639          2.000       True      False False          0              0          Marcus Stoinis\n    13613       20       4736          3.000       True      False False          0              0             Josh Inglis\n    13613       20       3539          3.000       True      False False          0              0           Ben McDermott\n    13613       20        838          4.000       True      False False          0              0           Glenn Maxwell\n    136


REPEATED PLAYER IDS WITHIN MATCH-TEAM
Repeated player-ID groups: 35


' match_id  team_id  player_id  records\n    13613       20        223        2\n    13613       20        250        2\n    13613       20        838        2\n    13613       20       2639        2\n    13613       20       3539        2\n    13613       20       4736        2\n    13613       27       1100        2\n    13613       27       2351        2\n    13613       27       2436        2\n    13613       27       4895        2\n    13613       27       4902        2\n    13613       27       6931        2\n    21176       17       2246        2\n    21176       17       3323        2\n    21176       17       3374        2\n    21176       17       3700        2\n    21176       17       4106        2\n    21176       17      25534        2\n    21176       27       1481        2\n    21176       27       2407        2\n    21176       27       3672        2\n    21176       27       4895        2\n    21176       27       6931        2\n    28962       18         47        2\

### Detailed repeated-player registration audit

In [68]:
# Identify all repeated player IDs within a match-team
repeated_keys = repeated_player_ids[
    ["match_id", "team_id", "player_id"]
]

# Retrieve every occurrence of those player records
repeated_player_details = playing_eleven.merge(
    repeated_keys,
    on=["match_id", "team_id", "player_id"],
    how="inner"
).merge(
    player_names,
    on="player_id",
    how="left"
).merge(
    matches[["match_id", "match_date", "home_team", "away_team", "result"]],
    on="match_id",
    how="left"
)

print("ALL REPEATED PLAYER RECORDS")
display(
    repeated_player_details.sort_values(
        ["match_id", "team_id", "player_id", "batting_order"]
    ).to_string(index=False)
)

# Summarize the repeated records by player and match
print("\nREPEATED PLAYER GROUP SUMMARY")
display(
    repeated_player_details.groupby(
        ["match_id", "team_id", "player_id", "player_name"]
    ).agg(
        records=("player_id", "size"),
        batting_orders=("batting_order", lambda x: sorted(x.dropna().unique().tolist())),
        played_values=("is_played", lambda x: sorted(x.astype(str).unique().tolist())),
        impact_on=("is_impact", "sum"),
        impact_off=("is_impact_off", "sum"),
        captain_flags=("is_captain", lambda x: sorted(x.dropna().astype(str).unique().tolist())),
        wk_flags=("is_wk", lambda x: sorted(x.dropna().astype(str).unique().tolist()))
    ).reset_index().to_string(index=False)
)

# Check whether the repeated rows are exact duplicates across all original XI columns
original_xi_columns = playing_eleven.columns.tolist()

print("\nREPEATED ROWS — UNIQUE ORIGINAL XI RECORDS PER PLAYER-MATCH")
display(
    repeated_player_details.groupby(
        ["match_id", "team_id", "player_id"]
    )[original_xi_columns]
    .apply(lambda x: x.drop_duplicates().shape[0])
    .reset_index(name="distinct_original_records")
    .to_string(index=False)
)

# Affected match-level summary
print("\nAFFECTED MATCHES")
display(
    repeated_player_details[
        ["match_id", "match_date", "home_team", "away_team"]
    ].drop_duplicates()
    .sort_values("match_id")
    .to_string(index=False)
)

ALL REPEATED PLAYER RECORDS


' match_id  team_id  player_id  batting_order  is_played is_captain is_wk  is_impact  is_impact_off          player_name match_date   home_team   away_team                                result\n    13613       20        223          5.000       True      False False          0              0         Steven Smith 2022-02-13   Australia   Sri Lanka Australia Won the one over Eliminator\n    13613       20        223          6.000       True      False False          0              0         Steven Smith 2022-02-13   Australia   Sri Lanka Australia Won the one over Eliminator\n    13613       20        250          2.000       True       True False          0              0          Aaron Finch 2022-02-13   Australia   Sri Lanka Australia Won the one over Eliminator\n    13613       20        250          4.000       True       True False          0              0          Aaron Finch 2022-02-13   Australia   Sri Lanka Australia Won the one over Eliminator\n    13613       20        838


REPEATED PLAYER GROUP SUMMARY


' match_id  team_id  player_id          player_name  records  batting_orders played_values  impact_on  impact_off captain_flags wk_flags\n    13613       20        223         Steven Smith        2      [5.0, 6.0]        [True]          0           0       [False]  [False]\n    13613       20        250          Aaron Finch        2      [2.0, 4.0]        [True]          0           0        [True]  [False]\n    13613       20        838        Glenn Maxwell        2      [1.0, 4.0]        [True]          0           0       [False]  [False]\n    13613       20       2639       Marcus Stoinis        2      [2.0, 6.0]        [True]          0           0       [False]  [False]\n    13613       20       3539        Ben McDermott        2      [1.0, 3.0]        [True]          0           0       [False]  [False]\n    13613       20       4736          Josh Inglis        2      [3.0, 5.0]        [True]          0           0       [False]  [False]\n    13613       27       1100     Dinesh


REPEATED ROWS — UNIQUE ORIGINAL XI RECORDS PER PLAYER-MATCH


' match_id  team_id  player_id  distinct_original_records\n    13613       20        223                          2\n    13613       20        250                          2\n    13613       20        838                          2\n    13613       20       2639                          2\n    13613       20       3539                          2\n    13613       20       4736                          2\n    13613       27       1100                          2\n    13613       27       2351                          2\n    13613       27       2436                          2\n    13613       27       4895                          2\n    13613       27       4902                          2\n    13613       27       6931                          2\n    21176       17       2246                          2\n    21176       17       3323                          2\n    21176       17       3374                          2\n    21176       17       3700                          2\n    21176    


AFFECTED MATCHES


' match_id match_date   home_team   away_team\n    13613 2022-02-13   Australia   Sri Lanka\n    21176 2023-04-02 New Zealand   Sri Lanka\n    28962 2024-01-17       India Afghanistan'

### Verify repeated batting orders against match innings

In [69]:
affected_matches = [13613, 21176, 28962]

# 1. Match metadata and innings-level delivery counts
print("MATCH RESULTS AND INNINGS COUNTS")
display(
    matches[matches["match_id"].isin(affected_matches)][
        ["match_id", "match_date", "home_team", "away_team", "result"]
    ].to_string(index=False)
)

innings_summary = (
    ball_by_ball[ball_by_ball["match_id"].isin(affected_matches)]
    .groupby(["match_id", "innings"])
    .agg(
        delivery_records=("innings_ball_num", "size"),
        distinct_strikers=("striker_id", "nunique"),
        distinct_non_strikers=("non_striker_id", "nunique"),
        distinct_bowlers=("bowler_id", "nunique")
    )
    .reset_index()
)

print("\nDELIVERY COUNTS BY INNINGS")
display(innings_summary.to_string(index=False))

# 2. Show repeated players' delivery involvement by innings and role
repeated_keys = repeated_player_ids[
    repeated_player_ids["match_id"].isin(affected_matches)
][["match_id", "team_id", "player_id"]]

player_roles = []

for role_column, role_name in [
    ("striker_id", "striker"),
    ("non_striker_id", "non_striker"),
    ("bowler_id", "bowler")
]:
    temp = (
        ball_by_ball[
            ball_by_ball["match_id"].isin(affected_matches)
        ]
        .groupby(["match_id", "innings", role_column])
        .size()
        .reset_index(name="delivery_count")
        .rename(columns={role_column: "player_id"})
    )
    temp["role"] = role_name
    player_roles.append(temp)

role_detail = pd.concat(player_roles, ignore_index=True)

role_detail = role_detail.merge(
    repeated_keys, on=["match_id", "player_id"], how="inner"
).merge(
    player_names, on="player_id", how="left"
)

print("\nREPEATED PLAYERS' DELIVERY INVOLVEMENT BY INNINGS")
display(
    role_detail.sort_values(
        ["match_id", "player_id", "innings", "role"]
    ).to_string(index=False)
)

# 3. Compare XI batting positions for each repeated player
print("\nREPEATED PLAYERS AND THEIR RECORDED BATTING POSITIONS")
display(
    repeated_player_details[
        repeated_player_details["match_id"].isin(affected_matches)
    ][
        ["match_id", "team_id", "player_id", "player_name",
         "batting_order", "is_played"]
    ].sort_values(
        ["match_id", "team_id", "player_id", "batting_order"]
    ).to_string(index=False)
)

MATCH RESULTS AND INNINGS COUNTS


' match_id match_date   home_team   away_team                                result\n    13613 2022-02-13   Australia   Sri Lanka Australia Won the one over Eliminator\n    21176 2023-04-02 New Zealand   Sri Lanka          Sri Lanka won the super over\n    28962 2024-01-17       India Afghanistan       India Won Super Over By 10 Runs'


DELIVERY COUNTS BY INNINGS


' match_id  innings  delivery_records  distinct_strikers  distinct_non_strikers  distinct_bowlers\n    13613        1               124                  8                      8                 5\n    13613        2               122                 10                      9                 6\n    21176        1               129                  7                      6                 5\n    21176        2               123                 10                     10                 6\n    28962        1               127                  6                      5                 6\n    28962        2               129                  8                      7                 6'


REPEATED PLAYERS' DELIVERY INVOLVEMENT BY INNINGS


' match_id  innings  player_id  delivery_count        role  team_id          player_name\n    13613        1        223              27 non_striker       20         Steven Smith\n    13613        1        223              16     striker       20         Steven Smith\n    13613        1        250              27 non_striker       20          Aaron Finch\n    13613        1        250              20     striker       20          Aaron Finch\n    13613        1        838              13 non_striker       20        Glenn Maxwell\n    13613        1        838              12     striker       20        Glenn Maxwell\n    13613        2        838               6      bowler       20        Glenn Maxwell\n    13613        2       1100              15 non_striker       27     Dinesh Chandimal\n    13613        2       1100              22     striker       27     Dinesh Chandimal\n    13613        2       2351               4 non_striker       27 Danushka Gunathilaka\n    13613        2  


REPEATED PLAYERS AND THEIR RECORDED BATTING POSITIONS


' match_id  team_id  player_id          player_name  batting_order  is_played\n    13613       20        223         Steven Smith          5.000       True\n    13613       20        223         Steven Smith          6.000       True\n    13613       20        250          Aaron Finch          2.000       True\n    13613       20        250          Aaron Finch          4.000       True\n    13613       20        838        Glenn Maxwell          1.000       True\n    13613       20        838        Glenn Maxwell          4.000       True\n    13613       20       2639       Marcus Stoinis          2.000       True\n    13613       20       2639       Marcus Stoinis          6.000       True\n    13613       20       3539        Ben McDermott          1.000       True\n    13613       20       3539        Ben McDermott          3.000       True\n    13613       20       4736          Josh Inglis          3.000       True\n    13613       20       4736          Josh Inglis          5.0

### Reconcile XI batting order with delivery-level batting order

In [70]:
affected_matches = [13613, 21176, 28962]

# Get observed batting order from deliveries where the player was striker
delivery_batting_order = (
    ball_by_ball[
        ball_by_ball["match_id"].isin(affected_matches)
    ]
    .groupby(["match_id", "batting_team_id", "striker_id"])
    .agg(
        delivery_batting_orders=("batting_order",
                                 lambda x: sorted(x.dropna().unique().tolist())),
        striker_delivery_rows=("innings_ball_num", "size")
    )
    .reset_index()
    .rename(columns={
        "batting_team_id": "team_id",
        "striker_id": "player_id"
    })
)

# Attach XI records for the repeated players
xi_order_comparison = repeated_player_details[
    repeated_player_details["match_id"].isin(affected_matches)
][[
    "match_id", "team_id", "player_id", "player_name",
    "batting_order", "is_played"
]].merge(
    delivery_batting_order,
    on=["match_id", "team_id", "player_id"],
    how="left"
)

print("XI BATTING ORDER VS DELIVERY-LEVEL BATTING ORDER")
display(
    xi_order_comparison.sort_values(
        ["match_id", "team_id", "player_id", "batting_order"]
    ).to_string(index=False)
)

# Summarize whether delivery batting order agrees with each XI row
xi_order_comparison["xi_order_matches_delivery"] = (
    xi_order_comparison.apply(
        lambda row: (
            pd.notna(row["batting_order"]) and
            pd.notna(row["delivery_batting_orders"]) and
            row["batting_order"] in row["delivery_batting_orders"]
        ),
        axis=1
    )
)

print("\nAGREEMENT SUMMARY")
display(
    xi_order_comparison.groupby(
        ["xi_order_matches_delivery"]
    ).size().reset_index(name="records").to_string(index=False)
)

XI BATTING ORDER VS DELIVERY-LEVEL BATTING ORDER


' match_id  team_id  player_id          player_name  batting_order  is_played delivery_batting_orders  striker_delivery_rows\n    13613       20        223         Steven Smith          5.000       True                     [5]                     16\n    13613       20        223         Steven Smith          6.000       True                     [5]                     16\n    13613       20        250          Aaron Finch          2.000       True                     [2]                     20\n    13613       20        250          Aaron Finch          4.000       True                     [2]                     20\n    13613       20        838        Glenn Maxwell          1.000       True                     [4]                     12\n    13613       20        838        Glenn Maxwell          4.000       True                     [4]                     12\n    13613       20       2639       Marcus Stoinis          2.000       True                     [6]                     20\


AGREEMENT SUMMARY


' xi_order_matches_delivery  records\n                     False       40\n                      True       35'

### Player-level batting-order reconciliation

In [71]:
# Collapse repeated XI records to one row per match-team-player
xi_player_orders = (
    repeated_player_details[
        repeated_player_details["match_id"].isin(affected_matches)
    ]
    .groupby(
        ["match_id", "team_id", "player_id", "player_name"]
    )
    .agg(
        xi_batting_orders=(
            "batting_order",
            lambda x: sorted(x.dropna().unique().tolist())
        ),
        xi_record_count=("player_id", "size")
    )
    .reset_index()
)

# Collapse delivery observations to one row per match-team-player
delivery_player_orders = (
    ball_by_ball[
        ball_by_ball["match_id"].isin(affected_matches)
    ]
    .groupby(["match_id", "batting_team_id", "striker_id"])
    .agg(
        delivery_batting_orders=(
            "batting_order",
            lambda x: sorted(x.dropna().unique().tolist())
        ),
        striker_delivery_rows=("innings_ball_num", "size")
    )
    .reset_index()
    .rename(columns={
        "batting_team_id": "team_id",
        "striker_id": "player_id"
    })
)

# Join and compare
player_order_audit = xi_player_orders.merge(
    delivery_player_orders,
    on=["match_id", "team_id", "player_id"],
    how="left"
)

player_order_audit["delivery_order_in_xi_orders"] = (
    player_order_audit.apply(
        lambda row: (
            len(row["delivery_batting_orders"]) > 0
            and any(
                order in row["xi_batting_orders"]
                for order in row["delivery_batting_orders"]
            )
        ),
        axis=1
    )
)

print("UNIQUE PLAYER-LEVEL ORDER COMPARISON")
display(
    player_order_audit.sort_values(
        ["match_id", "team_id", "player_id"]
    ).to_string(index=False)
)

print("\nPLAYER-LEVEL AGREEMENT SUMMARY")
display(
    player_order_audit.groupby(
        "delivery_order_in_xi_orders"
    ).size().reset_index(name="unique_players").to_string(index=False)
)

print("\nPLAYERS WITH NO STRIKER DELIVERY ORDER")
display(
    player_order_audit[
        player_order_audit["delivery_batting_orders"].isna()
    ].to_string(index=False)
)

UNIQUE PLAYER-LEVEL ORDER COMPARISON


' match_id  team_id  player_id          player_name xi_batting_orders  xi_record_count delivery_batting_orders  striker_delivery_rows  delivery_order_in_xi_orders\n    13613       20        223         Steven Smith        [5.0, 6.0]                2                     [5]                     16                         True\n    13613       20        250          Aaron Finch        [2.0, 4.0]                2                     [2]                     20                         True\n    13613       20        838        Glenn Maxwell        [1.0, 4.0]                2                     [4]                     12                         True\n    13613       20       2639       Marcus Stoinis        [2.0, 6.0]                2                     [6]                     20                         True\n    13613       20       3539        Ben McDermott        [1.0, 3.0]                2                     [1]                     15                         True\n    13613       20   


PLAYER-LEVEL AGREEMENT SUMMARY


' delivery_order_in_xi_orders  unique_players\n                        True              35'


PLAYERS WITH NO STRIKER DELIVERY ORDER


'Empty DataFrame\nColumns: [match_id, team_id, player_id, player_name, xi_batting_orders, xi_record_count, delivery_batting_orders, striker_delivery_rows, delivery_order_in_xi_orders]\nIndex: []'

### Determine the actual grain of playing_eleven

In [72]:
# 1. Distribution of played players per match-team
print("PLAYED-PLAYER COUNT PER MATCH-TEAM")
display(
    roster_summary.groupby("played_records")
    .size()
    .reset_index(name="match_team_groups")
    .sort_values("played_records")
    .to_string(index=False)
)

# 2. Compare impact flags with played-player counts
roster_summary["has_impact_flags"] = (
    (roster_summary["impact_on"] > 0) |
    (roster_summary["impact_off"] > 0)
)

print("\nPLAYED COUNTS BY IMPACT-FLAG PRESENCE")
display(
    roster_summary.groupby(
        ["has_impact_flags", "played_records"]
    ).size().reset_index(name="match_team_groups")
    .sort_values(["has_impact_flags", "played_records"])
    .to_string(index=False)
)

# 3. Check whether non-playing records have batting order or role flags
not_played = playing_eleven[playing_eleven["is_played"] == False]

print("\nNON-PLAYING RECORD FIELD COMPLETENESS")
display(
    pd.DataFrame({
        "check": [
            "Total non-playing records",
            "Non-playing with batting order",
            "Non-playing with captain flag present",
            "Non-playing with wicketkeeper flag present",
            "Non-playing with impact flag",
            "Non-playing with impact-off flag"
        ],
        "count": [
            len(not_played),
            not_played["batting_order"].notna().sum(),
            not_played["is_captain"].notna().sum(),
            not_played["is_wk"].notna().sum(),
            (not_played["is_impact"] == 1).sum(),
            (not_played["is_impact_off"] == 1).sum()
        ]
    }).to_string(index=False)
)

# 4. Inspect the largest match-team groups compactly
print("\nLARGEST MATCH-TEAM GROUPS")
largest = roster_summary.nlargest(10, "total_records").merge(
    matches[["match_id", "match_date", "home_team", "away_team"]],
    on="match_id", how="left"
)

display(
    largest[[
        "match_id", "match_date", "home_team", "away_team",
        "team_id", "total_records", "played_records",
        "not_played_records", "impact_on", "impact_off"
    ]].to_string(index=False)
)

# 5. Count distinct player IDs against row count
print("\nROW COUNT VS DISTINCT PLAYER COUNT")
display(
    pd.DataFrame({
        "metric": [
            "Total playing_eleven rows",
            "Distinct match-team-player combinations",
            "Repeated player-match-team combinations",
            "Total records in repeated combinations"
        ],
        "value": [
            len(playing_eleven),
            playing_eleven[["match_id", "team_id", "player_id"]].drop_duplicates().shape[0],
            len(repeated_player_ids),
            repeated_player_details.shape[0]
        ]
    }).to_string(index=False)
)

PLAYED-PLAYER COUNT PER MATCH-TEAM


' played_records  match_team_groups\n              0                 23\n             10                  1\n             11               1168\n             12                556\n             13                  6\n             16                  1\n             17                  3\n             19                  1\n             20                  1'


PLAYED COUNTS BY IMPACT-FLAG PRESENCE


' has_impact_flags  played_records  match_team_groups\n            False               0                 23\n            False              10                  1\n            False              11               1168\n            False              12                  4\n            False              16                  1\n            False              17                  3\n            False              19                  1\n            False              20                  1\n             True              12                552\n             True              13                  6'


NON-PLAYING RECORD FIELD COMPLETENESS


'                                     check  count\n                 Total non-playing records   8667\n            Non-playing with batting order      0\n     Non-playing with captain flag present      0\nNon-playing with wicketkeeper flag present      0\n              Non-playing with impact flag      0\n          Non-playing with impact-off flag      0'


LARGEST MATCH-TEAM GROUPS


' match_id match_date           home_team                   away_team  team_id  total_records  played_records  not_played_records  impact_on  impact_off\n    40756 2025-05-20 Chennai Super Kings            Rajasthan Royals        1             27              12                  15          1           1\n    31243 2024-04-11      Mumbai Indians Royal Challengers Bengaluru        2             26              12                  14          1           1\n    40754 2025-05-26        Punjab Kings              Mumbai Indians        2             26              12                  14          1           1\n    40755 2025-05-18      Delhi Capitals              Gujarat Titans     3508             26              12                  14          1           1\n    40758 2025-05-22      Gujarat Titans        Lucknow Super Giants     3508             26              12                  14          1           1\n    40759 2025-05-21      Mumbai Indians              Delhi Capitals        2    


ROW COUNT VS DISTINCT PLAYER COUNT


'                                 metric  value\n              Total playing_eleven rows  28381\nDistinct match-team-player combinations  28341\nRepeated player-match-team combinations     35\n Total records in repeated combinations     75'

### Investigate unusually high played-player counts

In [73]:
# 1. Identify match-team groups with more than 13 played records
high_played_groups = roster_summary[
    roster_summary["played_records"] > 13
].copy()

high_played_groups = high_played_groups.merge(
    matches[[
        "match_id", "match_date", "home_team", "away_team", "result"
    ]],
    on="match_id", how="left"
)

print("MATCH-TEAM GROUPS WITH MORE THAN 13 PLAYED RECORDS")
display(
    high_played_groups.sort_values(
        ["played_records", "match_id"],
        ascending=[False, True]
    ).to_string(index=False)
)

# 2. Show all player records in those groups
high_played_details = playing_eleven.merge(
    high_played_groups[["match_id", "team_id"]],
    on=["match_id", "team_id"],
    how="inner"
).merge(
    player_names, on="player_id", how="left"
)

print("\nPLAYER RECORDS IN HIGH-PLAYED GROUPS")
display(
    high_played_details.sort_values(
        ["match_id", "team_id", "is_played", "batting_order"],
        ascending=[True, True, False, True]
    ).to_string(index=False)
)

# 3. Count distinct delivery participants by match-team
delivery_participants = []

for role_column, role_name in [
    ("striker_id", "striker"),
    ("non_striker_id", "non_striker"),
    ("bowler_id", "bowler")
]:
    temp = (
        ball_by_ball[
            ball_by_ball["match_id"].isin(
                high_played_groups["match_id"].unique()
            )
        ][["match_id", "batting_team_id", "bowling_team_id", role_column]]
        .copy()
    )

    temp["team_id"] = temp.apply(
        lambda row: (
            row["batting_team_id"]
            if role_name in ["striker", "non_striker"]
            else row["bowling_team_id"]
        ),
        axis=1
    )
    temp = temp[["match_id", "team_id", role_column]].rename(
        columns={role_column: "player_id"}
    )
    temp["role"] = role_name
    delivery_participants.append(temp)

delivery_participants = pd.concat(
    delivery_participants, ignore_index=True
).drop_duplicates(["match_id", "team_id", "player_id", "role"])

delivery_player_summary = (
    delivery_participants.groupby(["match_id", "team_id"])
    .agg(
        distinct_delivery_players=("player_id", "nunique"),
        striker_players=("role", lambda x: (x == "striker").sum()),
        non_striker_players=("role", lambda x: (x == "non_striker").sum()),
        bowler_players=("role", lambda x: (x == "bowler").sum())
    )
    .reset_index()
)

# A player may appear in multiple roles, so calculate the union separately
delivery_player_union = (
    delivery_participants.groupby(["match_id", "team_id"])["player_id"]
    .nunique()
    .rename("unique_players_in_deliveries")
    .reset_index()
)

high_played_comparison = high_played_groups.merge(
    delivery_player_union,
    on=["match_id", "team_id"],
    how="left"
)

print("\nPLAYED XI COUNT VS DISTINCT DELIVERY PARTICIPANTS")
display(
    high_played_comparison[[
        "match_id", "team_id", "match_date",
        "home_team", "away_team", "result",
        "played_records", "total_records",
        "unique_players_in_deliveries"
    ]].to_string(index=False)
)

# 4. Check the 23 groups with zero played records
zero_played_groups = roster_summary[
    roster_summary["played_records"] == 0
].merge(
    matches[[
        "match_id", "match_date", "home_team", "away_team", "result"
    ]],
    on="match_id", how="left"
)

print("\nMATCH-TEAM GROUPS WITH ZERO PLAYED RECORDS")
display(
    zero_played_groups.to_string(index=False)
)

MATCH-TEAM GROUPS WITH MORE THAN 13 PLAYED RECORDS


' match_id  team_id  total_records  played_records  not_played_records  distinct_players  batting_positions_present  impact_on  impact_off  has_impact_flags match_date   home_team   away_team                                result\n    28962       78             20              20                   0                11                         20          0           0             False 2024-01-17       India Afghanistan       India Won Super Over By 10 Runs\n    28962       18             19              19                   0                11                         19          0           0             False 2024-01-17       India Afghanistan       India Won Super Over By 10 Runs\n    13613       20             17              17                   0                11                         17          0           0             False 2022-02-13   Australia   Sri Lanka Australia Won the one over Eliminator\n    13613       27             17              17                   0          


PLAYER RECORDS IN HIGH-PLAYED GROUPS


' match_id  team_id  player_id  batting_order  is_played is_captain is_wk  is_impact  is_impact_off          player_name\n    13613       20       3539          1.000       True      False False          0              0        Ben McDermott\n    13613       20        838          1.000       True      False False          0              0        Glenn Maxwell\n    13613       20        250          2.000       True       True False          0              0          Aaron Finch\n    13613       20       2639          2.000       True      False False          0              0       Marcus Stoinis\n    13613       20       4736          3.000       True      False False          0              0          Josh Inglis\n    13613       20       3539          3.000       True      False False          0              0        Ben McDermott\n    13613       20        838          4.000       True      False False          0              0        Glenn Maxwell\n    13613       20        250  


PLAYED XI COUNT VS DISTINCT DELIVERY PARTICIPANTS


' match_id  team_id match_date   home_team   away_team                                result  played_records  total_records  unique_players_in_deliveries\n    13613       20 2022-02-13   Australia   Sri Lanka Australia Won the one over Eliminator              17             17                            11\n    13613       27 2022-02-13   Australia   Sri Lanka Australia Won the one over Eliminator              17             17                            11\n    21176       17 2023-04-02 New Zealand   Sri Lanka          Sri Lanka won the super over              17             17                            11\n    21176       27 2023-04-02 New Zealand   Sri Lanka          Sri Lanka won the super over              16             16                            11\n    28962       18 2024-01-17       India Afghanistan       India Won Super Over By 10 Runs              19             19                            11\n    28962       78 2024-01-17       India Afghanistan       India Won Super


MATCH-TEAM GROUPS WITH ZERO PLAYED RECORDS


' match_id  team_id  total_records  played_records  not_played_records  distinct_players  batting_positions_present  impact_on  impact_off  has_impact_flags match_date                   home_team             away_team          result\n    31281        8             23               0                  23                23                          0          0           0             False 2024-05-13              Gujarat Titans Kolkata Knight Riders Match Abandoned\n    31281     3508             25               0                  25                25                          0          0           0             False 2024-05-13              Gujarat Titans Kolkata Knight Riders Match Abandoned\n    31284      242             25               0                  25                25                          0          0           0             False 2024-05-16         Sunrisers Hyderabad        Gujarat Titans Match Abandoned\n    31284     3508             25               0              

### Exact XI-to-delivery player-set reconciliation

In [74]:
affected_groups = high_played_groups[["match_id", "team_id"]].drop_duplicates()

results = []

for _, group in affected_groups.iterrows():
    match_id = group["match_id"]
    team_id = group["team_id"]

    xi_ids = set(
        playing_eleven.loc[
            (playing_eleven["match_id"] == match_id) &
            (playing_eleven["team_id"] == team_id),
            "player_id"
        ].dropna().astype(int)
    )

    deliveries = ball_by_ball[
        ball_by_ball["match_id"] == match_id
    ]

    delivery_ids = set()

    for _, row in deliveries.iterrows():
        if row["batting_team_id"] == team_id:
            delivery_ids.add(int(row["striker_id"]))
            delivery_ids.add(int(row["non_striker_id"]))

        if row["bowling_team_id"] == team_id:
            delivery_ids.add(int(row["bowler_id"]))

    results.append({
        "match_id": match_id,
        "team_id": team_id,
        "xi_unique_players": len(xi_ids),
        "delivery_unique_players": len(delivery_ids),
        "xi_not_in_deliveries": sorted(xi_ids - delivery_ids),
        "delivery_not_in_xi": sorted(delivery_ids - xi_ids),
        "exact_match": xi_ids == delivery_ids
    })

player_set_audit = pd.DataFrame(results)

print("EXACT PLAYER-SET RECONCILIATION")
display(player_set_audit.to_string(index=False))

print("\nOVERALL RESULT")
print("Exact player-set matches:", player_set_audit["exact_match"].sum())
print("Total affected match-team groups:", len(player_set_audit))

EXACT PLAYER-SET RECONCILIATION


' match_id  team_id  xi_unique_players  delivery_unique_players xi_not_in_deliveries delivery_not_in_xi  exact_match\n    13613       20                 11                       11                   []                 []         True\n    13613       27                 11                       11                   []                 []         True\n    21176       17                 11                       11                   []                 []         True\n    21176       27                 11                       11                   []                 []         True\n    28962       18                 11                       11                   []                 []         True\n    28962       78                 11                       11                   []                 []         True'


OVERALL RESULT
Exact player-set matches: 6
Total affected match-team groups: 6


### Audit match-team groups with zero played records

In [75]:
zero_groups = roster_summary[
    roster_summary["played_records"] == 0
].copy()

zero_detail = zero_groups.merge(
    matches[[
        "match_id", "match_date", "comp_id",
        "home_team", "away_team", "result"
    ]],
    on="match_id", how="left"
)

# Count delivery records for each match
delivery_counts = (
    ball_by_ball.groupby("match_id")
    .size()
    .rename("delivery_records")
    .reset_index()
)

zero_detail = zero_detail.merge(
    delivery_counts, on="match_id", how="left"
)

zero_detail["delivery_records"] = (
    zero_detail["delivery_records"].fillna(0).astype(int)
)

print("ALL ZERO-PLAYED MATCH-TEAM GROUPS")
display(
    zero_detail.sort_values(
        ["match_id", "team_id"]
    ).to_string(index=False)
)

print("\nZERO-PLAYED GROUPS BY MATCH RESULT")
display(
    zero_detail.groupby("result")
    .agg(
        match_team_groups=("team_id", "size"),
        distinct_matches=("match_id", "nunique"),
        total_registered_players=("total_records", "sum"),
        matches_with_deliveries=("delivery_records", lambda x: (x > 0).sum())
    )
    .reset_index()
    .to_string(index=False)
)

print("\nZERO-PLAYED GROUPS WITH DELIVERY DATA")
display(
    zero_detail[zero_detail["delivery_records"] > 0]
    .to_string(index=False)
)

print("\nZERO-PLAYED GROUPS WITH NON-ABANDONED RESULTS")
display(
    zero_detail[
        ~zero_detail["result"].str.contains(
            "abandoned|no result", case=False, na=False
        )
    ].to_string(index=False)
)

ALL ZERO-PLAYED MATCH-TEAM GROUPS


' match_id  team_id  total_records  played_records  not_played_records  distinct_players  batting_positions_present  impact_on  impact_off  has_impact_flags match_date  comp_id                   home_team             away_team          result  delivery_records\n    31281        8             23               0                  23                23                          0          0           0             False 2024-05-13     2193              Gujarat Titans Kolkata Knight Riders Match Abandoned                 0\n    31281     3508             25               0                  25                25                          0          0           0             False 2024-05-13     2193              Gujarat Titans Kolkata Knight Riders Match Abandoned                 0\n    31284      242             25               0                  25                25                          0          0           0             False 2024-05-16     2193         Sunrisers Hyderabad        Gujar


ZERO-PLAYED GROUPS BY MATCH RESULT


'         result  match_team_groups  distinct_matches  total_registered_players  matches_with_deliveries\nMatch Abandoned                 23                12                       419                        0'


ZERO-PLAYED GROUPS WITH DELIVERY DATA


'Empty DataFrame\nColumns: [match_id, team_id, total_records, played_records, not_played_records, distinct_players, batting_positions_present, impact_on, impact_off, has_impact_flags, match_date, comp_id, home_team, away_team, result, delivery_records]\nIndex: []'


ZERO-PLAYED GROUPS WITH NON-ABANDONED RESULTS


'Empty DataFrame\nColumns: [match_id, team_id, total_records, played_records, not_played_records, distinct_players, batting_positions_present, impact_on, impact_off, has_impact_flags, match_date, comp_id, home_team, away_team, result, delivery_records]\nIndex: []'

# Competition and season coverage

### Competition and season coverage audit

In [76]:
# 1. Match counts by competition
match_coverage = (
    matches.groupby("comp_id")
    .agg(
        match_records=("match_id", "nunique"),
        first_match_date=("match_date", "min"),
        last_match_date=("match_date", "max")
    )
    .reset_index()
)

# 2. Ball-by-ball coverage by competition
match_comp = matches[["match_id", "comp_id"]].drop_duplicates()

delivery_match_counts = (
    ball_by_ball[["match_id"]].drop_duplicates()
    .merge(match_comp, on="match_id", how="left")
    .groupby("comp_id")
    .agg(delivery_matches=("match_id", "nunique"))
    .reset_index()
)

# 3. Playing-XI coverage by competition
xi_match_counts = (
    playing_eleven[["match_id"]].drop_duplicates()
    .merge(match_comp, on="match_id", how="left")
    .groupby("comp_id")
    .agg(xi_matches=("match_id", "nunique"))
    .reset_index()
)

# 4. Mapping coverage by competition
mapping_coverage = (
    competition_team_player_mapping.groupby("comp_id")
    .agg(
        mapping_rows=("player_id", "size"),
        mapped_players=("player_id", "nunique"),
        mapped_teams=("team_id", "nunique")
    )
    .reset_index()
)

# 5. Combine all coverage information
competition_coverage = (
    competitions.merge(match_coverage, on="comp_id", how="left")
    .merge(delivery_match_counts, on="comp_id", how="left")
    .merge(xi_match_counts, on="comp_id", how="left")
    .merge(mapping_coverage, on="comp_id", how="left")
)

coverage_columns = [
    "match_records", "delivery_matches", "xi_matches",
    "mapping_rows", "mapped_players", "mapped_teams"
]

competition_coverage[coverage_columns] = (
    competition_coverage[coverage_columns].fillna(0).astype(int)
)

print("COMPETITION COVERAGE — ALL COMPETITIONS")
display(
    competition_coverage.sort_values(
        ["match_format", "season", "comp_name"]
    ).to_string(index=False)
)

# 6. Summarize by format and season
print("\nCOVERAGE SUMMARY BY FORMAT AND SEASON")
display(
    competition_coverage.groupby(
        ["match_format", "season"]
    ).agg(
        competitions=("comp_id", "nunique"),
        competitions_with_matches=(
            "match_records", lambda x: (x > 0).sum()
        ),
        total_matches=("match_records", "sum"),
        matches_with_deliveries=("delivery_matches", "sum"),
        matches_with_xi=("xi_matches", "sum")
    ).reset_index().sort_values(
        ["match_format", "season"]
    ).to_string(index=False)
)

# 7. Competitions with no match records
print("\nCOMPETITIONS WITHOUT MATCH RECORDS")
display(
    competition_coverage[
        competition_coverage["match_records"] == 0
    ][["comp_id", "comp_name", "match_format", "season"]]
    .to_string(index=False)
)

COMPETITION COVERAGE — ALL COMPETITIONS


' comp_id                                               comp_name match_format  season  match_records first_match_date last_match_date  delivery_matches  xi_matches  mapping_rows  mapped_players  mapped_teams\n    1358                                                IPL 2022          T20    2022             74       2022-03-26      2022-05-29                74          74           251             251            10\n    1715                                                IPL 2023          T20    2023             74       2023-03-31      2023-05-29                74          74           264             264            10\n    2193                                                IPL 2024          T20    2024             74       2024-03-22      2024-05-26                71          74           263             263            10\n    2670                   Mens T20 Emerging Teams Asia Cup 2024          T20    2024             15       2024-10-18      2024-10-27                15          15


COVERAGE SUMMARY BY FORMAT AND SEASON


'match_format  season  competitions  competitions_with_matches  total_matches  matches_with_deliveries  matches_with_xi\n         T20    2022             1                          1             74                       74               74\n         T20    2023             1                          1             74                       74               74\n         T20    2024             2                          2             89                       86               89\n         T20    2025             2                          2             90                       89               90\n         T20    2026             1                          1             74                       74               74\n        T20I    2022            24                         24             94                       94               94\n        T20I 2022/23             3                          3             12                       12               12\n        T20I    2023            19     


COMPETITIONS WITHOUT MATCH RECORDS


' comp_id                                      comp_name match_format season\n    3405       Afghanistan v Sri Lanka T20I Series 2026         T20I   2026\n    3473 United Arab Emirates in Nepal T20I Series 2026         T20I   2026\n    2940           India in Bangladesh T20I Series 2025         T20I   2025'

### Exact IPL and T20I coverage

In [78]:

# SECTION 8.2 — Exact IPL and T20I Coverage (Fixed)

# 1. Identify IPL and T20I competitions
competition_coverage["scope"] = "OTHER"

competition_coverage.loc[
    competition_coverage["comp_name"].str.contains(
        r"\bIPL\b", case=False, na=False
    ),
    "scope"
] = "IPL"

competition_coverage.loc[
    competition_coverage["match_format"].str.upper() == "T20I",
    "scope"
] = "T20I"

# 2. Select target competitions
target_competitions = competition_coverage[
    competition_coverage["scope"].isin(["IPL", "T20I"])
].copy()

# 3. Competition-level coverage
print("TARGET COMPETITIONS — IPL AND T20I")

display(
    target_competitions[
        [
            "comp_id", "comp_name", "scope", "season",
            "match_records", "delivery_matches",
            "xi_matches", "mapping_rows"
        ]
    ].sort_values(
        ["scope", "season", "comp_name"]
    ).to_string(index=False)
)

# 4. Season-level coverage summary
print("\nIPL AND T20I SEASON SUMMARY")

season_summary = (
    target_competitions.groupby(["scope", "season"])
    .agg(
        competitions=("comp_id", "nunique"),
        matches=("match_records", "sum"),
        matches_with_deliveries=("delivery_matches", "sum"),
        matches_with_xi=("xi_matches", "sum"),
        mapping_rows=("mapping_rows", "sum")
    )
    .reset_index()
    .sort_values(["scope", "season"])
)

display(season_summary.to_string(index=False))

# 5. Identify incomplete competition coverage
print("\nTARGET COMPETITIONS WITH INCOMPLETE MATCH COVERAGE")

incomplete_coverage = target_competitions[
    (
        target_competitions["delivery_matches"] <
        target_competitions["match_records"]
    ) |
    (
        target_competitions["xi_matches"] <
        target_competitions["match_records"]
    )
]

display(
    incomplete_coverage[
        [
            "comp_id", "comp_name", "scope", "season",
            "match_records", "delivery_matches", "xi_matches"
        ]
    ].to_string(index=False)
)

# 6. Parse dates safely in separate columns
target_competitions["first_date_parsed"] = pd.to_datetime(
    target_competitions["first_match_date"],
    errors="coerce"
)

target_competitions["last_date_parsed"] = pd.to_datetime(
    target_competitions["last_match_date"],
    errors="coerce"
)

print("\nSEASON LABELS AND ACTUAL MATCH DATE RANGES")

date_summary = (
    target_competitions.groupby(["scope", "season"])
    .agg(
        first_match=("first_date_parsed", "min"),
        last_match=("last_date_parsed", "max"),
        competitions=("comp_id", "nunique")
    )
    .reset_index()
    .sort_values(["scope", "season"])
)

display(date_summary.to_string(index=False))

TARGET COMPETITIONS — IPL AND T20I


' comp_id                                               comp_name scope  season  match_records  delivery_matches  xi_matches  mapping_rows\n    1358                                                IPL 2022   IPL    2022             74                74          74           251\n    1715                                                IPL 2023   IPL    2023             74                74          74           264\n    2193                                                IPL 2024   IPL    2024             74                71          74           263\n    2725                                                IPL 2025   IPL    2025             75                74          75           255\n    3298                                                IPL 2026   IPL    2026             74                74          74           275\n    1543                 Afghanistan in Ireland T20I Series 2022  T20I    2022              5                 5           5            34\n    1458                Af


IPL AND T20I SEASON SUMMARY


'scope  season  competitions  matches  matches_with_deliveries  matches_with_xi  mapping_rows\n  IPL    2022             1       74                       74               74           251\n  IPL    2023             1       74                       74               74           264\n  IPL    2024             1       74                       71               74           263\n  IPL    2025             1       75                       74               75           255\n  IPL    2026             1       74                       74               74           275\n T20I    2022            24       94                       94               94           988\n T20I 2022/23             3       12                       12               12           117\n T20I    2023            19       63                       63               63           635\n T20I 2023/24             1        3                        3                3            32\n T20I    2024            31      109                      1


TARGET COMPETITIONS WITH INCOMPLETE MATCH COVERAGE


' comp_id                                  comp_name scope season  match_records  delivery_matches  xi_matches\n    2193                                   IPL 2024   IPL   2024             74                71          74\n    2725                                   IPL 2025   IPL   2025             75                74          75\n    2398    Netherlands Tri-Nation T20I Series 2024  T20I   2024              6                 5           6\n    2548      Australia in England T20I Series 2024  T20I   2024              3                 2           3\n    2354       Pakistan in England T20I Series 2024  T20I   2024              4                 2           4\n    2442  Pakistan in South Africa T20I Series 2024  T20I   2024              3                 2           3\n    2873    West Indies in Ireland T20I Series 2025  T20I   2025              3                 1           2\n    2921     South Africa in India T20I Series 2025  T20I   2025              5                 4           5\n


SEASON LABELS AND ACTUAL MATCH DATE RANGES


'scope  season first_match last_match  competitions\n  IPL    2022  2022-03-26 2022-05-29             1\n  IPL    2023  2023-03-31 2023-05-29             1\n  IPL    2024  2024-03-22 2024-05-26             1\n  IPL    2025  2025-03-22 2025-06-03             1\n  IPL    2026  2026-03-28 2026-05-31             1\n T20I    2022  2022-01-22 2022-10-14            24\n T20I 2022/23  2022-09-28 2022-11-22             3\n T20I    2023  2023-01-03 2023-12-31            19\n T20I 2023/24  2023-12-07 2023-12-10             1\n T20I    2024  2024-01-11 2024-12-20            31\n T20I 2024/25  2024-12-28 2025-01-02             1\n T20I    2025  2025-01-22 2025-12-19            31\n T20I    2026  2026-01-07 2026-11-01            24'

### Complete T20I totals and missing-match investigation

In [79]:
# 1. Complete T20I season summary
t20i_coverage = target_competitions[
    target_competitions["scope"] == "T20I"
].copy()

print("COMPLETE T20I SEASON SUMMARY")

display(
    t20i_coverage.groupby("season").agg(
        competitions=("comp_id", "nunique"),
        matches=("match_records", "sum"),
        matches_with_deliveries=("delivery_matches", "sum"),
        matches_with_xi=("xi_matches", "sum"),
        mapping_rows=("mapping_rows", "sum")
    ).reset_index().to_string(index=False)
)

print("\nTOTAL T20I COVERAGE")

print("Competitions:", t20i_coverage["comp_id"].nunique())
print("Match records:", t20i_coverage["match_records"].sum())
print("Matches with deliveries:", t20i_coverage["delivery_matches"].sum())
print("Matches with XI:", t20i_coverage["xi_matches"].sum())

# 2. Identify all target matches without ball-by-ball data
target_comp_ids = target_competitions["comp_id"].unique()

matches_without_deliveries = matches[
    matches["comp_id"].isin(target_comp_ids) &
    ~matches["match_id"].isin(ball_by_ball["match_id"].unique())
].copy()

matches_without_deliveries = matches_without_deliveries.merge(
    competitions[["comp_id", "comp_name", "match_format", "season"]],
    on="comp_id",
    how="left"
)

print("\nALL TARGET MATCHES WITHOUT BALL-BY-BALL DATA")

display(
    matches_without_deliveries[
        [
            "match_id", "comp_id", "comp_name",
            "match_format", "season", "match_date",
            "home_team", "away_team", "result"
        ]
    ].sort_values(["match_format", "season", "match_date"])
    .to_string(index=False)
)

# 3. Classify missing-delivery matches by result
print("\nMISSING-DELIVERY MATCHES BY RESULT")

display(
    matches_without_deliveries.groupby(
        ["match_format", "result"]
    ).size().reset_index(name="matches")
    .to_string(index=False)
)

# 4. Identify target matches without playing-XI records
matches_without_xi = matches[
    matches["comp_id"].isin(target_comp_ids) &
    ~matches["match_id"].isin(playing_eleven["match_id"].unique())
].copy()

print("\nTARGET MATCHES WITHOUT PLAYING-XI RECORDS")

display(
    matches_without_xi[
        [
            "match_id", "comp_id", "match_date",
            "home_team", "away_team", "result"
        ]
    ].sort_values(["comp_id", "match_date"])
    .to_string(index=False)
)


COMPLETE T20I SEASON SUMMARY


' season  competitions  matches  matches_with_deliveries  matches_with_xi  mapping_rows\n   2022            24       94                       94               94           988\n2022/23             3       12                       12               12           117\n   2023            19       63                       63               63           635\n2023/24             1        3                        3                3            32\n   2024            31      109                      104              109          1096\n2024/25             1        3                        3                3            29\n   2025            31      126                      121              124          1154\n   2026            24       72                       70               71           777'


TOTAL T20I COVERAGE
Competitions: 134
Match records: 482
Matches with deliveries: 470
Matches with XI: 479

ALL TARGET MATCHES WITHOUT BALL-BY-BALL DATA


' match_id  comp_id                                  comp_name match_format season match_date                   home_team             away_team          result\n    31281     2193                                   IPL 2024          T20   2024 2024-05-13              Gujarat Titans Kolkata Knight Riders Match Abandoned\n    31284     2193                                   IPL 2024          T20   2024 2024-05-16         Sunrisers Hyderabad        Gujarat Titans Match Abandoned\n    31288     2193                                   IPL 2024          T20   2024 2024-05-19            Rajasthan Royals Kolkata Knight Riders Match Abandoned\n    40761     2725                                   IPL 2025          T20   2025 2025-05-17 Royal Challengers Bengaluru Kolkata Knight Riders Match Abandoned\n    32622     2398    Netherlands Tri-Nation T20I Series 2024         T20I   2024 2024-05-20                     Ireland              Scotland Match Abandoned\n    32610     2354       Pakistan in En


MISSING-DELIVERY MATCHES BY RESULT


'match_format          result  matches\n         T20 Match Abandoned        4\n        T20I Match Abandoned       12'


TARGET MATCHES WITHOUT PLAYING-XI RECORDS


' match_id  comp_id match_date home_team    away_team          result\n    36210     2628 2025-09-14   England South Africa Match Abandoned\n    41503     2873 2025-06-14   Ireland  West Indies Match Abandoned\n    48513     3317 2026-01-10 Sri Lanka     Pakistan Match Abandoned'

### Audit competitions outside the IPL/T20I scope

In [80]:
# 1. Competitions outside the target scope
other_competitions = competition_coverage[
    competition_coverage["scope"] == "OTHER"
].copy()

print("ALL OUT-OF-SCOPE COMPETITIONS")

display(
    other_competitions[
        [
            "comp_id", "comp_name", "match_format", "season",
            "match_records", "delivery_matches",
            "xi_matches", "mapping_rows"
        ]
    ].sort_values(
        ["match_format", "season", "comp_name"]
    ).to_string(index=False)
)

# 2. Separate competitions with and without match records
print("\nOUT-OF-SCOPE COMPETITION SUMMARY")

display(
    other_competitions.groupby("match_format").agg(
        competitions=("comp_id", "nunique"),
        competitions_with_matches=(
            "match_records", lambda x: (x > 0).sum()
        ),
        total_matches=("match_records", "sum"),
        matches_with_deliveries=("delivery_matches", "sum"),
        matches_with_xi=("xi_matches", "sum")
    ).reset_index().to_string(index=False)
)

# 3. Identify actual matches belonging to out-of-scope competitions
other_comp_ids = other_competitions["comp_id"].tolist()

other_matches = matches[
    matches["comp_id"].isin(other_comp_ids)
].merge(
    competitions[["comp_id", "comp_name", "match_format", "season"]],
    on="comp_id",
    how="left"
)

print("\nOUT-OF-SCOPE MATCH RECORDS")

display(
    other_matches[
        [
            "match_id", "comp_id", "comp_name",
            "match_format", "season", "match_date",
            "home_team", "away_team", "result"
        ]
    ].sort_values(["comp_id", "match_date"])
    .to_string(index=False)
)

ALL OUT-OF-SCOPE COMPETITIONS


' comp_id                             comp_name match_format season  match_records  delivery_matches  xi_matches  mapping_rows\n    2670 Mens T20 Emerging Teams Asia Cup 2024          T20   2024             15                15          15           137\n    3289            Asia Cup Rising Stars 2025          T20   2025             15                15          15           140'


OUT-OF-SCOPE COMPETITION SUMMARY


'match_format  competitions  competitions_with_matches  total_matches  matches_with_deliveries  matches_with_xi\n         T20             2                          2             30                       30               30'


OUT-OF-SCOPE MATCH RECORDS


' match_id  comp_id                             comp_name match_format season match_date            home_team            away_team                                result\n    37085     2670 Mens T20 Emerging Teams Asia Cup 2024          T20   2024 2024-10-18         Bangladesh A            Hong Kong         Bangladesh A Won by 5 wickets\n    37086     2670 Mens T20 Emerging Teams Asia Cup 2024          T20   2024 2024-10-18          Sri Lanka A        Afghanistan A          Afghanistan A Won by 11 runs\n    37087     2670 Mens T20 Emerging Teams Asia Cup 2024          T20   2024 2024-10-19 United Arab Emirates                 Oman United Arab Emirates Won by 4 wickets\n    37088     2670 Mens T20 Emerging Teams Asia Cup 2024          T20   2024 2024-10-19              India A           Pakistan A                 India A Won by 7 runs\n    37089     2670 Mens T20 Emerging Teams Asia Cup 2024          T20   2024 2024-10-20          Sri Lanka A            Hong Kong            Sri Lanka A W

### Final competition-scope reconciliation

In [81]:
print("COMPETITION RECONCILIATION")
print("Total competition reference records:", len(competitions))
print("Target IPL/T20I competitions:", len(target_competitions))
print("Out-of-scope competitions:", len(other_competitions))
print(
    "Reconciled total:",
    len(target_competitions) + len(other_competitions)
)

print("\nTARGET COMPETITIONS BY SCOPE")
display(
    target_competitions.groupby("scope").agg(
        competitions=("comp_id", "nunique"),
        match_records=("match_records", "sum"),
        matches_with_deliveries=("delivery_matches", "sum"),
        matches_with_xi=("xi_matches", "sum")
    ).reset_index().to_string(index=False)
)

# Target competitions that exist in the reference table but have no matches
reference_only_target = target_competitions[
    target_competitions["match_records"] == 0
].copy()

print("\nTARGET COMPETITIONS WITH NO MATCH RECORDS")
display(
    reference_only_target[
        ["comp_id", "comp_name", "match_format", "season"]
    ].sort_values(["match_format", "season", "comp_name"])
    .to_string(index=False)
)

# Check for unexpected competition IDs or scope overlaps
print("\nSCOPE VALIDATION")
print(
    "Competition IDs duplicated across target and out-of-scope:",
    len(
        set(target_competitions["comp_id"]) &
        set(other_competitions["comp_id"])
    )
)
print(
    "Competition IDs missing from both classifications:",
    len(
        set(competitions["comp_id"]) -
        set(target_competitions["comp_id"]) -
        set(other_competitions["comp_id"])
    )
)

# Final target match totals
print("\nFINAL TARGET MATCH TOTALS")
print("IPL matches:", int(
    target_competitions.loc[
        target_competitions["scope"] == "IPL", "match_records"
    ].sum()
))
print("T20I matches:", int(
    target_competitions.loc[
        target_competitions["scope"] == "T20I", "match_records"
    ].sum()
))
print("Combined target matches:", int(
    target_competitions["match_records"].sum()
))

COMPETITION RECONCILIATION
Total competition reference records: 141
Target IPL/T20I competitions: 139
Out-of-scope competitions: 2
Reconciled total: 141

TARGET COMPETITIONS BY SCOPE


'scope  competitions  match_records  matches_with_deliveries  matches_with_xi\n  IPL             5            371                      367              371\n T20I           134            482                      470              479'


TARGET COMPETITIONS WITH NO MATCH RECORDS


' comp_id                                      comp_name match_format season\n    2940           India in Bangladesh T20I Series 2025         T20I   2025\n    3405       Afghanistan v Sri Lanka T20I Series 2026         T20I   2026\n    3473 United Arab Emirates in Nepal T20I Series 2026         T20I   2026'


SCOPE VALIDATION
Competition IDs duplicated across target and out-of-scope: 0
Competition IDs missing from both classifications: 0

FINAL TARGET MATCH TOTALS
IPL matches: 371
T20I matches: 482
Combined target matches: 853


# Match-level eligibility audit

### Match-level eligibility audit for IPL and T20I

In [82]:
target_comp_ids = target_competitions["comp_id"].unique()

# Restrict match metadata to target IPL/T20I competitions
target_matches = matches[
    matches["comp_id"].isin(target_comp_ids)
].copy()

# Delivery coverage and innings structure
delivery_summary = (
    ball_by_ball[
        ball_by_ball["match_id"].isin(target_matches["match_id"])
    ]
    .groupby("match_id")
    .agg(
        delivery_rows=("innings_ball_num", "size"),
        innings_count=("innings", "nunique"),
        max_innings=("innings", "max")
    )
    .reset_index()
)

# Playing-XI coverage and participation counts
xi_summary = (
    playing_eleven[
        playing_eleven["match_id"].isin(target_matches["match_id"])
    ]
    .groupby("match_id")
    .agg(
        xi_rows=("player_id", "size"),
        distinct_xi_players=("player_id", "nunique"),
        played_players=("is_played", lambda x: x.sum())
    )
    .reset_index()
)

# Build a separate derived audit table
match_audit = (
    target_matches
    .merge(
        competitions[["comp_id", "comp_name", "match_format", "season"]],
        on="comp_id",
        how="left"
    )
    .merge(delivery_summary, on="match_id", how="left")
    .merge(xi_summary, on="match_id", how="left")
)

# Coverage flags
match_audit["has_deliveries"] = match_audit["delivery_rows"].notna()
match_audit["has_xi_records"] = match_audit["xi_rows"].notna()

# Match-result categories
result_lower = match_audit["result"].fillna("").str.lower()

match_audit["is_abandoned"] = result_lower.str.contains("abandoned")
match_audit["is_no_result"] = result_lower.str.contains("no result")
match_audit["has_super_over_data"] = (
    match_audit["max_innings"].fillna(0) > 2
)
match_audit["has_revised_match_metadata"] = (
    match_audit[
        [
            "first_inn_revised_overs",
            "second_inn_revised_overs",
            "revised_target"
        ]
    ].notna().any(axis=1)
)

# Broad classification only; this is not yet a final eligibility decision
match_audit["result_category"] = "Other result"
match_audit.loc[match_audit["is_abandoned"], "result_category"] = "Abandoned"
match_audit.loc[
    match_audit["is_no_result"], "result_category"
] = "No Result"

print("TARGET MATCH AUDIT TOTALS")
print("Matches:", len(match_audit))
print("With deliveries:", int(match_audit["has_deliveries"].sum()))
print("Without deliveries:", int((~match_audit["has_deliveries"]).sum()))
print("With XI records:", int(match_audit["has_xi_records"].sum()))
print("Without XI records:", int((~match_audit["has_xi_records"]).sum()))
print("Abandoned:", int(match_audit["is_abandoned"].sum()))
print("No Result:", int(match_audit["is_no_result"].sum()))
print("Matches with Super Over data:", int(match_audit["has_super_over_data"].sum()))
print(
    "Matches with revised-overs/target metadata:",
    int(match_audit["has_revised_match_metadata"].sum())
)

print("\nMATCH COUNTS BY RESULT CATEGORY AND DELIVERY COVERAGE")
display(
    match_audit.groupby(
        ["result_category", "has_deliveries"]
    ).size().reset_index(name="matches").to_string(index=False)
)

print("\nMATCH COUNTS BY FORMAT AND RESULT CATEGORY")
display(
    match_audit.groupby(
        ["match_format", "result_category"]
    ).size().reset_index(name="matches").to_string(index=False)
)

print("\nMATCHES WITH DELIVERY DATA BUT NO XI RECORDS")
display(
    match_audit[
        match_audit["has_deliveries"] &
        ~match_audit["has_xi_records"]
    ][
        [
            "match_id", "comp_name", "match_date",
            "home_team", "away_team", "result",
            "delivery_rows"
        ]
    ].to_string(index=False)
)

TARGET MATCH AUDIT TOTALS
Matches: 853
With deliveries: 837
Without deliveries: 16
With XI records: 850
Without XI records: 3
Abandoned: 19
No Result: 21
Matches with Super Over data: 4
Matches with revised-overs/target metadata: 34

MATCH COUNTS BY RESULT CATEGORY AND DELIVERY COVERAGE


'result_category  has_deliveries  matches\n      Abandoned           False       16\n      Abandoned            True        3\n      No Result            True       21\n   Other result            True      813'


MATCH COUNTS BY FORMAT AND RESULT CATEGORY


'match_format result_category  matches\n         T20       Abandoned        5\n         T20       No Result        4\n         T20    Other result      362\n        T20I       Abandoned       14\n        T20I       No Result       17\n        T20I    Other result      451'


MATCHES WITH DELIVERY DATA BUT NO XI RECORDS


'Empty DataFrame\nColumns: [match_id, comp_name, match_date, home_team, away_team, result, delivery_rows]\nIndex: []'

### Investigate abandoned matches with deliveries

In [83]:
abandoned_with_deliveries = match_audit[
    match_audit["is_abandoned"] &
    match_audit["has_deliveries"]
].copy()

print("ABANDONED MATCHES WITH DELIVERY DATA")

display(
    abandoned_with_deliveries[
        [
            "match_id", "comp_id", "comp_name", "match_format",
            "match_date", "home_team", "away_team", "result",
            "delivery_rows", "innings_count", "max_innings",
            "first_inn_revised_overs", "second_inn_revised_overs",
            "revised_target"
        ]
    ].to_string(index=False)
)

# Detailed innings-level delivery profile for these matches
exception_ids = abandoned_with_deliveries["match_id"].tolist()

exception_innings = (
    ball_by_ball[
        ball_by_ball["match_id"].isin(exception_ids)
    ]
    .groupby(["match_id", "innings"])
    .agg(
        delivery_rows=("innings_ball_num", "size"),
        first_sequence=("innings_ball_num", "min"),
        last_sequence=("innings_ball_num", "max"),
        distinct_batters=("striker_id", "nunique"),
        distinct_bowlers=("bowler_id", "nunique"),
        wickets=("is_wicket", "sum"),
        recorded_runs=("runs", "sum")
    )
    .reset_index()
)

print("\nINNINGS-LEVEL PROFILE")
display(exception_innings.to_string(index=False))

# Inspect delivery sequence endpoints to identify truncated innings
print("\nFIRST AND LAST DELIVERY RECORDS")

for match_id in exception_ids:
    match_rows = ball_by_ball[
        ball_by_ball["match_id"] == match_id
    ].sort_values(["innings", "innings_ball_num"])

    print(f"\nMATCH {match_id}")
    display(
        pd.concat([
            match_rows.groupby("innings", group_keys=False).head(2),
            match_rows.groupby("innings", group_keys=False).tail(2)
        ])[
            [
                "innings", "overid", "over_ball_num",
                "innings_ball_num", "striker_name",
                "bowler_name", "runs", "is_wicket",
                "extras_type", "delivery_ist_time"
            ]
        ].to_string(index=False)
    )

# Check the four target matches with Super Over data
print("\nTARGET MATCHES WITH SUPER OVER DATA")

display(
    match_audit[
        match_audit["has_super_over_data"]
    ][
        [
            "match_id", "comp_name", "match_date",
            "home_team", "away_team", "result",
            "innings_count", "max_innings"
        ]
    ].to_string(index=False)
)

ABANDONED MATCHES WITH DELIVERY DATA


' match_id  comp_id                                comp_name match_format match_date            home_team           away_team          result  delivery_rows  innings_count  max_innings  first_inn_revised_overs  second_inn_revised_overs  revised_target\n    15410     1439   South Africa in India T20I Series 2022         T20I 2022-06-19                India        South Africa Match Abandoned         22.000          1.000        1.000                  114.000                       NaN             NaN\n    20226     1715                                 IPL 2023          T20 2023-05-03 Lucknow Super Giants Chennai Super Kings Match Abandoned        118.000          1.000        1.000                      NaN                       NaN             NaN\n    21406     1799 New Zealand in Pakistan T20I Series 2023         T20I 2023-04-20             Pakistan         New Zealand Match Abandoned        118.000          1.000        1.000                      NaN                       NaN         


INNINGS-LEVEL PROFILE


' match_id  innings  delivery_rows  first_sequence  last_sequence  distinct_batters  distinct_bowlers  wickets  recorded_runs\n    15410        1             22               1             22                 4                 3        2             26\n    20226        1            118               1            118                 8                 6        7            119\n    21406        1            118               1            118                 6                 6        5            155'


FIRST AND LAST DELIVERY RECORDS

MATCH 15410


' innings  overid  over_ball_num  innings_ball_num    striker_name    bowler_name  runs  is_wicket extras_type       delivery_ist_time\n       1       1              1                 1    Ishan Kishan Keshav Maharaj     0      False         NaN 2022-06-19 19:53:13 UTC\n       1       1              2                 2    Ishan Kishan Keshav Maharaj     6      False         NaN 2022-06-19 19:53:16 UTC\n       1       4              2                21 Ruturaj Gaikwad    Lungi Ngidi     0       True         NaN 2022-06-19 20:04:09 UTC\n       1       4              3                22    Rishabh Pant    Lungi Ngidi     1      False         NaN 2022-06-19 20:06:31 UTC'

... (2 outputs left out)


MATCH 21406


' innings  overid  over_ball_num  innings_ball_num    striker_name    bowler_name  runs  is_wicket extras_type       delivery_ist_time\n       1       1              1                 1      Tom Latham Shaheen Afridi     4      False         NaN 2023-04-20 21:30:43 UTC\n       1       1              2                 2      Tom Latham Shaheen Afridi     0      False         NaN 2023-04-20 21:31:25 UTC\n       1      19              4               117    Mark Chapman     Haris Rauf     1      False         NaN 2023-04-20 22:52:50 UTC\n       1      19              5               118 Rachin Ravindra     Haris Rauf     0       True         NaN 2023-04-20 22:53:34 UTC'


TARGET MATCHES WITH SUPER OVER DATA


' match_id                           comp_name match_date            home_team             away_team                                                            result  innings_count  max_innings\n    34429 India in Sri Lanka T20I Series 2024 2024-07-30            Sri Lanka                 India  India Tied with Srilanka (India Won the Super Over by 2 wickets)          4.000        4.000\n    40725                            IPL 2025 2025-04-16       Delhi Capitals      Rajasthan Royals Delhi Capitals Tied with Rajasthan Royals (DC won the Super Over)          4.000        4.000\n    45512                       Asia Cup 2025 2025-09-26                India             Sri Lanka                              India won the Super Over(Match Tied)          4.000        4.000\n    50359                            IPL 2026 2026-04-26 Lucknow Super Giants Kolkata Knight Riders              Kolkata Knight Riders Won by Super over (Match Tied)          4.000        4.000'

### Revised-overs and revised-target match audit

In [84]:
revised_cols = [
    "first_inn_revised_overs",
    "second_inn_revised_overs",
    "revised_target"
]

print("REVISED METADATA FIELD SUMMARY")

for col in revised_cols:
    non_null = match_audit[col].dropna()

    print(f"\n{col}")
    print("Non-null matches:", len(non_null))
    print("Unique values:", non_null.nunique())
    print("Minimum:", non_null.min() if len(non_null) else None)
    print("Maximum:", non_null.max() if len(non_null) else None)

    print("\nValue frequencies:")
    display(
        non_null.value_counts()
        .sort_index()
        .rename_axis("value")
        .reset_index(name="matches")
        .to_string(index=False)
    )

# List all matches with at least one revised field
revised_matches = match_audit[
    match_audit[revised_cols].notna().any(axis=1)
].copy()

print("\nALL MATCHES WITH REVISED METADATA")

display(
    revised_matches[
        [
            "match_id", "comp_name", "match_format",
            "match_date", "home_team", "away_team", "result",
            "first_inn_revised_overs",
            "second_inn_revised_overs",
            "revised_target",
            "has_deliveries", "delivery_rows",
            "innings_count", "max_innings"
        ]
    ].sort_values(["match_format", "match_date"])
    .to_string(index=False)
)

print("\nREVISED METADATA BY DELIVERY COVERAGE")

display(
    revised_matches.groupby("has_deliveries").agg(
        matches=("match_id", "nunique"),
        first_innings_revised=("first_inn_revised_overs", "count"),
        second_innings_revised=("second_inn_revised_overs", "count"),
        revised_targets=("revised_target", "count")
    ).reset_index().to_string(index=False)
)

REVISED METADATA FIELD SUMMARY

first_inn_revised_overs
Non-null matches: 25
Unique values: 13
Minimum: 30.0
Maximum: 114.0

Value frequencies:


'  value  matches\n 30.000        2\n 42.000        1\n 48.000        1\n 54.000        4\n 60.000        1\n 66.000        3\n 72.000        1\n 78.000        1\n 84.000        3\n 90.000        2\n102.000        2\n108.000        2\n114.000        2'


second_inn_revised_overs
Non-null matches: 25
Unique values: 12
Minimum: 30.0
Maximum: 114.0

Value frequencies:


'  value  matches\n 30.000        2\n 42.000        2\n 48.000        3\n 54.000        2\n 60.000        2\n 66.000        1\n 72.000        1\n 78.000        1\n 84.000        1\n 90.000        5\n102.000        2\n114.000        3'


revised_target
Non-null matches: 23
Unique values: 23
Minimum: 33.0
Maximum: 213.0

Value frequencies:


'  value  matches\n 33.000        1\n 56.000        1\n 69.000        1\n 78.000        1\n 91.000        1\n 94.000        1\n 96.000        1\n103.000        1\n104.000        1\n115.000        1\n116.000        1\n125.000        1\n126.000        1\n130.000        1\n136.000        1\n143.000        1\n147.000        1\n151.000        1\n152.000        1\n168.000        1\n171.000        1\n203.000        1\n213.000        1'


ALL MATCHES WITH REVISED METADATA


' match_id                                    comp_name match_format match_date                   home_team                   away_team                                             result  first_inn_revised_overs  second_inn_revised_overs  revised_target  has_deliveries  delivery_rows  innings_count  max_innings\n    21586                                     IPL 2023          T20 2023-05-29         Chennai Super Kings              Gujarat Titans Chennai Super Kings Won by 5 wickets  (DLS method)                      NaN                    90.000         171.000            True        216.000          2.000        2.000\n    40727                                     IPL 2025          T20 2025-04-18 Royal Challengers Bengaluru                Punjab Kings                      Punjab Kings Won by 5 wickets                   84.000                    84.000          96.000            True        160.000          2.000        2.000\n    40749                                     IPL 2025      


REVISED METADATA BY DELIVERY COVERAGE


' has_deliveries  matches  first_innings_revised  second_innings_revised  revised_targets\n           True       34                     25                      25               23'

### Test revised-overs values against actual innings data

In [85]:
revised_ids = revised_matches["match_id"].tolist()

# Actual delivery profile by match and innings
actual_innings = (
    ball_by_ball[ball_by_ball["match_id"].isin(revised_ids)]
    .groupby(["match_id", "innings"])
    .agg(
        delivery_rows=("innings_ball_num", "size"),
        max_over_id=("overid", "max"),
        max_sequence=("innings_ball_num", "max"),
        wickets=("is_wicket", "sum")
    )
    .reset_index()
)

# Separate first and second innings
first_actual = actual_innings[
    actual_innings["innings"] == 1
][
    ["match_id", "delivery_rows", "max_over_id", "max_sequence"]
].rename(columns={
    "delivery_rows": "inn1_delivery_rows",
    "max_over_id": "inn1_max_over",
    "max_sequence": "inn1_max_sequence"
})

second_actual = actual_innings[
    actual_innings["innings"] == 2
][
    ["match_id", "delivery_rows", "max_over_id", "max_sequence"]
].rename(columns={
    "delivery_rows": "inn2_delivery_rows",
    "max_over_id": "inn2_max_over",
    "max_sequence": "inn2_max_sequence"
})

revised_comparison = (
    revised_matches[
        [
            "match_id", "comp_name", "match_date",
            "home_team", "away_team", "result",
            "first_inn_revised_overs",
            "second_inn_revised_overs",
            "revised_target"
        ]
    ]
    .merge(first_actual, on="match_id", how="left")
    .merge(second_actual, on="match_id", how="left")
)

# Test the hypothesis that metadata values are balls rather than overs
revised_comparison["first_value_div6"] = (
    revised_comparison["first_inn_revised_overs"] / 6
)
revised_comparison["second_value_div6"] = (
    revised_comparison["second_inn_revised_overs"] / 6
)

print("REVISED METADATA VS OBSERVED INNINGS LENGTH")
display(revised_comparison.to_string(index=False))

print("\nSUMMARY OF DIVISION-BY-SIX TEST")
print(
    "First innings values divisible by 6:",
    int(
        (
            revised_comparison["first_inn_revised_overs"].dropna() % 6 == 0
        ).sum()
    )
)
print(
    "Second innings values divisible by 6:",
    int(
        (
            revised_comparison["second_inn_revised_overs"].dropna() % 6 == 0
        ).sum()
    )
)

# Check whether observed innings actually reach the candidate limits
first_test = revised_comparison.dropna(
    subset=["first_value_div6", "inn1_max_over"]
)
second_test = revised_comparison.dropna(
    subset=["second_value_div6", "inn2_max_over"]
)

print(
    "\nFirst innings observed max over <= metadata/6:",
    int((first_test["inn1_max_over"] <= first_test["first_value_div6"]).sum()),
    "of", len(first_test)
)
print(
    "Second innings observed max over <= metadata/6:",
    int((second_test["inn2_max_over"] <= second_test["second_value_div6"]).sum()),
    "of", len(second_test)
)

REVISED METADATA VS OBSERVED INNINGS LENGTH


' match_id                                    comp_name match_date                   home_team                   away_team                                             result  first_inn_revised_overs  second_inn_revised_overs  revised_target  inn1_delivery_rows  inn1_max_over  inn1_max_sequence  inn2_delivery_rows  inn2_max_over  inn2_max_sequence  first_value_div6  second_value_div6\n    13612      Sri Lanka in Australia T20I Series 2022 2022-02-11                   Australia                   Sri Lanka              Australia Won by 20 runs (D/L method)                      NaN                   114.000         143.000                 124             20                124             118.000         19.000            118.000               NaN             19.000\n    15410       South Africa in India T20I Series 2022 2022-06-19                       India                South Africa                                    Match Abandoned                  114.000                       NaN    


SUMMARY OF DIVISION-BY-SIX TEST
First innings values divisible by 6: 25
Second innings values divisible by 6: 25

First innings observed max over <= metadata/6: 25 of 25
Second innings observed max over <= metadata/6: 23 of 23


### Audit No Result matches

In [86]:
no_result_matches = match_audit[
    match_audit["is_no_result"]
].copy()

# Innings-level profile
no_result_innings = (
    ball_by_ball[
        ball_by_ball["match_id"].isin(no_result_matches["match_id"])
    ]
    .groupby(["match_id", "innings"])
    .agg(
        delivery_rows=("innings_ball_num", "size"),
        max_over=("overid", "max"),
        last_sequence=("innings_ball_num", "max"),
        wickets=("is_wicket", "sum"),
        distinct_batters=("striker_id", "nunique"),
        distinct_bowlers=("bowler_id", "nunique")
    )
    .reset_index()
)

print("NO RESULT MATCH METADATA")
display(
    no_result_matches[
        [
            "match_id", "comp_name", "match_format",
            "match_date", "home_team", "away_team", "result",
            "delivery_rows", "innings_count", "max_innings",
            "has_revised_match_metadata"
        ]
    ].sort_values(["match_format", "match_date"])
    .to_string(index=False)
)

print("\nNO RESULT INNINGS PROFILE")
display(
    no_result_innings.sort_values(["match_id", "innings"])
    .to_string(index=False)
)

print("\nNO RESULT MATCHES BY INNINGS COUNT")
display(
    no_result_matches.groupby(
        ["innings_count", "has_revised_match_metadata"]
    ).size().reset_index(name="matches")
    .to_string(index=False)
)

print("\nNO RESULT MATCHES WITH ONLY ONE INNINGS")
display(
    no_result_matches[
        no_result_matches["innings_count"] == 1
    ][
        [
            "match_id", "comp_name", "match_date",
            "home_team", "away_team", "delivery_rows"
        ]
    ].to_string(index=False)
)

NO RESULT MATCH METADATA


' match_id                                          comp_name match_format match_date             home_team                away_team    result  delivery_rows  innings_count  max_innings  has_revised_match_metadata\n    40737                                           IPL 2025          T20 2025-04-26 Kolkata Knight Riders             Punjab Kings No Result        130.000          2.000        2.000                       False\n    40748                                           IPL 2025          T20 2025-05-05   Sunrisers Hyderabad           Delhi Capitals No Result        123.000          1.000        1.000                       False\n    40751                                           IPL 2025          T20 2025-05-08          Punjab Kings           Delhi Capitals No Result         63.000          1.000        1.000                       False\n    50136                                           IPL 2026          T20 2026-04-06 Kolkata Knight Riders             Punjab Kings No Result  


NO RESULT INNINGS PROFILE


' match_id  innings  delivery_rows  max_over  last_sequence  wickets  distinct_batters  distinct_bowlers\n    15546        1             82        13             82        8                 9                 5\n    17765        1             75        12             75        2                 4                 5\n    17765        2             25         4             25        3                 5                 3\n    28333        1             66        11             66        2                 4                 5\n    31707        1              2         1              2        1                 1                 1\n    35424        1            125        20            125        5                 7                 6\n    37353        1             30         5             30        0                 2                 3\n    39983        1             56         9             56        5                 7                 5\n    39985        1            113        18           


NO RESULT MATCHES BY INNINGS COUNT


' innings_count  has_revised_match_metadata  matches\n         1.000                       False       11\n         1.000                        True        8\n         2.000                       False        1\n         2.000                        True        1'


NO RESULT MATCHES WITH ONLY ONE INNINGS


' match_id                                          comp_name match_date             home_team                away_team  delivery_rows\n    15546         Bangladesh in West Indies T20I Series 2022 2022-07-02           West Indies               Bangladesh         82.000\n    28333         Bangladesh in New Zealand T20I Series 2023 2023-12-29           New Zealand               Bangladesh         66.000\n    31707           New Zealand in Pakistan T20I Series 2024 2024-04-18              Pakistan              New Zealand          2.000\n    37353            England in West Indies T20I Series 2024 2024-11-18           West Indies                  England         30.000\n    35424 Netherlands Tri-Nation T20I Series [Aug 2024] 2024 2024-08-24                Canada United States Of America        125.000\n    40748                                           IPL 2025 2025-05-05   Sunrisers Hyderabad           Delhi Capitals        123.000\n    40751                                           IP

### Inspect the two two-innings No Result matches

In [87]:
two_innings_no_result = no_result_matches[
    no_result_matches["innings_count"] == 2
].copy()

print("TWO-INNINGS NO RESULT MATCHES")

display(
    two_innings_no_result[
        [
            "match_id", "comp_name", "match_format",
            "match_date", "home_team", "away_team",
            "result", "delivery_rows",
            "first_inn_revised_overs",
            "second_inn_revised_overs",
            "revised_target"
        ]
    ].to_string(index=False)
)

# Inspect every innings in these matches
two_innings_ids = two_innings_no_result["match_id"].tolist()

two_innings_detail = (
    ball_by_ball[
        ball_by_ball["match_id"].isin(two_innings_ids)
    ]
    .groupby(["match_id", "innings"])
    .agg(
        delivery_rows=("innings_ball_num", "size"),
        max_over=("overid", "max"),
        last_sequence=("innings_ball_num", "max"),
        wickets=("is_wicket", "sum"),
        recorded_runs=("runs", "sum"),
        distinct_batters=("striker_id", "nunique"),
        distinct_bowlers=("bowler_id", "nunique")
    )
    .reset_index()
)

print("\nINNINGS COMPLETENESS PROFILE")
display(two_innings_detail.to_string(index=False))

# First and last three deliveries of each innings
print("\nINNINGS ENDPOINT RECORDS")

for match_id in two_innings_ids:
    print(f"\nMATCH {match_id}")

    match_rows = ball_by_ball[
        ball_by_ball["match_id"] == match_id
    ].sort_values(["innings", "innings_ball_num"])

    display(
        pd.concat([
            match_rows.groupby("innings", group_keys=False).head(3),
            match_rows.groupby("innings", group_keys=False).tail(3)
        ])[
            [
                "innings", "overid", "over_ball_num",
                "innings_ball_num", "striker_name",
                "bowler_name", "runs", "is_wicket",
                "extras_type", "delivery_ist_time"
            ]
        ].to_string(index=False)
    )

TWO-INNINGS NO RESULT MATCHES


' match_id                             comp_name match_format match_date             home_team    away_team    result  delivery_rows  first_inn_revised_overs  second_inn_revised_overs  revised_target\n    17765 England in Australia T20I Series 2022         T20I 2022-10-14             Australia      England No Result        100.000                   72.000                    72.000         130.000\n    40737                              IPL 2025          T20 2025-04-26 Kolkata Knight Riders Punjab Kings No Result        130.000                      NaN                       NaN             NaN'


INNINGS COMPLETENESS PROFILE


' match_id  innings  delivery_rows  max_over  last_sequence  wickets  recorded_runs  distinct_batters  distinct_bowlers\n    17765        1             75        12             75        2            105                 4                 5\n    17765        2             25         4             25        3             23                 5                 3\n    40737        1            123        20            123        4            198                 6                 6\n    40737        2              7         1              7        0              5                 2                 1'


INNINGS ENDPOINT RECORDS

MATCH 17765


' innings  overid  over_ball_num  innings_ball_num   striker_name    bowler_name  runs  is_wicket extras_type       delivery_ist_time\n       1       1              1                 1    Jos Buttler Mitchell Starc     0      False        wide 2022-10-14 13:41:22 UTC\n       1       1              2                 2    Jos Buttler Mitchell Starc     2      False         NaN 2022-10-14 13:41:48 UTC\n       1       1              3                 3    Jos Buttler Mitchell Starc     0      False         NaN 2022-10-14 13:42:15 UTC\n       2       1              1                 1    Aaron Finch   Chris Woakes     0       True         NaN 2022-10-14 16:20:29 UTC\n       2       1              2                 2 Mitchell Marsh   Chris Woakes     0       True         NaN 2022-10-14 16:22:06 UTC\n       2       1              3                 3   Steven Smith   Chris Woakes     0      False         NaN 2022-10-14 16:23:41 UTC\n       1      12              4                73    Jos Butt


MATCH 40737


' innings  overid  over_ball_num  innings_ball_num       striker_name   bowler_name  runs  is_wicket extras_type       delivery_ist_time\n       1       1              1                 1      Priyansh Arya Vaibhav Arora     4      False         NaN 2025-04-26 14:00:49 UTC\n       1       1              2                 2      Priyansh Arya Vaibhav Arora     1      False         NaN 2025-04-26 14:01:41 UTC\n       1       1              3                 3  Prabhsimran Singh Vaibhav Arora     0      False         NaN 2025-04-26 14:02:09 UTC\n       2       1              1                 1 Rahmanullah Gurbaz  Marco Jansen     0      False        wide 2025-04-26 15:58:09 UTC\n       2       1              2                 2 Rahmanullah Gurbaz  Marco Jansen     0      False         NaN 2025-04-26 15:58:34 UTC\n       2       1              3                 3 Rahmanullah Gurbaz  Marco Jansen     1      False         NaN 2025-04-26 15:59:13 UTC\n       1      20              4         

### Check legal-ball evidence for match 17765

In [88]:
audit_ids = [17765, 40737]

no_result_delivery_check = (
    ball_by_ball[
        ball_by_ball["match_id"].isin(audit_ids)
    ]
    .groupby(["match_id", "innings"])
    .agg(
        delivery_records=("innings_ball_num", "size"),
        wides=("extras_type", lambda x: (x == "wide").sum()),
        oversteps=("extras_type", lambda x: (x == "Overstep").sum()),
        other_extras=("extras_type", lambda x: x.isin(
            ["other", "legbyes", "byes"]
        ).sum()),
        wickets=("is_wicket", "sum"),
        recorded_runs=("runs", "sum")
    )
    .reset_index()
)

no_result_delivery_check["records_minus_known_illegal"] = (
    no_result_delivery_check["delivery_records"]
    - no_result_delivery_check["wides"]
    - no_result_delivery_check["oversteps"]
)

display(no_result_delivery_check.to_string(index=False))

' match_id  innings  delivery_records  wides  oversteps  other_extras  wickets  recorded_runs  records_minus_known_illegal\n    17765        1                75      3          0             1        2            105                           72\n    17765        2                25      2          0             2        3             23                           23\n    40737        1               123      2          1             0        4            198                          120\n    40737        2                 7      1          0             1        0              5                            6'

### Build the final eligibility summary

In [89]:
eligibility = match_audit.copy()

eligibility["proposed_category"] = "Include - standard result match"

# Abandoned matches
eligibility.loc[
    eligibility["is_abandoned"] & ~eligibility["has_deliveries"],
    "proposed_category"
] = "Exclude - abandoned, no deliveries"

eligibility.loc[
    eligibility["is_abandoned"] & eligibility["has_deliveries"],
    "proposed_category"
] = "Exclude - abandoned, partial deliveries"

# No Result matches
eligibility.loc[
    eligibility["is_no_result"] & (eligibility["innings_count"] == 1),
    "proposed_category"
] = "Exclude - No Result, one innings"

eligibility.loc[
    eligibility["is_no_result"] & (eligibility["innings_count"] == 2),
    "proposed_category"
] = "Exclude pending validation - No Result, two innings"

# Separate Super Over matches
eligibility.loc[
    eligibility["has_super_over_data"],
    "proposed_category"
] = "Include regulation innings only - Super Over separate"

# Revised match flag, preserving prior classification
eligibility["has_revised_metadata"] = (
    eligibility[
        [
            "first_inn_revised_overs",
            "second_inn_revised_overs",
            "revised_target"
        ]
    ].notna().any(axis=1)
)

print("PROPOSED ELIGIBILITY CATEGORY COUNTS")

display(
    eligibility.groupby("proposed_category")
    .agg(
        matches=("match_id", "nunique"),
        matches_with_deliveries=("has_deliveries", "sum")
    )
    .reset_index()
    .sort_values("matches", ascending=False)
    .to_string(index=False)
)

print("\nREVISED METADATA FLAG COUNTS")
display(
    eligibility.groupby(
        ["proposed_category", "has_revised_metadata"]
    ).size().reset_index(name="matches")
    .to_string(index=False)
)

print("\nPROPOSED PRIMARY POPULATION")
print(
    "Standard completed-result matches:",
    int((eligibility["proposed_category"] == "Include - standard result match").sum())
)
print(
    "Super Over matches with regulation innings retained:",
    int((eligibility["proposed_category"] == "Include regulation innings only - Super Over separate").sum())
)
print(
    "Total matches with some proposed primary-analysis use:",
    int(eligibility["proposed_category"].isin([
        "Include - standard result match",
        "Include regulation innings only - Super Over separate"
    ]).sum())
)

PROPOSED ELIGIBILITY CATEGORY COUNTS


'                                    proposed_category  matches  matches_with_deliveries\n                      Include - standard result match      809                      809\n                     Exclude - No Result, one innings       19                       19\n                   Exclude - abandoned, no deliveries       16                        0\nInclude regulation innings only - Super Over separate        4                        4\n              Exclude - abandoned, partial deliveries        3                        3\n  Exclude pending validation - No Result, two innings        2                        2'


REVISED METADATA FLAG COUNTS


'                                    proposed_category  has_revised_metadata  matches\n                     Exclude - No Result, one innings                 False       11\n                     Exclude - No Result, one innings                  True        8\n                   Exclude - abandoned, no deliveries                 False       16\n              Exclude - abandoned, partial deliveries                 False        2\n              Exclude - abandoned, partial deliveries                  True        1\n  Exclude pending validation - No Result, two innings                 False        1\n  Exclude pending validation - No Result, two innings                  True        1\n                      Include - standard result match                 False      785\n                      Include - standard result match                  True       24\nInclude regulation innings only - Super Over separate                 False        4'


PROPOSED PRIMARY POPULATION
Standard completed-result matches: 809
Super Over matches with regulation innings retained: 4
Total matches with some proposed primary-analysis use: 813


### Final overlap and reconciliation check

In [90]:
print("SUPER OVER OVERLAP CHECK")

super_over_overlap = eligibility[
    eligibility["has_super_over_data"]
][
    [
        "match_id", "result",
        "is_abandoned", "is_no_result",
        "proposed_category"
    ]
]

display(super_over_overlap.to_string(index=False))

print("\nCATEGORY TOTAL RECONCILIATION")

print("Total target matches:", len(eligibility))
print("Unique match IDs:", eligibility["match_id"].nunique())

print("\nEligibility category totals:")
print(eligibility["proposed_category"].value_counts().to_string())

# Check for any match satisfying multiple exclusion conditions
eligibility["exclusion_condition_count"] = (
    eligibility["is_abandoned"].astype(int)
    + eligibility["is_no_result"].astype(int)
)

print(
    "\nMatches satisfying multiple exclusion conditions:",
    int((eligibility["exclusion_condition_count"] > 1).sum())
)

# Confirm primary population
primary_categories = [
    "Include - standard result match",
    "Include regulation innings only - Super Over separate"
]

primary_population = eligibility[
    eligibility["proposed_category"].isin(primary_categories)
]

print("\nFINAL PRIMARY POPULATION CHECK")
print("Primary matches:", len(primary_population))
print("Primary matches with deliveries:",
      int(primary_population["has_deliveries"].sum()))
print("Primary matches without deliveries:",
      int((~primary_population["has_deliveries"]).sum()))
print("Primary matches with XI:",
      int(primary_population["has_xi_records"].sum()))

SUPER OVER OVERLAP CHECK


' match_id                                                            result  is_abandoned  is_no_result                                     proposed_category\n    34429  India Tied with Srilanka (India Won the Super Over by 2 wickets)         False         False Include regulation innings only - Super Over separate\n    40725 Delhi Capitals Tied with Rajasthan Royals (DC won the Super Over)         False         False Include regulation innings only - Super Over separate\n    45512                              India won the Super Over(Match Tied)         False         False Include regulation innings only - Super Over separate\n    50359              Kolkata Knight Riders Won by Super over (Match Tied)         False         False Include regulation innings only - Super Over separate'


CATEGORY TOTAL RECONCILIATION
Total target matches: 853
Unique match IDs: 853

Eligibility category totals:
proposed_category
Include - standard result match                          809
Exclude - No Result, one innings                          19
Exclude - abandoned, no deliveries                        16
Include regulation innings only - Super Over separate      4
Exclude - abandoned, partial deliveries                    3
Exclude pending validation - No Result, two innings        2

Matches satisfying multiple exclusion conditions: 0

FINAL PRIMARY POPULATION CHECK
Primary matches: 813
Primary matches with deliveries: 813
Primary matches without deliveries: 0
Primary matches with XI: 813


# Build the primary delivery-level audit population



### Primary delivery population audit

In [91]:
primary_categories = [
    "Include - standard result match",
    "Include regulation innings only - Super Over separate"
]

primary_match_ids = eligibility.loc[
    eligibility["proposed_category"].isin(primary_categories),
    "match_id"
].unique()

# All deliveries from the proposed primary match population
primary_deliveries = ball_by_ball[
    ball_by_ball["match_id"].isin(primary_match_ids)
].copy()

# Regulation innings only
primary_regulation_deliveries = primary_deliveries[
    primary_deliveries["innings"].isin([1, 2])
].copy()

# Super Over / extra innings, retained separately
primary_super_over_deliveries = primary_deliveries[
    primary_deliveries["innings"] > 2
].copy()

print("PRIMARY DELIVERY POPULATION")
print("Eligible matches:", len(primary_match_ids))
print("All delivery records:", len(primary_deliveries))
print("Regulation delivery records:", len(primary_regulation_deliveries))
print("Super Over / extra-innings records:", len(primary_super_over_deliveries))

print("\nREGULATION DELIVERY COUNTS BY INNINGS")
display(
    primary_regulation_deliveries.groupby("innings").agg(
        delivery_records=("match_id", "size"),
        matches=("match_id", "nunique"),
        distinct_batters=("striker_id", "nunique"),
        distinct_bowlers=("bowler_id", "nunique")
    ).reset_index().to_string(index=False)
)

print("\nSUPER OVER / EXTRA-INNINGS COUNTS")
display(
    primary_super_over_deliveries.groupby(
        ["match_id", "innings"]
    ).agg(
        delivery_records=("innings_ball_num", "size"),
        distinct_batters=("striker_id", "nunique"),
        distinct_bowlers=("bowler_id", "nunique")
    ).reset_index().to_string(index=False)
)

print("\nCOMPETITION FORMAT COVERAGE IN REGULATION DELIVERIES")
regulation_match_ids = primary_regulation_deliveries["match_id"].unique()

display(
    eligibility[
        eligibility["match_id"].isin(regulation_match_ids)
    ].groupby("match_format").agg(
        matches=("match_id", "nunique")
    ).reset_index().to_string(index=False)
)

print("\nPOPULATION INTEGRITY CHECKS")
print(
    "Regulation deliveries with innings outside 1–2:",
    int((~primary_regulation_deliveries["innings"].isin([1, 2])).sum())
)
print(
    "Primary delivery records outside eligible matches:",
    int((~primary_deliveries["match_id"].isin(primary_match_ids)).sum())
)
print(
    "Super Over match IDs:",
    sorted(primary_super_over_deliveries["match_id"].unique().tolist())
)

PRIMARY DELIVERY POPULATION
Eligible matches: 813
All delivery records: 192261
Regulation delivery records: 192235
Super Over / extra-innings records: 26

REGULATION DELIVERY COUNTS BY INNINGS


' innings  delivery_records  matches  distinct_batters  distinct_bowlers\n       1            100333      813               700               521\n       2             91902      813               744               522'


SUPER OVER / EXTRA-INNINGS COUNTS


' match_id  innings  delivery_records  distinct_batters  distinct_bowlers\n    34429        3                 4                 3                 1\n    34429        4                 1                 1                 1\n    40725        3                 6                 2                 1\n    40725        4                 4                 2                 1\n    45512        3                 6                 3                 1\n    45512        4                 1                 1                 1\n    50359        3                 3                 3                 1\n    50359        4                 1                 1                 1'


COMPETITION FORMAT COVERAGE IN REGULATION DELIVERIES


'match_format  matches\n         T20      362\n        T20I      451'


POPULATION INTEGRITY CHECKS
Regulation deliveries with innings outside 1–2: 0
Primary delivery records outside eligible matches: 0
Super Over match IDs: [34429, 40725, 45512, 50359]


### Audit unresolved anomalies inside the primary population

In [92]:
p = primary_regulation_deliveries.copy()

print("PRIMARY POPULATION: FIELD QUALITY CHECKS")
print("Total regulation deliveries:", len(p))

# 1. Extras fields
print("\n1. EXTRAS FIELD CHECKS")

print("extras1_runs distribution:")
display(
    p["extras1_runs"].value_counts(dropna=False)
    .sort_index()
    .rename_axis("extras1_code")
    .reset_index(name="deliveries")
    .to_string(index=False)
)

print("\nextras_type distribution:")
display(
    p["extras_type"].fillna("NO_EXTRAS")
    .value_counts()
    .rename_axis("extras_type")
    .reset_index(name="deliveries")
    .to_string(index=False)
)

print(
    "Non-zero extras_2_runs:",
    int((p["extras_2_runs"] != 0).sum())
)

# 2. Boundary flag inconsistencies
four_mismatch = (p["runs"] == 4) & (~p["is_four"])
six_mismatch = (p["runs"] == 6) & (~p["is_six"])

print("\n2. BOUNDARY FLAG CHECKS")
print("Runs=4 but is_four=False:", int(four_mismatch.sum()))
print("Runs=6 but is_six=False:", int(six_mismatch.sum()))
print("Both boundary flags true:", int((p["is_four"] & p["is_six"]).sum()))

# 3. Wicket and dismissed-player consistency
print("\n3. WICKET FIELD CHECKS")

print("Wicket deliveries:", int(p["is_wicket"].sum()))
print(
    "Wickets missing dismissal_type:",
    int((p["is_wicket"] & p["dismissal_type"].isna()).sum())
)
print(
    "Non-wickets with dismissal_type:",
    int((~p["is_wicket"] & p["dismissal_type"].notna()).sum())
)
print(
    "Wickets missing dismissed_player_id:",
    int((p["is_wicket"] & p["dismissed_player_id"].isna()).sum())
)
print(
    "Non-wickets with dismissed_player_id:",
    int((~p["is_wicket"] & p["dismissed_player_id"].notna()).sum())
)

# 4. Bowling speed reliability
print("\n4. BOWLING SPEED CHECKS")

print("Zero speed:", int((p["bowl_speed"] == 0).sum()))
print("Negative speed:", int((p["bowl_speed"] < 0).sum()))
print("Positive speed below 50:", int(
    ((p["bowl_speed"] > 0) & (p["bowl_speed"] < 50)).sum()
))
print("Speed above 200:", int((p["bowl_speed"] > 200).sum()))
print("Speed above 500:", int((p["bowl_speed"] > 500).sum()))

# 5. Timestamp quality
print("\n5. TIMESTAMP CHECKS")

p["parsed_delivery_time"] = pd.to_datetime(
    p["delivery_ist_time"], errors="coerce", utc=True
)

print("Unparseable timestamps:", int(p["parsed_delivery_time"].isna().sum()))
print(
    "Midnight timestamps:",
    int(
        (p["parsed_delivery_time"].dt.hour == 0)
        .where(p["parsed_delivery_time"].notna(), False)
        .sum()
    )
)
print(
    "Duplicate timestamp values after first occurrence:",
    int(p["delivery_ist_time"].duplicated().sum())
)

# 6. Extended overs without known wides/oversteps
print("\n6. EXTENDED OVER CHECKS")

primary_over_audit = (
    p.groupby(["match_id", "innings", "overid"])
    .agg(
        delivery_rows=("over_ball_num", "size"),
        wides=("extras_type", lambda x: (x == "wide").sum()),
        oversteps=("extras_type", lambda x: (x == "Overstep").sum())
    )
    .reset_index()
)

primary_over_audit["known_illegal"] = (
    primary_over_audit["wides"] + primary_over_audit["oversteps"]
)

exceptional_overs = primary_over_audit[
    (primary_over_audit["delivery_rows"] > 6) &
    (primary_over_audit["known_illegal"] == 0)
]

print("Overs with more than 6 delivery records:",
      int((primary_over_audit["delivery_rows"] > 6).sum()))
print("Extended overs with no wide/overstep:",
      len(exceptional_overs))
print(
    "Delivery records in those exceptional overs:",
    int(exceptional_overs["delivery_rows"].sum())
)

# 7. Summary of unresolved high-priority flags
print("\n7. SUMMARY")
print("Boundary mismatches:", int(four_mismatch.sum() + six_mismatch.sum()))
print("Wickets missing dismissed player:",
      int((p["is_wicket"] & p["dismissed_player_id"].isna()).sum()))
print("Non-wickets with dismissed player:",
      int((~p["is_wicket"] & p["dismissed_player_id"].notna()).sum()))
print("Extended overs without known wides/oversteps:",
      len(exceptional_overs))

PRIMARY POPULATION: FIELD QUALITY CHECKS
Total regulation deliveries: 192235

1. EXTRAS FIELD CHECKS
extras1_runs distribution:


' extras1_code  deliveries\n            1         557\n            2        7419\n            3        2635\n            4         505\n            5         256\n            6      180862\n            7           1'


extras_type distribution:


'extras_type  deliveries\n  NO_EXTRAS      180862\n       wide        7419\n    legbyes        2635\n   Overstep         557\n       byes         505\n      other         256\n    Penalty           1'

Non-zero extras_2_runs: 35

2. BOUNDARY FLAG CHECKS
Runs=4 but is_four=False: 20
Runs=6 but is_six=False: 3
Both boundary flags true: 0

3. WICKET FIELD CHECKS
Wicket deliveries: 10392
Wickets missing dismissal_type: 0
Non-wickets with dismissal_type: 0
Wickets missing dismissed_player_id: 1
Non-wickets with dismissed_player_id: 11

4. BOWLING SPEED CHECKS
Zero speed: 70197
Negative speed: 1
Positive speed below 50: 22
Speed above 200: 18
Speed above 500: 17

5. TIMESTAMP CHECKS
Unparseable timestamps: 0
Midnight timestamps: 3986
Duplicate timestamp values after first occurrence: 165

6. EXTENDED OVER CHECKS
Overs with more than 6 delivery records: 6450
Extended overs with no wide/overstep: 147
Delivery records in those exceptional overs: 1036

7. SUMMARY
Boundary mismatches: 23
Wickets missing dismissed player: 1
Non-wickets with dismissed player: 11
Extended overs without known wides/oversteps: 147


### Player identity and participation audit within the primary population

In [93]:
p = primary_regulation_deliveries.copy()

# 1. Player identity observations from actual regulation deliveries
identity_observations = pd.concat([
    p[["striker_id", "striker_name"]].rename(
        columns={"striker_id": "player_id", "striker_name": "player_name"}
    ),
    p[["non_striker_id", "non_striker_name"]].rename(
        columns={"non_striker_id": "player_id", "non_striker_name": "player_name"}
    ),
    p[["bowler_id", "bowler_name"]].rename(
        columns={"bowler_id": "player_id", "bowler_name": "player_name"}
    )
], ignore_index=True).dropna().drop_duplicates()

id_name_counts = (
    identity_observations.groupby("player_id")["player_name"]
    .nunique()
)

name_id_counts = (
    identity_observations.groupby("player_name")["player_id"]
    .nunique()
)

print("1. PRIMARY POPULATION IDENTITY CHECKS")
print("Unique player IDs:", identity_observations["player_id"].nunique())
print("Unique player names:", identity_observations["player_name"].nunique())
print("IDs associated with multiple names:", int((id_name_counts > 1).sum()))
print("Names associated with multiple IDs:", int((name_id_counts > 1).sum()))

print("\nNAME COLLISIONS IN PRIMARY DELIVERIES")
collision_names = name_id_counts[name_id_counts > 1].index

display(
    identity_observations[
        identity_observations["player_name"].isin(collision_names)
    ].drop_duplicates().sort_values("player_name")
    .to_string(index=False)
)

# 2. Playing-XI duplicate player registrations in eligible matches
primary_xi = playing_eleven[
    playing_eleven["match_id"].isin(primary_match_ids)
].copy()

xi_duplicate_groups = (
    primary_xi.groupby(["match_id", "team_id", "player_id"])
    .size()
    .reset_index(name="record_count")
)

xi_duplicate_groups = xi_duplicate_groups[
    xi_duplicate_groups["record_count"] > 1
]

print("\n2. REPEATED PLAYING-XI PLAYER REGISTRATIONS")
print("Repeated match-team-player groups:", len(xi_duplicate_groups))
print("Records in repeated groups:",
      int(xi_duplicate_groups["record_count"].sum()))

display(xi_duplicate_groups.to_string(index=False))

# 3. Compare distinct delivery participants with players marked as played
delivery_participants = pd.concat([
    p[["match_id", "batting_team_id", "striker_id"]].rename(
        columns={"batting_team_id": "team_id", "striker_id": "player_id"}
    ),
    p[["match_id", "batting_team_id", "non_striker_id"]].rename(
        columns={"batting_team_id": "team_id", "non_striker_id": "player_id"}
    ),
    p[["match_id", "bowling_team_id", "bowler_id"]].rename(
        columns={"bowling_team_id": "team_id", "bowler_id": "player_id"}
    )
], ignore_index=True).drop_duplicates()

played_xi = primary_xi[
    primary_xi["is_played"] == True
][["match_id", "team_id", "player_id"]].drop_duplicates()

delivery_keys = set(map(tuple, delivery_participants.to_numpy()))
played_keys = set(map(tuple, played_xi.to_numpy()))

missing_from_played_xi = delivery_keys - played_keys
played_without_delivery = played_keys - delivery_keys

print("\n3. DELIVERY PARTICIPATION VS PLAYED XI")
print("Distinct delivery match-team-player combinations:", len(delivery_keys))
print("Distinct played-XI combinations:", len(played_keys))
print("Delivery participants absent from played XI:", len(missing_from_played_xi))
print("Played-XI players with no delivery involvement:", len(played_without_delivery))

print("\nDELIVERY PARTICIPANTS ABSENT FROM PLAYED XI")
display(
    pd.DataFrame(
        list(missing_from_played_xi),
        columns=["match_id", "team_id", "player_id"]
    ).to_string(index=False)
)

# 4. Check whether all primary delivery participants have mapping records
primary_mapping = competition_team_player_mapping[
    competition_team_player_mapping["comp_id"].isin(
        target_competitions["comp_id"]
    )
]

match_comp = target_matches[["match_id", "comp_id"]].drop_duplicates()

participant_comp = delivery_participants.merge(
    match_comp, on="match_id", how="left"
)

mapping_keys = set(map(tuple, primary_mapping[
    ["comp_id", "team_id", "player_id"]
].drop_duplicates().to_numpy()))

participant_mapping_keys = set(map(tuple, participant_comp[
    ["comp_id", "team_id", "player_id"]
].drop_duplicates().to_numpy()))

unmapped_participants = participant_mapping_keys - mapping_keys

print("\n4. COMPETITION-TEAM-PLAYER MAPPING")
print("Unique participant-comp-team-player combinations:",
      len(participant_mapping_keys))
print("Unmapped combinations:", len(unmapped_participants))

if unmapped_participants:
    display(
        pd.DataFrame(
            list(unmapped_participants),
            columns=["comp_id", "team_id", "player_id"]
        ).to_string(index=False)
    )

1. PRIMARY POPULATION IDENTITY CHECKS
Unique player IDs: 892
Unique player names: 890
IDs associated with multiple names: 0
Names associated with multiple IDs: 2

NAME COLLISIONS IN PRIMARY DELIVERIES


' player_id player_name\n      7446 Abdul Samad\n      6922 Abdul Samad\n      7338  Haider Ali\n     28756  Haider Ali'


2. REPEATED PLAYING-XI PLAYER REGISTRATIONS
Repeated match-team-player groups: 35
Records in repeated groups: 75


' match_id  team_id  player_id  record_count\n    13613       20        223             2\n    13613       20        250             2\n    13613       20        838             2\n    13613       20       2639             2\n    13613       20       3539             2\n    13613       20       4736             2\n    13613       27       1100             2\n    13613       27       2351             2\n    13613       27       2436             2\n    13613       27       4895             2\n    13613       27       4902             2\n    13613       27       6931             2\n    21176       17       2246             2\n    21176       17       3323             2\n    21176       17       3374             2\n    21176       17       3700             2\n    21176       17       4106             2\n    21176       17      25534             2\n    21176       27       1481             2\n    21176       27       2407             2\n    21176       27       3672             2\n    21176


3. DELIVERY PARTICIPATION VS PLAYED XI
Distinct delivery match-team-player combinations: 17937
Distinct played-XI combinations: 18449
Delivery participants absent from played XI: 0
Played-XI players with no delivery involvement: 512

DELIVERY PARTICIPANTS ABSENT FROM PLAYED XI


'Empty DataFrame\nColumns: [match_id, team_id, player_id]\nIndex: []'


4. COMPETITION-TEAM-PLAYER MAPPING
Unique participant-comp-team-player combinations: 4778
Unmapped combinations: 0


### Investigate identity collisions and XI-only participants

In [94]:
# 1. Investigate the two same-name player pairs
collision_pairs = {
    "Abdul Samad": [6922, 7446],
    "Haider Ali": [7338, 28756]
}

print("1. SAME-NAME PLAYER PAIR INVESTIGATION")

for player_name, player_ids in collision_pairs.items():
    print(f"\n{player_name}: IDs {player_ids}")

    pair_xi = primary_xi[
        primary_xi["player_id"].isin(player_ids)
    ].merge(
        target_matches[["match_id", "comp_id", "match_date"]],
        on="match_id",
        how="left"
    )

    print("Playing-XI records:")
    display(
        pair_xi[
            [
                "match_id", "comp_id", "match_date",
                "team_id", "player_id", "is_played"
            ]
        ].sort_values(["match_id", "player_id"])
        .to_string(index=False)
    )

    match_sets = {
        player_id: set(
            pair_xi.loc[
                pair_xi["player_id"] == player_id, "match_id"
            ]
        )
        for player_id in player_ids
    }

    common_matches = (
        match_sets[player_ids[0]] &
        match_sets[player_ids[1]]
    )

    print("Common match IDs:", sorted(common_matches))

# 2. Investigate players marked played but absent from deliveries
print("\n2. PLAYED-XI PARTICIPANTS WITHOUT DELIVERY INVOLVEMENT")

xi_only_keys = played_keys - delivery_keys

xi_only_df = pd.DataFrame(
    list(xi_only_keys),
    columns=["match_id", "team_id", "player_id"]
)

xi_only_detail = xi_only_df.merge(
    primary_xi,
    on=["match_id", "team_id", "player_id"],
    how="left"
).merge(
    target_matches[["match_id", "comp_id", "match_date"]],
    on="match_id",
    how="left"
)

print("Unique XI-only participants:", len(xi_only_df))
print("\nFlag distribution among XI-only participants:")

for col in ["is_captain", "is_wk", "is_impact", "is_impact_off"]:
    print(f"\n{col}")
    display(
        xi_only_detail[col].value_counts(dropna=False)
        .rename_axis(col)
        .reset_index(name="records")
        .to_string(index=False)
    )

# 3. Summarise XI-only participants by competition format
xi_only_detail = xi_only_detail.merge(
    competitions[["comp_id", "match_format", "comp_name"]],
    on="comp_id",
    how="left"
)

print("\nXI-ONLY PARTICIPANTS BY FORMAT")
display(
    xi_only_detail.groupby("match_format").agg(
        records=("player_id", "size"),
        distinct_players=("player_id", "nunique"),
        distinct_matches=("match_id", "nunique")
    ).reset_index().to_string(index=False)
)

# 4. Check how many XI-only participants have batting order
print("\nBATTING ORDER COVERAGE FOR XI-ONLY PARTICIPANTS")
print(
    "With batting order:",
    int(xi_only_detail["batting_order"].notna().sum())
)
print(
    "Without batting order:",
    int(xi_only_detail["batting_order"].isna().sum())
)

1. SAME-NAME PLAYER PAIR INVESTIGATION

Abdul Samad: IDs [6922, 7446]
Playing-XI records:


' match_id  comp_id match_date  team_id  player_id  is_played\n    14098     1358 2022-03-29      242       7446       True\n    14105     1358 2022-04-04      242       7446       True\n    20184     1715 2023-04-02      242       7446       True\n    20190     1715 2023-04-07      242       7446       True\n    20205     1715 2023-04-18      242       7446       True\n    20220     1715 2023-04-29      242       7446       True\n    20227     1715 2023-05-04      242       7446       True\n    20232     1715 2023-05-07      242       7446       True\n    20238     1715 2023-05-13      242       7446       True\n    20242     1715 2023-05-15      242       7446       True\n    20245     1715 2023-05-18      242       7446       True\n    30058     2193 2024-03-23      242       7446       True\n    30063     2193 2024-03-27      242       7446       True\n    30067     2193 2024-03-31      242       7446       True\n    30073     2193 2024-04-05      242       7446       True\n    312

Common match IDs: []

Haider Ali: IDs [7338, 28756]
Playing-XI records:


' match_id  comp_id match_date  team_id  player_id  is_played\n    17553     1595 2022-09-20       25       7338       True\n    17554     1595 2022-09-22       25       7338       True\n    17555     1595 2022-09-23       25       7338       True\n    17557     1595 2022-09-28       25       7338       True\n    17558     1595 2022-09-30       25       7338       True\n    17737     1608 2022-10-07       25       7338       True\n    17738     1608 2022-10-08       25       7338       True\n    17740     1608 2022-10-11       25       7338       True\n    17742     1608 2022-10-13       25       7338       True\n    17743     1608 2022-10-14       25       7338       True\n    42783     2975 2025-05-17      272      28756       True\n    42784     2975 2025-05-19      272      28756       True\n    43171     2975 2025-05-21      272      28756       True\n    45496     3133 2025-09-10      272      28756       True\n    45501     3133 2025-09-15      272      28756       True\n    455

Common match IDs: []

2. PLAYED-XI PARTICIPANTS WITHOUT DELIVERY INVOLVEMENT
Unique XI-only participants: 512

Flag distribution among XI-only participants:

is_captain


' is_captain  records\n      False      486\n       True       26'

... (5 outputs left out)

' is_impact_off  records\n             0      497\n             1       15'


XI-ONLY PARTICIPANTS BY FORMAT


'match_format  records  distinct_players  distinct_matches\n         T20      299                99               176\n        T20I      213               129               148'


BATTING ORDER COVERAGE FOR XI-ONLY PARTICIPANTS
With batting order: 512
Without batting order: 0


### Investigate the 512 XI-only participants

In [95]:
# Build maximum observed striker batting order by match and batting team
observed_batting_order = (
    primary_regulation_deliveries
    .groupby(["match_id", "batting_team_id"])
    .agg(
        max_observed_batting_order=("batting_order", "max"),
        observed_striker_count=("striker_id", "nunique")
    )
    .reset_index()
    .rename(columns={"batting_team_id": "team_id"})
)

# Attach observed batting-order data to the 512 XI-only participants
xi_only_analysis = xi_only_detail.merge(
    observed_batting_order,
    on=["match_id", "team_id"],
    how="left"
)

# Compare each player's listed batting position with observed striker order
xi_only_analysis["batting_order_comparison"] = "Unknown"

xi_only_analysis.loc[
    xi_only_analysis["max_observed_batting_order"].isna(),
    "batting_order_comparison"
] = "No observed batting-order data for team"

xi_only_analysis.loc[
    xi_only_analysis["batting_order"] >
    xi_only_analysis["max_observed_batting_order"],
    "batting_order_comparison"
] = "Listed below last observed batter"

xi_only_analysis.loc[
    xi_only_analysis["batting_order"] <=
    xi_only_analysis["max_observed_batting_order"],
    "batting_order_comparison"
] = "Listed within observed batting-order range"

print("1. XI-ONLY PARTICIPANTS BY BATTING-ORDER COMPARISON")

display(
    xi_only_analysis["batting_order_comparison"]
    .value_counts(dropna=False)
    .rename_axis("classification")
    .reset_index(name="participants")
    .to_string(index=False)
)

print("\n2. XI-ONLY PARTICIPANTS BY BATTING POSITION")

display(
    xi_only_analysis.groupby("batting_order_comparison").agg(
        participants=("player_id", "size"),
        distinct_players=("player_id", "nunique"),
        distinct_matches=("match_id", "nunique")
    ).reset_index().to_string(index=False)
)

print("\n3. XI-ONLY PARTICIPANTS WITHIN OBSERVED BATTING-ORDER RANGE")

display(
    xi_only_analysis[
        xi_only_analysis["batting_order_comparison"] ==
        "Listed within observed batting-order range"
    ][
        [
            "match_id", "team_id", "player_id",
            "batting_order", "max_observed_batting_order",
            "is_captain", "is_wk", "is_impact", "is_impact_off"
        ]
    ].head(30).to_string(index=False)
)

print("\n4. SUMMARY BY FORMAT")

display(
    xi_only_analysis.groupby(
        ["match_format", "batting_order_comparison"]
    ).size().reset_index(name="participants")
    .to_string(index=False)
)

1. XI-ONLY PARTICIPANTS BY BATTING-ORDER COMPARISON


'                            classification  participants\n         Listed below last observed batter           511\nListed within observed batting-order range             1'


2. XI-ONLY PARTICIPANTS BY BATTING POSITION


'                  batting_order_comparison  participants  distinct_players  distinct_matches\n         Listed below last observed batter           511               202               323\nListed within observed batting-order range             1                 1                 1'


3. XI-ONLY PARTICIPANTS WITHIN OBSERVED BATTING-ORDER RANGE


' match_id  team_id  player_id  batting_order  max_observed_batting_order is_captain is_wk  is_impact  is_impact_off\n    51385       21       3281          6.000                           8      False False          0              0'


4. SUMMARY BY FORMAT


'match_format                   batting_order_comparison  participants\n         T20          Listed below last observed batter           299\n        T20I          Listed below last observed batter           212\n        T20I Listed within observed batting-order range             1'

### Investigate the single exception

In [97]:
exception_match_id = 51385
exception_team_id = 21
exception_player_id = 3281

print("1. MATCH DETAILS")

display(
    target_matches[
        target_matches["match_id"] == exception_match_id
    ][
        [
            "match_id", "comp_id", "match_date",
            "home_team", "away_team", "result"
        ]
    ].to_string(index=False)
)

print("\n2. COMPLETE TEAM PLAYING-XI RECORDS")

team_xi = primary_xi[
    (primary_xi["match_id"] == exception_match_id) &
    (primary_xi["team_id"] == exception_team_id)
].copy()

display(
    team_xi.sort_values("batting_order")[
        [
            "player_id", "batting_order", "is_played",
            "is_captain", "is_wk", "is_impact", "is_impact_off"
        ]
    ].to_string(index=False)
)

print("\n3. DELIVERY PARTICIPATION BY PLAYER")

team_deliveries = primary_regulation_deliveries[
    (
        (primary_regulation_deliveries["match_id"] == exception_match_id) &
        (primary_regulation_deliveries["batting_team_id"] == exception_team_id)
    ) |
    (
        (primary_regulation_deliveries["match_id"] == exception_match_id) &
        (primary_regulation_deliveries["bowling_team_id"] == exception_team_id)
    )
].copy()

team_participation = pd.concat([
    team_deliveries[
        ["striker_id", "striker_name", "batting_order"]
    ].rename(columns={
        "striker_id": "player_id",
        "striker_name": "player_name"
    }).assign(role="Striker"),

    team_deliveries[
        ["non_striker_id", "non_striker_name"]
    ].rename(columns={
        "non_striker_id": "player_id",
        "non_striker_name": "player_name"
    }).assign(role="Non-striker"),

    team_deliveries[
        ["bowler_id", "bowler_name"]
    ].rename(columns={
        "bowler_id": "player_id",
        "bowler_name": "player_name"
    }).assign(role="Bowler")
], ignore_index=True)

display(
    team_participation.groupby(
        ["player_id", "player_name", "role"]
    ).size().reset_index(name="delivery_records")
    .sort_values(["role", "player_id"])
    .to_string(index=False)
)

print("\n4. EXCEPTION PLAYER DELIVERY RECORDS")

display(
    team_participation[
        team_participation["player_id"] == exception_player_id
    ].to_string(index=False)
)

print("\n5. INNINGS-LEVEL PROFILE")

display(
    team_deliveries.groupby("innings").agg(
        delivery_records=("innings_ball_num", "size"),
        max_over=("overid", "max"),
        wickets=("is_wicket", "sum")
    ).reset_index().to_string(index=False)
)

1. MATCH DETAILS


' match_id  comp_id match_date home_team  away_team                      result\n    51385     3568 2026-07-19  Zimbabwe Bangladesh Bangladesh Won by 4 wickets'


2. COMPLETE TEAM PLAYING-XI RECORDS


' player_id  batting_order  is_played is_captain is_wk  is_impact  is_impact_off\n      4780          1.000       True      False False          0              0\n      7900          2.000       True      False False          0              0\n      7901          3.000       True      False  True          0              0\n      6651          4.000       True       True False          0              0\n      5839          5.000       True      False False          0              0\n      3281          6.000       True      False False          0              0\n     38855          7.000       True      False False          0              0\n      5837          8.000       True      False False          0              0\n      8380          9.000       True      False False          0              0\n      2681         10.000       True      False False          0              0\n     48616         11.000       True      False False          0              0\n     37939            NaN  


3. DELIVERY PARTICIPATION BY PLAYER


' player_id             player_name        role  delivery_records\n      2463               Ryan Burl      Bowler                 6\n      2681            Taskin Ahmed      Bowler                24\n      2984           Sikandar Raza      Bowler                26\n      4874         Richard Ngarava      Bowler                25\n      5837            Mahedi Hasan      Bowler                25\n      6523     Blessing Muzarabani      Bowler                24\n      8380 Mohammad Rishad Hossain      Bowler                24\n     21784              Brad Evans      Bowler                22\n     24017           Brian Bennett      Bowler                 6\n     33969        Tinotenda Maposa      Bowler                12\n     38855      Mohammad Saifuddin      Bowler                24\n     48616           Abdul Saqlain      Bowler                25\n      2463               Ryan Burl Non-striker                11\n      2984           Sikandar Raza Non-striker                17\n      487


4. EXCEPTION PLAYER DELIVERY RECORDS


'Empty DataFrame\nColumns: [player_id, player_name, batting_order, role]\nIndex: []'


5. INNINGS-LEVEL PROFILE


' innings  delivery_records  max_over  wickets\n       1               122        20        7\n       2               121        20        6'

### Verify the team and innings context

In [98]:
match_id = 51385
player_id = 3281

print("1. TEAM IDENTITIES")

display(
    target_matches[target_matches["match_id"] == match_id][
        ["match_id", "home_team_id", "home_team",
         "away_team_id", "away_team", "result"]
    ].to_string(index=False)
)

print("\n2. PLAYER XI RECORD")

display(
    primary_xi[
        (primary_xi["match_id"] == match_id) &
        (primary_xi["player_id"] == player_id)
    ].to_string(index=False)
)

print("\n3. TEAM-WISE DELIVERY PARTICIPATION")

match_data = primary_regulation_deliveries[
    primary_regulation_deliveries["match_id"] == match_id
]

display(
    match_data.groupby(
        ["innings", "batting_team_id", "bowling_team_id"]
    ).agg(
        deliveries=("innings_ball_num", "size"),
        batters=("striker_id", "nunique"),
        wickets=("is_wicket", "sum")
    ).reset_index().to_string(index=False)
)

print("\n4. BATTING ORDER OBSERVED BY INNINGS")

display(
    match_data.groupby(
        ["innings", "batting_team_id", "batting_order"]
    ).agg(
        batters=("striker_id", "nunique"),
        deliveries=("striker_id", "size")
    ).reset_index().sort_values(
        ["innings", "batting_order"]
    ).to_string(index=False)
)

print("\n5. PLAYER MAPPING")

display(
    competition_team_player_mapping[
        competition_team_player_mapping["player_id"] == player_id
    ].to_string(index=False)
)

1. TEAM IDENTITIES


' match_id  home_team_id home_team  away_team_id  away_team                      result\n    51385            72  Zimbabwe            21 Bangladesh Bangladesh Won by 4 wickets'


2. PLAYER XI RECORD


' match_id  team_id  player_id  batting_order  is_played is_captain is_wk  is_impact  is_impact_off\n    51385       21       3281          6.000       True      False False          0              0'


3. TEAM-WISE DELIVERY PARTICIPATION


' innings  batting_team_id  bowling_team_id  deliveries  batters  wickets\n       1               72               21         122        9        7\n       2               21               72         121        8        6'


4. BATTING ORDER OBSERVED BY INNINGS


' innings  batting_team_id  batting_order  batters  deliveries\n       1               72              1        1          39\n       1               72              2        1           5\n       1               72              3        1          54\n       1               72              4        1           6\n       1               72              5        1           7\n       1               72              6        1           3\n       1               72              7        1           1\n       1               72              8        1           6\n       1               72              9        1           1\n       2               21              1        1           5\n       2               21              2        1          58\n       2               21              3        1          23\n       2               21              4        1          21\n       2               21              5        1           9\n       2               21              6        1     


5. PLAYER MAPPING


' comp_id                                           comp_name  team_id  team_name  player_id      player_name\n    1452          Bangladesh in West Indies T20I Series 2022       21 Bangladesh       3281 Mosaddek Hossain\n    1523             Bangladesh in Zimbabwe T20I Series 2022       21 Bangladesh       3281 Mosaddek Hossain\n    1533                         Mens Twenty20 Asia Cup 2022       21 Bangladesh       3281 Mosaddek Hossain\n    1601 Bangladesh in United Arab Emirates T20I Series 2022       21 Bangladesh       3281 Mosaddek Hossain\n    1608                 New Zealand T20I Tri Series 2022/23       21 Bangladesh       3281 Mosaddek Hossain\n    3568             Bangladesh in Zimbabwe T20I Series 2026       21 Bangladesh       3281 Mosaddek Hossain'

# Consolidated audit summary

In [101]:

# SECTION 11.1 — CONSOLIDATED PHASE 1 AUDIT SUMMARY
# Read-only: this cell does not modify source datasets.

import pandas as pd

# 1. Recalculate core population counts
eligible_match_count = len(set(primary_match_ids))
regulation_delivery_count = len(primary_regulation_deliveries)
super_over_delivery_count = len(primary_deliveries[
    primary_deliveries["innings"].isin([3, 4])
])

regulation_match_count = primary_regulation_deliveries["match_id"].nunique()
regulation_innings_count = primary_regulation_deliveries[
    "innings"
].nunique()

print("=" * 65)
print("PHASE 1 — PRIMARY POPULATION SUMMARY")
print("=" * 65)

population_summary = pd.DataFrame([
    ["Eligible matches", eligible_match_count],
    ["Matches represented in regulation deliveries", regulation_match_count],
    ["Regulation deliveries (innings 1 and 2)", regulation_delivery_count],
    ["Super Over deliveries (innings 3 and 4)", super_over_delivery_count],
    ["Total deliveries including Super Overs",
     regulation_delivery_count + super_over_delivery_count],
    ["Distinct players in regulation delivery roles",
     len(set(primary_regulation_deliveries["striker_id"]) |
         set(primary_regulation_deliveries["non_striker_id"]) |
         set(primary_regulation_deliveries["bowler_id"]))],
], columns=["Metric", "Value"])

display(population_summary.to_string(index=False))

# 2. Match format split
print("\n" + "=" * 65)
print("MATCH FORMAT SPLIT")
print("=" * 65)

format_data = target_matches.merge(
    competitions[["comp_id", "match_format"]],
    on="comp_id",
    how="left",
    validate="many_to_one"
)

format_summary = (
    format_data.groupby("match_format", dropna=False)
    .agg(matches=("match_id", "nunique"))
    .reset_index()
)

print("MATCH FORMAT SPLIT")
display(format_summary.to_string(index=False))

# Confirm all target matches received a format
missing_format_count = format_data["match_format"].isna().sum()
print("\nMatches without a competition format:", missing_format_count)

assert missing_format_count == 0
assert format_summary["matches"].sum() == 853
# 3. Data-treatment register
print("\n" + "=" * 65)
print("DATA-TREATMENT REGISTER")
print("=" * 65)

treatment_register = pd.DataFrame([
    ["Match population",
     "813 eligible matches",
     "Use as proposed primary population",
     "Verified"],

    ["Super Overs",
     "4 matches; 26 deliveries",
     "Keep separate from regulation innings",
     "Verified"],

    ["Abandoned / No Result matches",
     "40 matches excluded or incomplete",
     "Exclude from primary completed-match population",
     "Verified"],

    ["Player identity",
     "Player IDs are unique keys; names can collide",
     "Use player_id, not player_name, as identity key",
     "Verified"],

    ["extras1_runs",
     "Encoded category; 6 represents no extras",
     "Do not treat as a run quantity",
     "Verified"],

    ["extras_2_runs",
     "35 non-zero records in regulation population; semantics unresolved",
     "Preserve raw; do not assign meaning yet",
     "Unresolved"],

    ["runs",
     "Batter runs versus total delivery runs not established",
     "Do not assume semantics without further validation",
     "Unresolved"],

    ["Boundary flags",
     "23 runs/flag mismatches in regulation population",
     "Preserve runs, is_four and is_six independently",
     "Caution"],

    ["Wicket fields",
     "is_wicket aligns with dismissal_type; one wicket lacks dismissed_player_id",
     "Preserve missing dismissed-player identity",
     "Caution"],

    ["Bowling speed",
     "Zero, negative, low and high values observed",
     "Exclude from primary modelling until validated",
     "Caution"],

    ["Delivery timestamps",
     "All parse; midnight placeholders and repeated values exist",
     "Do not use as sole event-ordering key",
     "Caution"],

    ["Over sequence",
     "147 extended overs without recorded wide/overstep in primary population",
     "Do not infer legal-ball counts from sequence numbers",
     "Unresolved"],

    ["Playing XI",
     "Registration table includes non-playing and repeated records",
     "Use distinct player IDs and is_played carefully",
     "Verified"],

    ["Impact flags",
     "Paired flags observed, but exact semantics not established",
     "Retain original fields; do not infer substitution direction",
     "Unresolved"],

    ["Mosaddek Hossain",
     "Match 51385: listed as played, no delivery-role involvement",
     "Retain as explicit XI/delivery inconsistency",
     "Unresolved"],
], columns=[
    "Area", "Evidence", "Proposed treatment", "Status"
])

display(treatment_register.to_string(index=False))

# 4. Final integrity assertions
print("\n" + "=" * 65)
print("FINAL POPULATION CHECKS")
print("=" * 65)

assert eligible_match_count == 813
assert regulation_match_count == 813
assert regulation_delivery_count == 192235
assert super_over_delivery_count == 26

print("PASS: 813 eligible matches represented in regulation deliveries.")
print("PASS: 192,235 regulation deliveries.")
print("PASS: 26 Super Over deliveries kept separately.")
print("PASS: No source DataFrame was modified.")

print("\nSection 11.1 complete.")


PHASE 1 — PRIMARY POPULATION SUMMARY


'                                       Metric  Value\n                             Eligible matches    813\n Matches represented in regulation deliveries    813\n      Regulation deliveries (innings 1 and 2) 192235\n      Super Over deliveries (innings 3 and 4)     26\n       Total deliveries including Super Overs 192261\nDistinct players in regulation delivery roles    892'


MATCH FORMAT SPLIT
MATCH FORMAT SPLIT


'match_format  matches\n         T20      371\n        T20I      482'


Matches without a competition format: 0

DATA-TREATMENT REGISTER


'                         Area                                                                   Evidence                                          Proposed treatment     Status\n             Match population                                                       813 eligible matches                          Use as proposed primary population   Verified\n                  Super Overs                                                   4 matches; 26 deliveries                       Keep separate from regulation innings   Verified\nAbandoned / No Result matches                                          40 matches excluded or incomplete             Exclude from primary completed-match population   Verified\n              Player identity                              Player IDs are unique keys; names can collide             Use player_id, not player_name, as identity key   Verified\n                 extras1_runs                                   Encoded category; 6 represents no extras          


FINAL POPULATION CHECKS
PASS: 813 eligible matches represented in regulation deliveries.
PASS: 192,235 regulation deliveries.
PASS: 26 Super Over deliveries kept separately.
PASS: No source DataFrame was modified.

Section 11.1 complete.


#